# Planar Ornstein–Uhlenbeck FEM: all benchmarks through $n=512$

이 노트북은 세 벤치마크의 $\mathcal B$와 $\mathcal E$ FEM 계산을
실제로 다시 실행합니다. 기본 설정은 `RUN_NEW_EXPERIMENT = True`입니다.

- 양의 trace: $A=\left(\begin{smallmatrix}2&1\\1&2\end{smallmatrix}\right)$,
  $G=\left(\begin{smallmatrix}3&4\\1&3\end{smallmatrix}\right)$.
- 음의 trace이지만 coercive인 예제:
  $A=\left(\begin{smallmatrix}2&1\\1&1\end{smallmatrix}\right)$,
  $G=\left(\begin{smallmatrix}0&1\\-2&-3\end{smallmatrix}\right)$.
- Noncoercive 계수군: $A=\operatorname{diag}(1,2)$,
  $G_k=k\left(\begin{smallmatrix}-1&1\\-1&-2\end{smallmatrix}\right)$,
  $k\in\{4,8,12,16,20\}$.

모든 경우에 $U=(-1,1)^2$,
$u_\star=(1-x_1^2)(1-x_2^2)$, $f=-Lu_\star$를 사용합니다.
P1 FEM, 같은 삼각형 격자, 같은 SuperLU 설정을 사용합니다.
$n\in\{8,16,32,64,128,256,512\}$이며 $n=512$에는
261,121개의 내부 미지수와 524,288개의 삼각형이 있습니다.

보고 항목은 절대오차와 상대오차($L^2$ 및 full $H^{1,2}$), 전체 풀이시간입니다.
수렴률 열은 사용하지 않습니다. 상대오차는 백분율이 아닌 비율입니다.
계수 $C$는 직접 공식으로 구하므로 $S^{-1}$를 사용하지 않습니다.

Google Colab에서 이 `.ipynb` 하나를 업로드하고 런타임 → 모두 실행을 선택하세요.
VS Code의 Jupyter 환경에서도 실행할 수 있습니다. CPU 런타임이면 됩니다.
전체 실험은 98개 설정에 대해 각각 1회 warm-up과 7회 새 조립 및 LU 풀이를 수행하므로
몇 개의 그래프만 다시 그리는 작업보다 오래 걸립니다. 실행시간은 머신에 따라 달라집니다.

In [ ]:
# The default performs a fresh full experiment.
RUN_NEW_EXPERIMENT = True
FULL_VALIDATION = True
QUICK_MODE = False

SIZES = [8, 16, 32, 64, 128, 256, 512]
K_VALUES = [4, 8, 12, 16, 20]
REPEATS = 7
ASSEMBLY_ORDER = 8
ERROR_ORDER = 8
BATCH_SIZE = 4096

# Optional smoke test, not the paper protocol.
if QUICK_MODE:
    SIZES = [8, 16]
    REPEATS = 1

if RUN_NEW_EXPERIMENT:
    if not SIZES or SIZES != sorted(set(SIZES)) or any(int(n) != n or n < 2 for n in SIZES):
        raise ValueError("SIZES must be increasing distinct integers of at least 2.")
    if not K_VALUES or len(K_VALUES) != len(set(K_VALUES)) or any(k <= 0 for k in K_VALUES):
        raise ValueError("K_VALUES must be distinct and positive.")
    if int(REPEATS) != REPEATS or REPEATS < 1:
        raise ValueError("REPEATS must be a positive integer.")
    if ASSEMBLY_ORDER < 8 or ERROR_ORDER < 8:
        raise ValueError("Keep both quadrature orders at least 8 for these benchmarks.")

`RUN_NEW_EXPERIMENT = False`는 이 파일에 포함된 새 실험의 기록 데이터를 읽어
표와 그림을 재생성하는 모드입니다. 이 모드에서는 FEM 계산이나 시간 측정을 다시 하지 않습니다.
기록 데이터는 원래 계산 머신의 결과이며, Google Colab에서 실행된 결과라고 주장하지 않습니다.
새 실행은 별도의 시간표시 폴더에 저장되어 기록 데이터를 덮어쓰지 않습니다.

`FULL_VALIDATION = True`는 전체 실험 후 추가 적분 정확도 및 선형계 풀이 검사를 실행합니다.
검증 시간은 벤치마크 풀이시간에 포함되지 않습니다.
`QUICK_MODE = True`는 설치 및 실행 확인용이며 논문용 LaTeX 본문을 생성하지 않습니다.

In [ ]:
import os
# Limit threads before numerical imports and before starting subprocesses.
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[name] = "1"

import sys
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import json
import csv
import shutil
import platform

required = ["numpy", "scipy", "matplotlib", "threadpoolctl"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import numpy as np
import scipy
import matplotlib
from IPython.display import display, Markdown, Image, FileLink

WORKDIR = Path.cwd() / "ou_fem_512_reproduction"
WORKDIR.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version())
print("NumPy:", np.__version__, "SciPy:", scipy.__version__, "Matplotlib:", matplotlib.__version__)
print("Workspace:", WORKDIR.resolve())
print("Mode:", "fresh FEM computations" if RUN_NEW_EXPERIMENT else "recorded-data replay")

아래 셀은 필요한 Python 소스, LaTeX 템플릿, 기록 CSV/JSON을 복원합니다.
각 파일을 SHA-256으로 확인합니다. 논문 그림은 선택된 CSV에서 새로 생성합니다.
외부 데이터 다운로드는 필요하지 않습니다.

In [ ]:
# @title Restore the self-contained code and recorded data
SOURCE_PAYLOAD = {'code/build_sections.py': {'sha256': '6e8e9e907afd5fbf85eb09b80306825d11d55d6be205085d3ed26d02c13cb545', 'base64': '...'}, 'code/make_all_figures.py': {'sha256': '795c1fcba5520e92e81f2015e6ee87e585db626211036e227d61ea1d14dc391e', 'base64': 'IiIiQ3JlYXRlIGFsbCBtYW51c2NyaXB0IGZpZ3VyZXMgZnJvbSBuZXdseSBtZWFzdXJlZCBGRU0gcmVzdWx0IGZpbGVzLgoKVXNhZ2U6CiAgICBweXRob24gY29kZS9tYWtlX2FsbF9maWd1cmVzLnB5IC0tcmVzdWx0cy1yb290IHJlc3VsdHMgLS1vdXRwdXQgLgoKVGhlIHRocmVlIGlucHV0IGRpcmVjdG9yaWVzIGFyZSBgYHBvc2l0aXZlYGAsIGBgbmVnYXRpdmVgYCwgYW5kCmBgbm9uY29lcmNpdmVgYC4gRWFjaCBtdXN0IGNvbnRhaW4gY29udmVyZ2VuY2UuY3N2LiBObyBGRU0gc29sdmUsIHRpbWluZwptZWFzdXJlbWVudCwgaW50ZXJwb2xhdGlvbiwgc21vb3RoaW5nLCBvciBzeW50aGV0aWMgcG9pbnQgaXMgcGVyZm9ybWVkIGhlcmUuCkV2ZXJ5IG1hcmtlciBpcyBhIHJvdyBvZiB0aGUgaW5wdXQgZmlsZS4gUmVsYXRpdmUgZXJyb3JzIHVzZSBleGFjdCBub3Jtcy4KVGhlIGVsZXZlbiBQREYgYmFzZW5hbWVzIG1hdGNoIHRoZSBtYW51c2NyaXB0OyBQTkcgcHJldmlld3MgYXJlIGFsc28gc2F2ZWQuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBtYXRwbG90bGliCgptYXRwbG90bGliLnVzZSgiQWdnIikKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdApmcm9tIG1hdHBsb3RsaWIubGluZXMgaW1wb3J0IExpbmUyRApmcm9tIG1hdHBsb3RsaWIudGlja2VyIGltcG9ydCBOdWxsRm9ybWF0dGVyCmltcG9ydCBudW1weSBhcyBucAoKCkVYQUNUX0wyX05PUk0gPSAxNi4wIC8gMTUuMApFWEFDVF9IMV9OT1JNID0gMTYuMCAqIG1hdGguc3FydCg2LjApIC8gMTUuMApERUZBVUxUX1NJWkVTID0gKDgsIDE2LCAzMiwgNjQsIDEyOCwgMjU2LCA1MTIpCkVYUEVDVEVEX0tTID0gKDQsIDgsIDEyLCAxNiwgMjApClNUWUxFUyA9IHsKICAgICJCIjogZGljdChjb2xvcj0iIzE3NjRBQiIsIG1hcmtlcj0ibyIsIGxpbmVzdHlsZT0iLSIsCiAgICAgICAgICAgICAgbWFya2VyZmFjZWNvbG9yPSIjMTc2NEFCIiwgbWFya2Vyc2l6ZT00LjAsCiAgICAgICAgICAgICAgbGFiZWw9ciIkXG1hdGhjYWx7Qn0kOiBvcmlnaW5hbCIsIHpvcmRlcj0zKSwKICAgICJFIjogZGljdChjb2xvcj0iI0Q2NUYxOCIsIG1hcmtlcj0icyIsIGxpbmVzdHlsZT0iLS0iLAogICAgICAgICAgICAgIG1hcmtlcmZhY2Vjb2xvcj0ibm9uZSIsIG1hcmtlcnNpemU9NS44LAogICAgICAgICAgICAgIGxhYmVsPXIiJFxtYXRoY2Fse0V9JDogd2VpZ2h0ZWQiLCB6b3JkZXI9NCksCn0KCgpkZWYgcmVhZF9yZXN1bHRzKHBhdGg6IFBhdGgsIHNpemVzOiB0dXBsZVtpbnQsIC4uLl0sIGZhbWlseTogYm9vbCkgLT4gbGlzdFtkaWN0XToKICAgIHdpdGggcGF0aC5vcGVuKG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIHN0cmVhbToKICAgICAgICByb3dzID0gbGlzdChjc3YuRGljdFJlYWRlcihzdHJlYW0pKQogICAgcmVxdWlyZWQgPSB7Im4iLCAiaCIsICJuZG9mIiwgIm1ldGhvZCIsICJsMiIsICJoMSIsCiAgICAgICAgICAgICAgICAidG90YWxfbWVkaWFuX3MiLCAidG90YWxfcTI1X3MiLCAidG90YWxfcTc1X3MifQogICAgaWYgZmFtaWx5OgogICAgICAgIHJlcXVpcmVkLmFkZCgiayIpCiAgICBpZiBub3Qgcm93cyBvciBub3QgcmVxdWlyZWQuaXNzdWJzZXQocm93c1swXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkVtcHR5IGZpbGUgb3IgbWlzc2luZyBjb2x1bW5zOiB7cGF0aH0iKQogICAgZXhwZWN0ZWQgPSB7KGssIG4sIG1ldGhvZCkKICAgICAgICAgICAgICAgIGZvciBrIGluIChFWFBFQ1RFRF9LUyBpZiBmYW1pbHkgZWxzZSAoTm9uZSwpKQogICAgICAgICAgICAgICAgZm9yIG4gaW4gc2l6ZXMgZm9yIG1ldGhvZCBpbiAoIkIiLCAiRSIpfQogICAgc2VlbiA9IHNldCgpCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgcm93WyJuIl0gPSBpbnQocm93WyJuIl0pCiAgICAgICAgcm93WyJuZG9mIl0gPSBpbnQocm93WyJuZG9mIl0pCiAgICAgICAgcm93WyJrIl0gPSBmbG9hdChyb3dbImsiXSkgaWYgZmFtaWx5IGVsc2UgTm9uZQogICAgICAgIGtleSA9IChyb3dbImsiXSwgcm93WyJuIl0sIHJvd1sibWV0aG9kIl0pCiAgICAgICAgaWYga2V5IGluIHNlZW46CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJEdXBsaWNhdGUgb2JzZXJ2YXRpb24ge2tleX0gaW4ge3BhdGh9IikKICAgICAgICBzZWVuLmFkZChrZXkpCiAgICAgICAgZm9yIG5hbWUgaW4gKCJoIiwgImwyIiwgImgxIiwgInRvdGFsX21lZGlhbl9zIiwKICAgICAgICAgICAgICAgICAgICAgInRvdGFsX3EyNV9zIiwgInRvdGFsX3E3NV9zIik6CiAgICAgICAgICAgIHJvd1tuYW1lXSA9IGZsb2F0KHJvd1tuYW1lXSkKICAgICAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUocm93W25hbWVdKSBvciByb3dbbmFtZV0gPD0gMDoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJOb25wb3NpdGl2ZS9ub25maW5pdGUge25hbWV9IGZvciB7a2V5fToge3BhdGh9IikKICAgICAgICBpZiByb3dbIm5kb2YiXSAhPSAocm93WyJuIl0gLSAxKSAqKiAyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5leHBlY3RlZCBudW1iZXIgb2YgZGVncmVlcyBvZiBmcmVlZG9tIGZvciB7a2V5fSIpCiAgICAgICAgaWYgbm90IG1hdGguaXNjbG9zZShyb3dbImgiXSwgMiAqIG1hdGguc3FydCgyKSAvIHJvd1sibiJdLCByZWxfdG9sPTFlLTEyKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuZXhwZWN0ZWQgbWVzaCBkaWFtZXRlciBmb3Ige2tleX0iKQogICAgICAgIGlmIHJvd1siaDEiXSA8IHJvd1sibDIiXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkZ1bGwgSDEgZXJyb3IgaXMgc21hbGxlciB0aGFuIEwyIGVycm9yIGZvciB7a2V5fSIpCiAgICAgICAgaWYgbm90IHJvd1sidG90YWxfcTI1X3MiXSA8PSByb3dbInRvdGFsX21lZGlhbl9zIl0gPD0gcm93WyJ0b3RhbF9xNzVfcyJdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWVkaWFuIG91dHNpZGUgaXRzIGludGVycXVhcnRpbGUgcmFuZ2UgZm9yIHtrZXl9IikKICAgICAgICByb3dbImwyX3JlbGF0aXZlIl0gPSByb3dbImwyIl0gLyBFWEFDVF9MMl9OT1JNCiAgICAgICAgcm93WyJoMV9yZWxhdGl2ZSJdID0gcm93WyJoMSJdIC8gRVhBQ1RfSDFfTk9STQogICAgaWYgc2VlbiAhPSBleHBlY3RlZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW5jb21wbGV0ZS91bmV4cGVjdGVkIG9ic2VydmF0aW9ucyBpbiB7cGF0aH07ICIKICAgICAgICAgICAgICAgICAgICAgICAgIGYibWlzc2luZz17c29ydGVkKGV4cGVjdGVkIC0gc2Vlbiwga2V5PXN0cil9LCAiCiAgICAgICAgICAgICAgICAgICAgICAgICBmImV4dHJhPXtzb3J0ZWQoc2VlbiAtIGV4cGVjdGVkLCBrZXk9c3RyKX0iKQogICAgcmV0dXJuIHJvd3MKCgpkZWYgc3Vic2V0KHJvd3M6IGxpc3RbZGljdF0sIG1ldGhvZDogc3RyLCBrPU5vbmUpIC0+IGxpc3RbZGljdF06CiAgICByZXR1cm4gc29ydGVkKChyb3cgZm9yIHJvdyBpbiByb3dzCiAgICAgICAgICAgICAgICAgICBpZiByb3dbIm1ldGhvZCJdID09IG1ldGhvZCBhbmQgcm93WyJrIl0gPT0gayksCiAgICAgICAgICAgICAgICAgIGtleT1sYW1iZGEgcm93OiByb3dbIm4iXSkKCgpkZWYgbGVnZW5kX2hhbmRsZXMoKSAtPiBsaXN0W0xpbmUyRF06CiAgICByZXR1cm4gW0xpbmUyRChbMF0sIFswXSwgKipzdHlsZSwgbGluZXdpZHRoPTEuMzUsIG1hcmtlcmVkZ2V3aWR0aD0wLjk1KQogICAgICAgICAgICBmb3Igc3R5bGUgaW4gU1RZTEVTLnZhbHVlcygpXQoKCmRlZiBlcnJvcl9heGVzKGF4LCByb3dzLCBtZXRyaWMsIHNpemVzLCBrPU5vbmUpOgogICAgZm9yIG1ldGhvZCwgc3R5bGUgaW4gU1RZTEVTLml0ZW1zKCk6CiAgICAgICAgZGF0YSA9IHN1YnNldChyb3dzLCBtZXRob2QsIGspCiAgICAgICAgYXgucGxvdChbcm93WyJuIl0gZm9yIHJvdyBpbiBkYXRhXSwgW3Jvd1ttZXRyaWNdIGZvciByb3cgaW4gZGF0YV0sCiAgICAgICAgICAgICAgICAqKnN0eWxlLCBsaW5ld2lkdGg9MS4zNSwgbWFya2VyZWRnZXdpZHRoPTAuOTUpCiAgICBheC5zZXRfeHNjYWxlKCJsb2ciLCBiYXNlPTIpCiAgICBheC5zZXRfeXNjYWxlKCJsb2ciKQogICAgYXguc2V0X3h0aWNrcyhzaXplcywgW3N0cihuKSBmb3IgbiBpbiBzaXplc10pCiAgICBheC54YXhpcy5zZXRfbWlub3JfZm9ybWF0dGVyKE51bGxGb3JtYXR0ZXIoKSkKICAgIGF4LnNldF94bGFiZWwociJTdWJkaXZpc2lvbnMgcGVyIGNvb3JkaW5hdGUsICRuJCIpCiAgICBheC5ncmlkKFRydWUsIHdoaWNoPSJtYWpvciIsIGNvbG9yPSIjRDBEMEQwIiwgbGluZXdpZHRoPTAuNTUsIGFscGhhPTAuNzUpCiAgICBheC5tYXJnaW5zKHg9MC4wOCwgeT0wLjEwKQoKCmRlZiBydW50aW1lX2F4ZXMoYXgsIHJvd3MsIHNpemVzLCBrPU5vbmUpOgogICAgZm9yIG1ldGhvZCwgc3R5bGUgaW4gU1RZTEVTLml0ZW1zKCk6CiAgICAgICAgZGF0YSA9IHN1YnNldChyb3dzLCBtZXRob2QsIGspCiAgICAgICAgc3ViZGl2aXNpb25zID0gbnAuYXJyYXkoW3Jvd1sibiJdIGZvciByb3cgaW4gZGF0YV0pCiAgICAgICAgYXguZmlsbF9iZXR3ZWVuKHN1YmRpdmlzaW9ucywgW3Jvd1sidG90YWxfcTI1X3MiXSBmb3Igcm93IGluIGRhdGFdLAogICAgICAgICAgICAgICAgICAgICAgICBbcm93WyJ0b3RhbF9xNzVfcyJdIGZvciByb3cgaW4gZGF0YV0sCiAgICAgICAgICAgICAgICAgICAgICAgIGNvbG9yPXN0eWxlWyJjb2xvciJdLCBhbHBoYT0wLjE3LCBsaW5ld2lkdGg9MCwgem9yZGVyPTEpCiAgICAgICAgYXgucGxvdChzdWJkaXZpc2lvbnMsIFtyb3dbInRvdGFsX21lZGlhbl9zIl0gZm9yIHJvdyBpbiBkYXRhXSwKICAgICAgICAgICAgICAgICoqc3R5bGUsIGxpbmV3aWR0aD0xLjM1LCBtYXJrZXJlZGdld2lkdGg9MC45NSkKICAgIGF4LnNldF94c2NhbGUoImxvZyIsIGJhc2U9MikKICAgIGF4LnNldF95c2NhbGUoImxvZyIpCiAgICBheC5zZXRfeHRpY2tzKHNpemVzLCBbc3RyKG4pIGZvciBuIGluIHNpemVzXSkKICAgIGF4LnhheGlzLnNldF9taW5vcl9mb3JtYXR0ZXIoTnVsbEZvcm1hdHRlcigpKQogICAgYXguc2V0X3hsYWJlbChyIlN1YmRpdmlzaW9ucyBwZXIgY29vcmRpbmF0ZSwgJG4kIikKICAgIGF4LnNldF95bGFiZWwoIlRvdGFsIHRpbWUgKHMpIikKICAgIGF4LmdyaWQoVHJ1ZSwgd2hpY2g9Im1ham9yIiwgY29sb3I9IiNEMEQwRDAiLCBsaW5ld2lkdGg9MC41NSwgYWxwaGE9MC43NSkKICAgIGF4Lm1hcmdpbnMoeD0wLjA4LCB5PTAuMTApCgoKZGVmIHNhdmUoZmlnLCBvdXRwdXQ6IFBhdGgsIHN0ZW06IHN0ciwgY3JlYXRlZDogbGlzdFtzdHJdKToKICAgIGZvciBleHRlbnNpb24gaW4gKCJwZGYiLCAicG5nIik6CiAgICAgICAgcGF0aCA9IG91dHB1dCAvIGYie3N0ZW19LntleHRlbnNpb259IgogICAgICAgIGt3YXJncyA9IHsiYmJveF9pbmNoZXMiOiAidGlnaHQiLCAicGFkX2luY2hlcyI6IDAuMDR9CiAgICAgICAgaWYgZXh0ZW5zaW9uID09ICJwZGYiOgogICAgICAgICAgICBrd2FyZ3NbIm1ldGFkYXRhIl0gPSB7CiAgICAgICAgICAgICAgICAiVGl0bGUiOiBzdGVtLAogICAgICAgICAgICAgICAgIlN1YmplY3QiOiAiT3JpZ2luYWwgYW5kIHdlaWdodGVkIFAxIEZFTTogbWVhc3VyZWQgcmVzdWx0cyIsCiAgICAgICAgICAgICAgICAiQ3JlYXRvciI6ICJtYWtlX2FsbF9maWd1cmVzLnB5IiwKICAgICAgICAgICAgfQogICAgICAgICMgUHVibGlzaCBhIGNvbXBsZXRlIGZpbGUgaW4gb25lIHJlbmFtZS4gUmVhZGVycyBuZXZlciBzZWUgYSBwYXJ0aWFsbHkKICAgICAgICAjIHdyaXR0ZW4gUERGIHdoZW4gcGxvdHRpbmcgYW5kIG1hbnVzY3JpcHQgY29tcGlsYXRpb24gcnVuIHNlcGFyYXRlbHkuCiAgICAgICAgdGVtcG9yYXJ5ID0gb3V0cHV0IC8gZiIue3N0ZW19LnBlbmRpbmcue2V4dGVuc2lvbn0iCiAgICAgICAgZmlnLnNhdmVmaWcodGVtcG9yYXJ5LCAqKmt3YXJncykKICAgICAgICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQogICAgICAgIGNyZWF0ZWQuYXBwZW5kKHN0cihwYXRoKSkKICAgIHBsdC5jbG9zZShmaWcpCgoKZGVmIG1ha2VfY29lcmNpdmVfZmlndXJlcyhyb3dzLCBwcmVmaXgsIHNpemVzLCBvdXRwdXQsIGNyZWF0ZWQpOgogICAgZm9yIHJlbGF0aXZlIGluIChGYWxzZSwgVHJ1ZSk6CiAgICAgICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KDYuOCwgMi43NSkpCiAgICAgICAgc3VmZml4ID0gIl9yZWxhdGl2ZSIgaWYgcmVsYXRpdmUgZWxzZSAiIgogICAgICAgIGZvciBheCwgbWV0cmljLCB0aXRsZSBpbiB6aXAoCiAgICAgICAgICAgICAgICBheGVzLCAoImwyIiwgImgxIiksCiAgICAgICAgICAgICAgICAociIkTF4yKFUpJCIsIHIiRnVsbCAkSF57MSwyfShVKSQiKSk6CiAgICAgICAgICAgIGVycm9yX2F4ZXMoYXgsIHJvd3MsIG1ldHJpYyArIHN1ZmZpeCwgc2l6ZXMpCiAgICAgICAgICAgIGF4LnNldF90aXRsZSh0aXRsZSkKICAgICAgICAgICAgYXguc2V0X3lsYWJlbCgiUmVsYXRpdmUgZXJyb3IiIGlmIHJlbGF0aXZlIGVsc2UgIkFic29sdXRlIGVycm9yIikKICAgICAgICBmaWcubGVnZW5kKGhhbmRsZXM9bGVnZW5kX2hhbmRsZXMoKSwgbG9jPSJsb3dlciBjZW50ZXIiLCBuY29sPTIsCiAgICAgICAgICAgICAgICAgICBiYm94X3RvX2FuY2hvcj0oMC41LCAtMC4wMTUpLCBjb2x1bW5zcGFjaW5nPTIuMywKICAgICAgICAgICAgICAgICAgIGhhbmRsZWxlbmd0aD0yLjgpCiAgICAgICAgZmlnLnN1YnBsb3RzX2FkanVzdChsZWZ0PTAuMDk1LCByaWdodD0wLjk4NSwgdG9wPTAuOTAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBib3R0b209MC4yOSwgd3NwYWNlPTAuMzcpCiAgICAgICAgc3RlbSA9IGYie3ByZWZpeH1feydyZWxhdGl2ZV9lcnJvcnMnIGlmIHJlbGF0aXZlIGVsc2UgJ2Vycm9ycyd9IgogICAgICAgIHNhdmUoZmlnLCBvdXRwdXQsIHN0ZW0sIGNyZWF0ZWQpCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDQuMzUsIDMuMCkpCiAgICBydW50aW1lX2F4ZXMoYXgsIHJvd3MsIHNpemVzKQogICAgYXgubGVnZW5kKGxvYz0idXBwZXIgbGVmdCIsIGhhbmRsZWxlbmd0aD0yLjgpCiAgICBmaWcuc3VicGxvdHNfYWRqdXN0KGxlZnQ9MC4xNiwgcmlnaHQ9MC45OCwgdG9wPTAuOTYsIGJvdHRvbT0wLjE5KQogICAgc2F2ZShmaWcsIG91dHB1dCwgZiJ7cHJlZml4fV9ydW50aW1lIiwgY3JlYXRlZCkKCgpkZWYgbWFrZV9ub25jb2VyY2l2ZV9maWd1cmVzKHJvd3MsIHNpemVzLCBvdXRwdXQsIGNyZWF0ZWQpOgogICAgZGVmaW5pdGlvbnMgPSAoCiAgICAgICAgKCJsMiIsIHIiVW53ZWlnaHRlZCAkTF4yKFUpJCBlcnJvciIsICJBYnNvbHV0ZSBlcnJvciIpLAogICAgICAgICgiaDEiLCByIkZ1bGwgdW53ZWlnaHRlZCAkSF57MSwyfShVKSQgZXJyb3IiLCAiQWJzb2x1dGUgZXJyb3IiKSwKICAgICAgICAoImwyX3JlbGF0aXZlIiwgciJSZWxhdGl2ZSB1bndlaWdodGVkICRMXjIoVSkkIGVycm9yIiwgIlJlbGF0aXZlIGVycm9yIiksCiAgICAgICAgKCJoMV9yZWxhdGl2ZSIsIHIiUmVsYXRpdmUgZnVsbCAkSF57MSwyfShVKSQgZXJyb3IiLCAiUmVsYXRpdmUgZXJyb3IiKSwKICAgICAgICAoInJ1bnRpbWUiLCAiVG90YWwgc29sdXRpb24gdGltZSIsICJUb3RhbCB0aW1lIChzKSIpLAogICAgKQogICAgZm9yIG1ldHJpYywgdGl0bGUsIHlsYWJlbCBpbiBkZWZpbml0aW9uczoKICAgICAgICBmaWcsIGdyaWQgPSBwbHQuc3VicGxvdHMoMiwgMywgZmlnc2l6ZT0oOC41LCA1LjQpKQogICAgICAgIGF4ZXMgPSBncmlkLnJhdmVsKCkKICAgICAgICBmb3IgYXgsIGsgaW4gemlwKGF4ZXMsIEVYUEVDVEVEX0tTKToKICAgICAgICAgICAgaWYgbWV0cmljID09ICJydW50aW1lIjoKICAgICAgICAgICAgICAgIHJ1bnRpbWVfYXhlcyhheCwgcm93cywgc2l6ZXMsIGs9aykKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGVycm9yX2F4ZXMoYXgsIHJvd3MsIG1ldHJpYywgc2l6ZXMsIGs9aykKICAgICAgICAgICAgICAgIGF4LnNldF95bGFiZWwoeWxhYmVsKQogICAgICAgICAgICBheC5zZXRfdGl0bGUocmYiJGs9e2t9JCIsIHBhZD01LCBmb250c2l6ZT0xMSkKICAgICAgICAgICAgYXgudGlja19wYXJhbXMoYXhpcz0ieSIsIGxhYmVsc2l6ZT05LjUpCiAgICAgICAgICAgIGF4LnRpY2tfcGFyYW1zKGF4aXM9IngiLCBsYWJlbHNpemU9OCkKICAgICAgICAgICAgYXgueGF4aXMubGFiZWwuc2V0X3NpemUoOSkKICAgICAgICAgICAgYXgueWF4aXMubGFiZWwuc2V0X3NpemUoMTApCiAgICAgICAgaW5mbyA9IGF4ZXNbLTFdCiAgICAgICAgaW5mby5zZXRfYXhpc19vZmYoKQogICAgICAgIGluZm8ubGVnZW5kKGhhbmRsZXM9bGVnZW5kX2hhbmRsZXMoKSwgbG9jPSJ1cHBlciBsZWZ0IiwKICAgICAgICAgICAgICAgICAgICBiYm94X3RvX2FuY2hvcj0oMC4wMiwgMC45NiksIGxhYmVsc3BhY2luZz0wLjksCiAgICAgICAgICAgICAgICAgICAgaGFuZGxlbGVuZ3RoPTIuOCwgZm9udHNpemU9MTApCiAgICAgICAgaWYgbWV0cmljID09ICJydW50aW1lIjoKICAgICAgICAgICAgZXhwbGFuYXRpb24gPSAoIk1lZGlhbiB0b3RhbCBzb2x1dGlvbiB0aW1lXG4iCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJTaGFkaW5nOiBpbnRlcnF1YXJ0aWxlIHJhbmdlXG5cbiIKICAgICAgICAgICAgICAgICAgICAgICAgICAgIkNvZWZmaWNpZW50IHNldHVwLCBhc3NlbWJseSxcbiIKICAgICAgICAgICAgICAgICAgICAgICAgICAgImZyZXNoIHNwYXJzZSBMVSBhbmQgc29sdmUiKQogICAgICAgIGVsaWYgbWV0cmljLmVuZHN3aXRoKCJyZWxhdGl2ZSIpOgogICAgICAgICAgICBub3JtID0gciIkMTYvMTUkIiBpZiBtZXRyaWMuc3RhcnRzd2l0aCgibDIiKSBlbHNlIHIiJDE2XHNxcnR7Nn0vMTUkIgogICAgICAgICAgICBleHBsYW5hdGlvbiA9ICgiQWJzb2x1dGUgZXJyb3IgZGl2aWRlZCBieVxuIgogICAgICAgICAgICAgICAgICAgICAgICAgICBmInRoZSBleGFjdCBzb2x1dGlvbiBub3JtIHtub3JtfVxuXG4iCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJEaW1lbnNpb25sZXNzIHJhdGlvc1xuIgogICAgICAgICAgICAgICAgICAgICAgICAgICAiVmVydGljYWwgc2NhbGVzIHZhcnkgYnkgcGFuZWwiKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGV4cGxhbmF0aW9uID0gKCJVbndlaWdodGVkIGVycm9yIG5vcm1zXG4iCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJWZXJ0aWNhbCBzY2FsZXMgdmFyeSBieSBwYW5lbFxuXG4iCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJFdmVyeSB0ZXN0ZWQgbWVzaCBpcyBzaG93biIpCiAgICAgICAgaW5mby50ZXh0KDAuMDU1LCAwLjUwLCBleHBsYW5hdGlvbiwgdHJhbnNmb3JtPWluZm8udHJhbnNBeGVzLAogICAgICAgICAgICAgICAgICBoYT0ibGVmdCIsIHZhPSJ0b3AiLCBmb250c2l6ZT05LjMsIGxpbmVzcGFjaW5nPTEuNDUpCiAgICAgICAgZmlnLnN1cHRpdGxlKHRpdGxlLCBmb250c2l6ZT0xMiwgeT0wLjk5NSkKICAgICAgICBmaWcuc3VicGxvdHNfYWRqdXN0KGxlZnQ9MC4wNzUsIHJpZ2h0PTAuOTg1LCBib3R0b209MC4xMDUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3A9MC45MSwgaHNwYWNlPTAuNDcsIHdzcGFjZT0wLjQwKQogICAgICAgIHNhdmUoZmlnLCBvdXRwdXQsIGYib3Vfbm9uY29lcmNpdmVfe21ldHJpY30iLCBjcmVhdGVkKQoKCmRlZiBtYWtlX2ZpZ3VyZXMocmVzdWx0cz0icmVzdWx0cyIsIG91dHB1dD0iZmlndXJlcyIsIHNpemVzPURFRkFVTFRfU0laRVMpOgogICAgc2l6ZXMgPSB0dXBsZShpbnQobikgZm9yIG4gaW4gc2l6ZXMpCiAgICBpZiBub3Qgc2l6ZXMgb3IgbGVuKHNldChzaXplcykpICE9IGxlbihzaXplcykgb3IgdHVwbGUoc29ydGVkKHNpemVzKSkgIT0gc2l6ZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwZWN0ZWQgbWVzaCBzaXplcyBtdXN0IGJlIHVuaXF1ZSBhbmQgaW5jcmVhc2luZy4iKQogICAgcmVzdWx0cywgb3V0cHV0ID0gUGF0aChyZXN1bHRzKSwgUGF0aChvdXRwdXQpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGx0LnJjUGFyYW1zLnVwZGF0ZSh7CiAgICAgICAgImZvbnQuZmFtaWx5IjogIkRlamFWdSBTYW5zIiwgImZvbnQuc2l6ZSI6IDksCiAgICAgICAgImF4ZXMubGFiZWxzaXplIjogOSwgImF4ZXMudGl0bGVzaXplIjogMTAsCiAgICAgICAgInh0aWNrLmxhYmVsc2l6ZSI6IDgsICJ5dGljay5sYWJlbHNpemUiOiA4LAogICAgICAgICJheGVzLnNwaW5lcy50b3AiOiBGYWxzZSwgImF4ZXMuc3BpbmVzLnJpZ2h0IjogRmFsc2UsCiAgICAgICAgImxlZ2VuZC5mcmFtZW9uIjogRmFsc2UsICJsZWdlbmQuZm9udHNpemUiOiA5LAogICAgICAgICJwZGYuZm9udHR5cGUiOiA0MiwgInBzLmZvbnR0eXBlIjogNDIsCiAgICAgICAgInNhdmVmaWcuZHBpIjogMjIwLAogICAgfSkKICAgICMgVmFsaWRhdGUgYWxsIGlucHV0cyBiZWZvcmUgZW1pdHRpbmcgYW55IGZpZ3VyZTsgYW4gaW5jb21wbGV0ZSBydW4gbXVzdAogICAgIyBuZXZlciBzaWxlbnRseSBwcm9kdWNlIGEgcGxvdCB3aXRoIG1pc3NpbmcgbWVhc3VyZW1lbnRzLgogICAgZGF0YSA9IHtjYXNlOiByZWFkX3Jlc3VsdHMocmVzdWx0cyAvIGNhc2UgLyAiY29udmVyZ2VuY2UuY3N2Iiwgc2l6ZXMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmYW1pbHk9KGNhc2UgPT0gIm5vbmNvZXJjaXZlIikpCiAgICAgICAgICAgIGZvciBjYXNlIGluICgicG9zaXRpdmUiLCAibmVnYXRpdmUiLCAibm9uY29lcmNpdmUiKX0KICAgIGNyZWF0ZWQgPSBbXQogICAgbWFrZV9jb2VyY2l2ZV9maWd1cmVzKGRhdGFbInBvc2l0aXZlIl0sICJvdV9wb3NpdGl2ZSIsIHNpemVzLCBvdXRwdXQsIGNyZWF0ZWQpCiAgICBtYWtlX2NvZXJjaXZlX2ZpZ3VyZXMoZGF0YVsibmVnYXRpdmUiXSwgIm91X2JlIiwgc2l6ZXMsIG91dHB1dCwgY3JlYXRlZCkKICAgIG1ha2Vfbm9uY29lcmNpdmVfZmlndXJlcyhkYXRhWyJub25jb2VyY2l2ZSJdLCBzaXplcywgb3V0cHV0LCBjcmVhdGVkKQogICAgbWFuaWZlc3QgPSB7CiAgICAgICAgImV4cGVjdGVkX24iOiBzaXplcywgIm5vbmNvZXJjaXZlX2siOiBFWFBFQ1RFRF9LUywKICAgICAgICAiZXhhY3RfbDJfbm9ybSI6IEVYQUNUX0wyX05PUk0sICJleGFjdF9mdWxsX2gxX25vcm0iOiBFWEFDVF9IMV9OT1JNLAogICAgICAgICJlcnJvcl94X2F4aXMiOiAiU3ViZGl2aXNpb25zIHBlciBjb29yZGluYXRlLCBuIChmaW5lIG1lc2hlcyBvbiB0aGUgcmlnaHQpIiwKICAgICAgICAicnVudGltZV94X2F4aXMiOiAiU3ViZGl2aXNpb25zIHBlciBjb29yZGluYXRlLCBuIChmaW5lIG1lc2hlcyBvbiB0aGUgcmlnaHQpIiwKICAgICAgICAiaW5wdXRzIjogewogICAgICAgICAgICBjYXNlOiB7InJvd3MiOiBsZW4oZGF0YVtjYXNlXSksICJzaGEyNTYiOiBoYXNobGliLnNoYTI1NigKICAgICAgICAgICAgICAgIChyZXN1bHRzIC8gY2FzZSAvICJjb252ZXJnZW5jZS5jc3YiKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpfQogICAgICAgICAgICBmb3IgY2FzZSBpbiBkYXRhfSwKICAgICAgICAib3V0cHV0cyI6IFtQYXRoKG5hbWUpLm5hbWUgZm9yIG5hbWUgaW4gY3JlYXRlZF0sCiAgICB9CiAgICAob3V0cHV0IC8gImZpZ3VyZV9tYW5pZmVzdC5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKG1hbmlmZXN0LCBpbmRlbnQ9MikgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgcmV0dXJuIGNyZWF0ZWQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVzdWx0cy1yb290IiwgIi0tcmVzdWx0cyIsIGRlc3Q9InJlc3VsdHMiLCBkZWZhdWx0PSJyZXN1bHRzIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0IiwgZGVmYXVsdD0iZmlndXJlcyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWV4cGVjdGVkLXNpemVzIiwgdHlwZT1pbnQsIG5hcmdzPSIrIiwgZGVmYXVsdD1ERUZBVUxUX1NJWkVTLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJSZXF1aXJlZCBtZXNoIHNpemVzIChkZWZhdWx0OiBhbGwgc2V2ZW4gdGhyb3VnaCA1MTIpLiIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgZm9yIGZpbGVuYW1lIGluIG1ha2VfZmlndXJlcyhhcmdzLnJlc3VsdHMsIGFyZ3Mub3V0cHV0LCBhcmdzLmV4cGVjdGVkX3NpemVzKToKICAgICAgICBwcmludChmaWxlbmFtZSkK'}, 'code/ou_be_fem.py': {'sha256': '7c28c9842eab5ff120e7842c4cc8125584359fb549f6e1e18c2c3a9a8327ab3b', 'base64': '...'}, 'code/ou_noncoercive_fem.py': {'sha256': '951f65e282ad1ba3e0a67c14d2d5b336052ba908762937eaeae2f1064da2f381', 'base64': '...'}, 'code/ou_positive_fem.py': {'sha256': '41ae1f86063e82fcffce7e9d183958c6a61477add5391e8d870c012f60a8157d', 'base64': '...'}, 'code/templates/negative_numerical_prefix.tex': {'sha256': '257ca1247486535bd1337a333c9bac406af9ff9d7b45039177af8f6e873efdf1', 'base64': 'XHN1YnNlY3Rpb257TnVtZXJpY2FsIGNvbXBhcmlzb24gZm9yIHRoZSBjb2VyY2l2ZSBleGFtcGxlfQpcbGFiZWx7c2VjOm91LWJlLW51bWVyaWNhbC1leGFtcGxlfQoKV2UgY29tcGFyZSB0aGUgZmluaXRlIGVsZW1lbnQgYXBwcm94aW1hdGlvbnMgYmFzZWQgb24KJFxtYXRoY2FsIEIkIGFuZCAkXG1hdGhjYWwgRSQgZm9yClxiZWdpbntlcXVhdGlvbip9ClU9KC0xLDEpXjIsClxxcXVhZApBPVxiZWdpbntwbWF0cml4fTImMVxcMSYxXGVuZHtwbWF0cml4fSwKXHFxdWFkCkc9XGJlZ2lue3BtYXRyaXh9MCYxXFwtMiYtM1xlbmR7cG1hdHJpeH0uClxlbmR7ZXF1YXRpb24qfQpBcyBlc3RhYmxpc2hlZCBhYm92ZSwgYm90aCBiaWxpbmVhciBmb3JtcyBhcmUgY29lcmNpdmUKZm9yIHRoZXNlIGNvZWZmaWNpZW50cywgYWx0aG91Z2ggJFxvcGVyYXRvcm5hbWV7dHJhY2V9KEcpPS0zJC4KV2UgcHJlc2NyaWJlIHRoZSBleGFjdCBzb2x1dGlvbgpcYmVnaW57ZXF1YXRpb259XGxhYmVse2VxOm91LWJlLWV4YWN0LXNvbHV0aW9ufQp1KHhfMSx4XzIpPSgxLXhfMV4yKSgxLXhfMl4yKQpcZW5ke2VxdWF0aW9ufQphbmQgc2V0ICRmPS1MdSQuIEV4cGxpY2l0bHksClxiZWdpbntlcXVhdGlvbn1cbGFiZWx7ZXE6b3UtYmUtZm9yY2luZ30KZih4XzEseF8yKQo9My14XzFeMi04eF8yXjItNnhfMXhfMgotMnhfMXhfMl4zKzR4XzFeM3hfMis2eF8xXjJ4XzJeMi4KXGVuZHtlcXVhdGlvbn0KVGhlIGV4YWN0IHNvbHV0aW9uIHNhdGlzZmllcyB0aGUgaG9tb2dlbmVvdXMgRGlyaWNobGV0CmNvbmRpdGlvbiBvbiB0aGUgZW50aXJlIGJvdW5kYXJ5IG9mICRVJC4KRm9yIHRoZSB3ZWlnaHRlZCBmb3JtdWxhdGlvbiwgdGhlIGV4cGxpY2l0IGNvZWZmaWNpZW50cyBhcmUKXGJlZ2lue2VxdWF0aW9uKn0KUz0tXGZyYWMxezczfVxiZWdpbntwbWF0cml4fTMwJjM2XFwzNiY4N1xlbmR7cG1hdHJpeH0sClxxcXVhZApDPVxiZWdpbntwbWF0cml4fTAmNC8zXFwtNC8zJjBcZW5ke3BtYXRyaXh9LApcZW5ke2VxdWF0aW9uKn0KYW5kIGhlbmNlClxiZWdpbntlcXVhdGlvbip9ClxyaG8oeF8xLHhfMikKPVxleHBcbGVmdCgtXGZyYWN7MzB4XzFeMis3MnhfMXhfMis4N3hfMl4yfXs3M31ccmlnaHQpLgpcZW5ke2VxdWF0aW9uKn0KCldlIGRpdmlkZSBlYWNoIGNvb3JkaW5hdGUgaW50ZXJ2YWwgaW50byAkbiQgZXF1YWwgc3ViaW50ZXJ2YWxzCmFuZCBzcGxpdCBldmVyeSByZXN1bHRpbmcgc3F1YXJlIGFsb25nIGl0cyBzb3V0aHdlc3QtdG8tbm9ydGhlYXN0CmRpYWdvbmFsLiBCb3RoIG1ldGhvZHMgdXNlIHRoZSBzYW1lIGNvbnRpbnVvdXMgcGllY2V3aXNlIGFmZmluZQpzcGFjZSwgd2l0aApcYmVnaW57ZXF1YXRpb24qfQpuXGluXHs4LDE2LDMyLDY0LDEyOCwyNTYsNTEyXH0sClxxcXVhZApoPVxmcmFjezJcc3FydDJ9e259LApccXF1YWQKTl9oPShuLTEpXjIuClxlbmR7ZXF1YXRpb24qfQpGb3IgZWl0aGVyIGFwcHJveGltYXRpb24gJHVfaCQsIHdlIG1lYXN1cmUgdGhlIHVud2VpZ2h0ZWQgZXJyb3JzClxiZWdpbntlcXVhdGlvbip9CmVfMChoKT1cfHUtdV9oXHxfe0xeMihVKX0sClxxcXVhZAplXzEoaCk9XGxlZnQoClx8dS11X2hcfF97TF4yKFUpfV4yKwpcfFxuYWJsYSB1LVxuYWJsYSB1X2hcfF97TF4yKFUpfV4yClxyaWdodCleezEvMn0uClxlbmR7ZXF1YXRpb24qfQpUaHVzICRlXzEkIGlzIHRoZSBmdWxsICRIXnsxLDJ9KFUpJCBlcnJvci4KV2UgYWxzbyBtZWFzdXJlIHRoZSBjb3JyZXNwb25kaW5nIHJlbGF0aXZlIGVycm9ycwpcYmVnaW57ZXF1YXRpb259XGxhYmVse2VxOm91LWJlLXJlbGF0aXZlLWVycm9yc30KZV8wXntcbWF0aHJte3JlbH19KGgpCj1cZnJhY3tlXzAoaCl9e1x8dVx8X3tMXjIoVSl9fSwKXHFxdWFkCmVfMV57XG1hdGhybXtyZWx9fShoKQo9XGZyYWN7ZV8xKGgpfXtcfHVcfF97SF57MSwyfShVKX19LgpcZW5ke2VxdWF0aW9ufQpGb3IgdGhlIHByZXNjcmliZWQgZXhhY3Qgc29sdXRpb24sClxiZWdpbntlcXVhdGlvbn1cbGFiZWx7ZXE6b3UtYmUtZXhhY3Qtbm9ybXN9Clx8dVx8X3tMXjIoVSl9PVxmcmFjezE2fXsxNX0sClxxcXVhZApcfHVcfF97SF57MSwyfShVKX09XGZyYWN7MTZcc3FydDZ9ezE1fS4KXGVuZHtlcXVhdGlvbn0KSW5kZWVkLCAkXHx1XHxfe0xeMihVKX1eMj0yNTYvMjI1JCBhbmQKJFx8XG5hYmxhIHVcfF97TF4yKFUpfV4yPTI1Ni80NSQuClJlbGF0aXZlIGVycm9ycyBhcmUgcmVwb3J0ZWQgYXMgZGltZW5zaW9ubGVzcyByYXRpb3M7CmZvciBleGFtcGxlLCAkMTBeey0yfSQgY29ycmVzcG9uZHMgdG8gJDFcJSQuCgo='}, 'code/templates/noncoercive_numerical_prefix.tex': {'sha256': 'de05a4dfcdd1d7c5b01cabec5e51b0b7ed418f841d735146a0035ad71d307fb3', 'base64': 'XHN1YnNlY3Rpb257TnVtZXJpY2FsIGNvbXBhcmlzb24gaW4gdGhlIG5vbmNvZXJjaXZlIHJlZ2ltZX0KXGxhYmVse3N1YnNlYzpub25jb2VyY2l2ZS1udW1lcmljYWwtY29tcGFyaXNvbn0KCldlIGNvbXBhcmUgdGhlIG9yaWdpbmFsIGFuZCB3ZWlnaHRlZCBmaW5pdGUgZWxlbWVudApmb3JtdWxhdGlvbnMgZm9yClxiZWdpbntlcXVhdGlvbn1cbGFiZWx7bmNiZW5jaG1hcmtjb2VmZmljaWVudHN9ClU9KC0xLDEpXjIsXHFxdWFkCkE9XGJlZ2lue3BtYXRyaXh9MSYwXFwwJjJcZW5ke3BtYXRyaXh9LFxxcXVhZApHX2s9a1xiZWdpbntwbWF0cml4fS0xJjFcXC0xJi0yXGVuZHtwbWF0cml4fSxccXF1YWQKa1xpblx7NCw4LDEyLDE2LDIwXH0uClxlbmR7ZXF1YXRpb259CkFsbCBmaXZlIHBhcmFtZXRlciB2YWx1ZXMgZXhjZWVkIHRoZSBjb2VyY2l2aXR5CnRocmVzaG9sZCAkXGRpc3BsYXlzdHlsZVxmcmFje1xwaV4yfXs0fSQuCldlIGRlbm90ZSB0aGUgY29ycmVzcG9uZGluZyBvcmlnaW5hbCBhbmQgd2VpZ2h0ZWQKYmlsaW5lYXIgZm9ybXMgYnkgJFxtYXRoY2FsIEJfayQgYW5kICRcbWF0aGNhbCBFX2skLApyZXNwZWN0aXZlbHkuCgpGb3IgZXZlcnkgJGskLCB3ZSBwcmVzY3JpYmUgdGhlIHNhbWUgZXhhY3Qgc29sdXRpb24KXGJlZ2lue2VxdWF0aW9ufVxsYWJlbHtuY2JlbmNobWFya2V4YWN0c29sdXRpb259CnVfXHN0YXIoeF8xLHhfMik9KDEteF8xXjIpKDEteF8yXjIpClxlbmR7ZXF1YXRpb259CmFuZCBzZXQgJGZfaz0tTF9rIHVfXHN0YXIkLCB3aGVyZSAkTF9rJCBkZW5vdGVzCnRoZSBPcm5zdGVpbi0tVWhsZW5iZWNrIG9wZXJhdG9yIHdpdGggY29lZmZpY2llbnRzCiRBJCBhbmQgJEdfayQuIEV4cGxpY2l0bHksClxiZWdpbntlcXVhdGlvbn1cbGFiZWx7bmNiZW5jaG1hcmtmb3JjaW5nfQpcYmVnaW57YWxpZ25lZH0KZl9rKHhfMSx4XzIpCj17fSYzLSgyKzJrKXhfMV4yLSgxKzRrKXhfMl4yCiAgICAgICArNmsgeF8xXjJ4XzJeMlxcCiAgICYrMmsgeF8xXjN4XzItMmsgeF8xeF8yXjMuClxlbmR7YWxpZ25lZH0KXGVuZHtlcXVhdGlvbn0KVGhlIHdlaWdodGVkIGNvZWZmaWNpZW50cyBhcmUKXGJlZ2lue2VxdWF0aW9ufVxsYWJlbHtuY2JlbmNobWFya3dlaWdodGVkY29lZmZpY2llbnRzfQpTX2s9LWtJXzIsXHFxdWFkClxyaG9fayh4KT1lXnstayh4XzFeMit4XzJeMil9LFxxcXVhZApDPVxiZWdpbntwbWF0cml4fTAmMS8yXFwtMS8yJjBcZW5ke3BtYXRyaXh9LgpcZW5ke2VxdWF0aW9ufQpUaHVzIHRoZSB3ZWlnaHRlZCBtYXRyaXggaXMgYXNzZW1ibGVkIHVzaW5nCiRccmhvX2soQS8yK0MpJCBhbmQgdGhlIHdlaWdodGVkIGxvYWQgJFxyaG9fayBmX2skLgoKV2UgdXNlIGNvbnRpbnVvdXMgcGllY2V3aXNlIGFmZmluZSBlbGVtZW50cyBvbiB0aGUKc2FtZSB1bmlmb3JtIHRyaWFuZ3VsYXIgbWVzaGVzIGZvciBib3RoIG1ldGhvZHMuCkVhY2ggY29vcmRpbmF0ZSBpbnRlcnZhbCBpcyBkaXZpZGVkIGludG8gJG4kIGVxdWFsCnN1YmludGVydmFscywgYW5kIGVhY2ggc3F1YXJlIGlzIHNwbGl0IGFsb25nIGl0cwpzb3V0aHdlc3QtdG8tbm9ydGhlYXN0IGRpYWdvbmFsLiBXZSB0YWtlClxiZWdpbntlcXVhdGlvbip9Cm5caW5cezgsMTYsMzIsNjQsMTI4LDI1Niw1MTJcfSxccXF1YWQKaD1cZnJhY3syXHNxcnR7Mn19e259LFxxcXVhZCBOX2g9KG4tMSleMi4KXGVuZHtlcXVhdGlvbip9ClRoZSBlcnJvcnMgYXJlIG1lYXN1cmVkIGluIHRoZSB1bndlaWdodGVkIG5vcm1zClxiZWdpbntlcXVhdGlvbip9CmVfe0xeMn09XHx1X1xzdGFyLXVfaFx8X3tMXjIoVSl9LFxxcXVhZAplX3tIXnsxLDJ9fT0KXGxlZnQoClx8dV9cc3Rhci11X2hcfF97TF4yKFUpfV4yKwpcfFxuYWJsYSB1X1xzdGFyLVxuYWJsYSB1X2hcfF97TF4yKFUpfV4yClxyaWdodCleezEvMn0uClxlbmR7ZXF1YXRpb24qfQpXZSBhbHNvIHJlcG9ydCB0aGUgcmVsYXRpdmUgZXJyb3JzClxiZWdpbntlcXVhdGlvbn1cbGFiZWx7bmNiZW5jaG1hcmtyZWxhdGl2ZWVycm9yc30KZV97TF4yfV57XG1hdGhybXtyZWx9fQo9ClxmcmFje2Vfe0xeMn19e1x8dV9cc3Rhclx8X3tMXjIoVSl9fSwKXHFxdWFkCmVfe0heezEsMn19XntcbWF0aHJte3JlbH19Cj0KXGZyYWN7ZV97SF57MSwyfX19e1x8dV9cc3Rhclx8X3tIXnsxLDJ9KFUpfX0uClxlbmR7ZXF1YXRpb259CkZvciB0aGUgcHJlc2NyaWJlZCBleGFjdCBzb2x1dGlvbiwKXGJlZ2lue2VxdWF0aW9ufVxsYWJlbHtuY2JlbmNobWFya2V4YWN0bm9ybXN9Clx8dV9cc3Rhclx8X3tMXjIoVSl9PVxmcmFjezE2fXsxNX0sClxxcXVhZApcfHVfXHN0YXJcfF97SF57MSwyfShVKX09XGZyYWN7MTZcc3FydHs2fX17MTV9LgpcZW5ke2VxdWF0aW9ufQpUaGVzZSBpZGVudGl0aWVzIGZvbGxvdyBmcm9tCiRcfHVfXHN0YXJcfF97TF4yKFUpfV4yPTI1Ni8yMjUkIGFuZAokXHxcbmFibGEgdV9cc3Rhclx8X3tMXjIoVSl9XjI9MjU2LzQ1JC4KVGhlIHJlbGF0aXZlIGVycm9ycyBhcmUgZGltZW5zaW9ubGVzcyByYXRpb3M7CmZvciBleGFtcGxlLCAkMTBeey0yfSQgY29ycmVzcG9uZHMgdG8gJDFcJSQuCgo='}, 'code/templates/positive_numerical_prefix.tex': {'sha256': 'c8a08c96fd0e19d7dc9b181f4affd48776c02bc5ff85f755827bb583d761cb5f', 'base64': 'XHN1YnNlY3Rpb257TnVtZXJpY2FsIGNvbXBhcmlzb24gZm9yIHRoZSBwb3NpdGl2ZS10cmFjZSBleGFtcGxlfQpcbGFiZWx7c3Vic2VjOm91LXBvc2l0aXZlLW51bWVyaWNhbC1jb21wYXJpc29ufQoKV2UgY29tcGFyZSB0aGUgZmluaXRlIGVsZW1lbnQgYXBwcm94aW1hdGlvbnMgYmFzZWQgb24KJFxtYXRoY2FsIEIkIGFuZCAkXG1hdGhjYWwgRSQgZm9yClxiZWdpbntlcXVhdGlvbip9ClU9KC0xLDEpXjIsClxxcXVhZApBPVxiZWdpbntwbWF0cml4fTImMVxcMSYyXGVuZHtwbWF0cml4fSwKXHFxdWFkCkc9XGJlZ2lue3BtYXRyaXh9MyY0XFwxJjNcZW5ke3BtYXRyaXh9LgpcZW5ke2VxdWF0aW9uKn0KQm90aCBiaWxpbmVhciBmb3JtcyBhcmUgY29lcmNpdmUgZm9yIHRoZXNlIGNvZWZmaWNpZW50cywKYW5kICRcb3BlcmF0b3JuYW1le3RyYWNlfShHKT02JC4KV2UgcHJlc2NyaWJlIHRoZSBleGFjdCBzb2x1dGlvbgpcYmVnaW57ZXF1YXRpb259XGxhYmVse2VxOm91LXBvc2l0aXZlLWV4YWN0LXNvbHV0aW9ufQp1KHhfMSx4XzIpPSgxLXhfMV4yKSgxLXhfMl4yKQpcZW5ke2VxdWF0aW9ufQphbmQgc2V0ICRmPS1MdSQuIEV4cGxpY2l0bHksClxiZWdpbntlcXVhdGlvbn1cbGFiZWx7ZXE6b3UtcG9zaXRpdmUtZm9yY2luZ30KXGJlZ2lue2FsaWduZWR9CmYoeF8xLHhfMikKPXt9JjQrNHhfMV4yKzR4XzJeMis2eF8xeF8yXFwKICAgJi0xMnhfMV4yeF8yXjItOHhfMXhfMl4zLTJ4XzFeM3hfMi4KXGVuZHthbGlnbmVkfQpcZW5ke2VxdWF0aW9ufQpUaGUgZXhhY3Qgc29sdXRpb24gc2F0aXNmaWVzIHRoZSBob21vZ2VuZW91cyBEaXJpY2hsZXQKY29uZGl0aW9uIG9uIHRoZSBlbnRpcmUgYm91bmRhcnkgb2YgJFUkLgpGb3IgdGhlIHdlaWdodGVkIGZvcm11bGF0aW9uLCB3ZSB1c2UKXGJlZ2lue2VxdWF0aW9uKn0KUz1cZnJhYzEyXGJlZ2lue3BtYXRyaXh9MiYxXFwxJjNcZW5ke3BtYXRyaXh9LApccXF1YWQKQz1cYmVnaW57cG1hdHJpeH0wJi0xLzJcXDEvMiYwXGVuZHtwbWF0cml4fSwKXHFxdWFkClxyaG8oeF8xLHhfMik9ClxleHBcbGVmdCh4XzFeMit4XzF4XzIrXGZyYWMzMnhfMl4yXHJpZ2h0KS4KXGVuZHtlcXVhdGlvbip9ClRoZSB3ZWlnaHRlZCBtYXRyaXggYW5kIGxvYWQgYXJlIGFzc2VtYmxlZCB1c2luZwokXHJobyhBLzIrQykkIGFuZCAkXHJobyBmJCwgcmVzcGVjdGl2ZWx5LgoKV2UgZGl2aWRlIGVhY2ggY29vcmRpbmF0ZSBpbnRlcnZhbCBpbnRvICRuJCBlcXVhbCBzdWJpbnRlcnZhbHMKYW5kIHNwbGl0IGV2ZXJ5IHJlc3VsdGluZyBzcXVhcmUgYWxvbmcgaXRzIHNvdXRod2VzdC10by1ub3J0aGVhc3QKZGlhZ29uYWwuIEJvdGggbWV0aG9kcyB1c2UgdGhlIHNhbWUgY29udGludW91cyBwaWVjZXdpc2UgYWZmaW5lCnNwYWNlLCB3aXRoClxiZWdpbntlcXVhdGlvbip9Cm5caW5cezgsMTYsMzIsNjQsMTI4LDI1Niw1MTJcfSwKXHFxdWFkCmg9XGZyYWN7MlxzcXJ0Mn17bn0sClxxcXVhZApOX2g9KG4tMSleMi4KXGVuZHtlcXVhdGlvbip9CkZvciBlaXRoZXIgYXBwcm94aW1hdGlvbiAkdV9oJCwgd2UgbWVhc3VyZSB0aGUgdW53ZWlnaHRlZCBlcnJvcnMKXGJlZ2lue2VxdWF0aW9uKn0KZV8wKGgpPVx8dS11X2hcfF97TF4yKFUpfSwKXHFxdWFkCmVfMShoKT1cbGVmdCgKXHx1LXVfaFx8X3tMXjIoVSl9XjIrClx8XG5hYmxhIHUtXG5hYmxhIHVfaFx8X3tMXjIoVSl9XjIKXHJpZ2h0KV57MS8yfS4KXGVuZHtlcXVhdGlvbip9ClRodXMgJGVfMSQgaXMgdGhlIGZ1bGwgJEheezEsMn0oVSkkIGVycm9yLgpUaGUgY29ycmVzcG9uZGluZyByZWxhdGl2ZSBlcnJvcnMgYXJlClxiZWdpbntlcXVhdGlvbn1cbGFiZWx7ZXE6b3UtcG9zaXRpdmUtcmVsYXRpdmUtZXJyb3JzfQplXzBee1xtYXRocm17cmVsfX0oaCkKPVxmcmFje2VfMChoKX17XHx1XHxfe0xeMihVKX19LApccXF1YWQKZV8xXntcbWF0aHJte3JlbH19KGgpCj1cZnJhY3tlXzEoaCl9e1x8dVx8X3tIXnsxLDJ9KFUpfX0uClxlbmR7ZXF1YXRpb259CkZvciB0aGUgcHJlc2NyaWJlZCBleGFjdCBzb2x1dGlvbiwKXGJlZ2lue2VxdWF0aW9uKn0KXHx1XHxfe0xeMihVKX09XGZyYWN7MTZ9ezE1fSwKXHFxdWFkClx8dVx8X3tIXnsxLDJ9KFUpfT1cZnJhY3sxNlxzcXJ0Nn17MTV9LgpcZW5ke2VxdWF0aW9uKn0KUmVsYXRpdmUgZXJyb3JzIGFyZSByZXBvcnRlZCBhcyBkaW1lbnNpb25sZXNzIHJhdGlvczsKZm9yIGV4YW1wbGUsICQxMF57LTJ9JCBjb3JyZXNwb25kcyB0byAkMVwlJC4KCg=='}, 'code/templates/sections_template.tex': {'sha256': '81fde24c90758af3d9f2af62c95cb8b58af68a79eabc8d01695d6889e5fa09d3', 'base64': '...'}, 'code/verify_results.py': {'sha256': '1b06ced62c343b086a23ec35e97af8bc877524d9f298b3e7a5ab71b9152374e2', 'base64': '...'}}
RECORDED_PAYLOAD = {'results/independent_validation.json': {'sha256': 'dbcd64da31ffed482193dc1f314b15a2291e11f4d8c44073e5c77a1c0b0b1a63', 'base64': '...'}, 'results/negative/convergence.csv': {'sha256': '9207be71cb5bbefa04aa289f38fbaaf58e02113f98ef22ecbd529fc8289010a1', 'base64': 'bixoLG5kb2YsdHJpYW5nbGVzLG1ldGhvZCxsMixoMSxsMl9yZWxhdGl2ZSxoMV9yZWxhdGl2ZSxoMV9zZW1pbm9ybSxjb2VmZl9tZWRpYW5fcyxhc3NlbWJseV9tZWRpYW5fcyxzb2x2ZV9tZWRpYW5fcyx0b3RhbF9tZWRpYW5fcyx0b3RhbF9xMjVfcyx0b3RhbF9xNzVfcyxtaW5fdG90YWxfcyxtYXhfdG90YWxfcyxzY2FsZWRfYWxnZWJyYWljX3Jlc2lkdWFsLG1hdHJpeF9ubnoNCjgsMC4zNTM1NTMzOTA1OTMyNzM4LDQ5LDEyOCxCLDAuMDM3MjU5MzA3NDY5MTI0MywwLjQ4NDgwMjc0NjcyNzA0MTcsMC4wMzQ5MzA2MDA3NTIzMDQwMywwLjE4NTU0OTg5OTI3ODI4MTIyLDAuNDgzMzY4ODUyMTYyNjE2NTUsMC4wLDAuMDAyMjU3NjY3MDAxNjg2MDU5LDAuMDAwMTMzOTEwOTk0OTE4NTUxMywwLjAwMjQxMTIxNjAwMjY1NjE0NywwLjAwMjMxNzQ5MDk5ODM5MzYyMywwLjAwMjcyMDY5MDQ5OTcwMDE0LDAuMDAyMTQ5MzAyOTk5ODk2NDYzLDAuMDA0MDAyNTQyOTk4OTMyNzY4LDEuNTg1NzQ5NzQyMDc3NDc0ZS0xNiwyODkNCjgsMC4zNTM1NTMzOTA1OTMyNzM4LDQ5LDEyOCxFLDAuMDMxNzA3MDM3MTU4NjA1NjUsMC40ODY4Njg1MzcyODA5MzQ1LDAuMDI5NzI1MzQ3MzM2MTkyOCwwLjE4NjM0MDU0NTAyMzk4NDE0LDAuNDg1ODM0OTg4ODQ3NzU2Myw4LjAyMjAwMDExMDk0MTAwOGUtMDUsMC4wMDI0OTY3MTUwMDIwNTI5NTE2LDAuMDAwMTI2ODkwMDAxNDExMDg0MDksMC4wMDI3MDQ5ODcwMDQxMDgyNjEzLDAuMDAyNTc3OTc2NTAxOTI1MTUsMC4wMDI3MTQxNzYwMDM3OTIwOTAzLDAuMDAyMzI4MzkyMDAxODAwMjM5LDAuMDAzMDk3MTA1MDAwMDUxNjY5OCwxLjQwMjk2NDcyMTIzMTA3MzhlLTE2LDI4OQ0KMTYsMC4xNzY3NzY2OTUyOTY2MzY5LDIyNSw1MTIsQiwwLjAwOTM3NDk5MjEyOTc3MDA0LDAuMjQzMTc3MzEwNzI3MDAyOTcsMC4wMDg3ODkwNTUxMjE2NTk0MTQsMC4wOTMwNzE5MjYyOTc0MDQ5MywwLjI0Mjk5NjUzMDc4Nzk2MDIsMC4wLDAuMDA3NTkzMTQ4MDAyMzQyNzg3LDAuMDAwNDQxNDIyMDAyMDQwNzgxMSwwLjAwODAzNDU3MDAwNDM4MzU2OCwwLjAwNzU1NjY0ODQ5OTMwMzI2NDUsMC4wMDg0MTQxNjk1MDA2MTEwMjQsMC4wMDcxMzIzMDgwMDQ2NTY4MDcsMC4wMDk5NDA5MjEwMDE1NDM3NTcsMS4yODk1MDQwMzk5MjA5NzUyZS0xNiwxNDU3DQoxNiwwLjE3Njc3NjY5NTI5NjYzNjksMjI1LDUxMixFLDAuMDA3OTU3MDQyNjgwNjUzNjkyLDAuMjQzNDc5MDMxMzMwNDg1NTQsMC4wMDc0NTk3Mjc1MTMxMTI4MzYsMC4wOTMxODc0MDQ2NjA0Mzg4MiwwLjI0MzM0ODk3NjEwMTAwOTc1LDguMjk4NTAwMzI0NTk5NDQ1ZS0wNSwwLjAwODA0ODE1MDk5NDAwMDAyNSwwLjAwMDQxNzYzNjAwMTY1NzIyODksMC4wMDg2MjE2MjEwMDM4NDAxMTksMC4wMDg0NTUyNzY1MDAxNzE3NDksMC4wMDkwOTg0NTE1MDA5MDQwNDgsMC4wMDgwOTYwOTI5OTkwMzk2ODMsMC4wMDk2NTUwMzA5OTcyMzcxOSwxLjU0NTQ0NTAzMDczODE0MWUtMTYsMTQ1Nw0KMzIsMC4wODgzODgzNDc2NDgzMTg0NSw5NjEsMjA0OCxCLDAuMDAyMzQ3NjA2NzY2MjkyNzQyLDAuMTIxNjg0MzU0NjEwMzc3ODMsMC4wMDIyMDA4ODEzNDMzOTk0NDU2LDAuMDQ2NTcyNTkwMzg2NzY3NDE0LDAuMTIxNjYxNzA2NzkxNDc1ODIsMC4wLDAuMDMyNDg0MzExOTk2MDg2MDM1LDAuMDAyMTY5ODM1MDAyODYwMDU0NCwwLjAzNDI3MjIxMzAwMDgwNDE4NiwwLjAzMjkyNDQxNjk5ODg2Njk0NiwwLjAzNTUyODE2NjQ5OTcxNjQ2LDAuMDMwMTMxODIyOTk3NjQ1OTQsMC4wMzc3NDI4ODgwMDE5NTc5MSwyLjUxMDU3ODA0MzE1ODQ4NjNlLTE2LDY0ODENCjMyLDAuMDg4Mzg4MzQ3NjQ4MzE4NDUsOTYxLDIwNDgsRSwwLjAwMTk5MTA0Njg5OTQ2OTg4OTQsMC4xMjE3MjM4Mjk2NzMxMzczMywwLjAwMTg2NjYwNjQ2ODI1MzAyMTIsMC4wNDY1ODc2OTg3ODY5NzQ0NiwwLjEyMTcwNzU0NDcyMzE1NjE2LDkuODk0ODAwMzE0MjE0MDhlLTA1LDAuMDMyNDE1Mzc4MDAyMDQyNDE2LDAuMDAyMTAwNjMxMDA1MzA1MzMxLDAuMDM0NjI0MTMwOTk2MjM1MTgsMC4wMzM3MjM5NDU1MDM4Nzk4MDQsMC4wMzcxNjE3NjE5OTY2ODU1MywwLjAzMTI5OTYwNDAwMTEwNTc2LDAuMDQ2NDE5NjQxOTk5ODE4ODUsMS43NTE4NjI1NDUzMTAwMTU2ZS0xNiw2NDgxDQo2NCwwLjA0NDE5NDE3MzgyNDE1OTIyLDM5NjksODE5MixCLDAuMDAwNTg3MTQzMzg3NjQwODM2OCwwLjA2MDg1NDA5NDM3MjYzOTczLDAuMDAwNTUwNDQ2OTI1OTEzMjg0NSwwLjAyMzI5MDg1NjI0NTY0Njk2LDAuMDYwODUxMjYxODE1NjQ3NiwwLjAsMC4xMzc2MDc1NzA5OTk1MzQ4LDAuMDExODU5OTc5MDAwMjkxNzg3LDAuMTQ5NjAwNTk5MDA0NzcwNjMsMC4xNDQzMjQ5ODEwMDM1OTIyNiwwLjE1MzYyNjEzMDk5OTMyNzg3LDAuMTI0OTI2ODcxOTk4NDY4NDEsMC4xNzI0NjM5MTU5OTQ3NDAxLDMuNDcxOTUwMzE3ODIzMDJlLTE2LDI3MjgxDQo2NCwwLjA0NDE5NDE3MzgyNDE1OTIyLDM5NjksODE5MixFLDAuMDAwNDk3ODcyNjU4NzEwODUwNywwLjA2MDg1OTA4ODQyOTA1MTkxLDAuMDAwNDY2NzU1NjE3NTQxNDIyNiwwLjAyMzI5Mjc2NzYzNDcwMjY2OCwwLjA2MDg1NzA1MTkxMDQ0NzIzLDAuMDAwMTEwNzY3MDAzMjA3MDAxODQsMC4xNDY3ODQ3NDQ5OTY2OTM5MiwwLjAxMjgxNDg2MDAwNTE2NDUxOSwwLjE1ODAxNjk4MjAwMDg0NTAyLDAuMTUzMDEyOTAzNDk4MTY1NTQsMC4xNjY0ODA0NDI5OTY1NDQzLDAuMTQ1NTgxOTExMDA1MTM3OTUsMC4xNjg2MjQ1NzIwMDQ3NzM2NSw0Ljk2MDI0ODYwMTQ2NjUyNWUtMTYsMjcyODENCjEyOCwwLjAyMjA5NzA4NjkxMjA3OTYxLDE2MTI5LDMyNzY4LEIsMC4wMDAxNDY4MDA5NDEzNDcxMjYsMC4wMzA0Mjg1MzUxNTY3MzMwNzgsMC4wMDAxMzc2MjU4ODI1MTI5MzA2MiwwLjAxMTY0NTk5NzYxNzg2NDg0NCwwLjAzMDQyODE4MTAzNzc4NDA0NSwwLjAsMC40NjQ1MjQ5MjE5OTc1MDg1LDAuMDY1NDY0MjIzMDAwMTYyNzUsMC41Mjk5ODkxNDQ5OTc2NzEyLDAuNTIwNTk0NjUxMDAzMTkzNiwwLjUzNzkwNjU4Mjk5ODQwMjksMC41MDI5NzE4MzM5OTM3OTcyLDAuNTcyMzE4NjEyMDAxMjU0Miw0Ljc5ODAyNjg5NDQ3NTg0MmUtMTYsMTExODg5DQoxMjgsMC4wMjIwOTcwODY5MTIwNzk2MSwxNjEyOSwzMjc2OCxFLDAuMDAwMTI0NDc1MTQ5NjA5NDUzMSwwLjAzMDQyOTE2MTMxODI0MDE1NiwwLjAwMDExNjY5NTQ1Mjc1ODg2MjMsMC4wMTE2NDYyMzcyNzAzOTQzNjIsMC4wMzA0Mjg5MDY3MjQ4MzM0MjUsOC44MzkyMDAwMTc5NzM3OGUtMDUsMC41MDA1NjEyNjI5OTc1MDc2LDAuMDYxOTg1NTY1MDAwMjEzNjgsMC41NjE0Njk3NzMwMDUyMjg1LDAuNTU3MzQ3ODcwNTA0NiwwLjU3MjMwMzk3OTUwMTE1NzMsMC41NTMwMjEzMTY5OTg2MzgyLDAuNTgyMjQ1MTc2MDAzMTk4MSw0LjM2NjcyNjEwNjExNzU1ODRlLTE2LDExMTg4OQ0KMjU2LDAuMDExMDQ4NTQzNDU2MDM5ODA2LDY1MDI1LDEzMTA3MixCLDMuNjcwMTE3NzkyMTUyNTkyZS0wNSwwLjAxNTIxNDQ1MzUxMzkyNDUsMy40NDA3MzU0MzAxNDMwNTVlLTA1LDAuMDA1ODIzMDY5OTcyNTYzOTkxLDAuMDE1MjE0NDA5MjQ3NTE2Mjc1LDAuMCwxLjg4MDIzMDEyODAwMTEzMDcsMC40NDY1NDMxMTU5OTg3OTI5LDIuMzI0MTMwMDM1OTk1MDQsMi4yMzYzMjUxODU1MDIyMSwyLjM2NDk4MzU0MzQ5ODk2NTQsMi4xMjk2MTUwNTUwMDQyNTEsMi41ODkwNDE2NzUwMDQ0MDk2LDUuNjg2MzE1NDE1MDc3NzAxZS0xNiw0NTMxMzcNCjI1NiwwLjAxMTA0ODU0MzQ1NjAzOTgwNiw2NTAyNSwxMzEwNzIsRSwzLjExMTkyMjc0Mzc5MTUzN2UtMDUsMC4wMTUyMTQ1MzE4NDM4MzA1MDksMi45MTc0Mjc1NzIzMDQ1NjU4ZS0wNSwwLjAwNTgyMzA5OTk1MTk4NjA3MywwLjAxNTIxNDUwMDAxODc1MjM4MiwwLjAwMDEwNTczNzk5OTU0NzI3MjkyLDIuMDI0NTczOTQ4OTk4Mjk1MiwwLjQ0Njc3OTgyNDAwMDI1OTQsMi40NzIwNTg1MTA5OTcxNTgsMi4zNjk0MDE4MTcwMDI2NTQsMi41MTE2OTY4OTEwMDA2MTA4LDIuMjY1NjYxODQ5OTk2ODMxNywyLjU3NDI4MzkzMTAwNDE3NDUsNS43MjAwMTc2NzU2NDg2NzRlLTE2LDQ1MzEzNw0KNTEyLDAuMDA1NTI0MjcxNzI4MDE5OTAzLDI2MTEyMSw1MjQyODgsQiw5LjE3NTM1MzQ4NzEzMjg3M2UtMDYsMC4wMDc2MDcyNDk5OTY3NDU5NDM2LDguNjAxODkzODk0MTg3MDY5ZS0wNiwwLjAwMjkxMTU0Mzg4MDkwODgzNjcsMC4wMDc2MDcyNDQ0NjMzOTY2ODcsMC4wLDcuMTkzOTI0NjU3OTkzMTkzLDQuMzAzOTIwOTgwMDA0NDc3LDExLjUxMjI5MjQzMjAwMjE0MSwxMS4yNTUxNzA3NTM1MDIxNywxMS41OTAyNzcxNzY5OTc5MTUsMTAuNzYzMDMzNTc1MDAwMDA5LDExLjgyMzA5NzMwNTk5ODA5LDkuMTE3NDcyMTM2NzkxODE2ZS0xNiwxODIzNzYxDQo1MTIsMC4wMDU1MjQyNzE3MjgwMTk5MDMsMjYxMTIxLDUyNDI4OCxFLDcuNzc5ODM0MzY5ODYyODk5ZS0wNiwwLjAwNzYwNzI1OTc4OTg0ODcwNSw3LjI5MzU5NDcyMTc0NjQ2OGUtMDYsMC4wMDI5MTE1NDc2MjkwNTAyMDY1LDAuMDA3NjA3MjU1ODExNjg1NzE5LDAuMDAwMTAwNzIxMDAxMjE3NjA3NDEsNy44MDY0ODY0MjQwMDA0NjgsNC4xMTc4NDk2NzcwMDA3NzcsMTIuMTUxNDI3MDE3MDAwOTcsMTEuODM0NjI0MDY4MDAwMjY3LDEyLjM3MzU4NTAzOTUwMjM1OSwxMS42MDg5NjAyNjgwMDAzNzQsMTIuNDU1NzczODk5MDA0MjcsMS4wNzg5OTU0ODQ0MTI4MTc5ZS0xNSwxODIzNzYxDQo='}, 'results/negative/metadata.json': {'sha256': '519a5cf64d48c412f8d2521913d78e62dc9e06273af5cbe89144624084246078', 'base64': 'ewogICJzdGFydGVkX3V0YyI6ICIyMDI2LTEwLTE2VDE0OjI2OjI2LjMxNjA5OCswMDowMCIsCiAgImNvbXBsZXRlZF91dGMiOiAiMjAyNi0xMC0xNlQxNDozMDozNS4xMDMyMzErMDA6MDAiLAogICJzb3VyY2Vfc2hhMjU2IjogIjdjMjhjOTg0MmVhYjVmZjEyMGU3ODQyYzRjYzgxMjU1ODQzNTlmYjU0OWY2ZTFlMThjMmMzYTlhODMyN2FiM2IiLAogICJkZXNjcmlwdGlvbiI6ICJQMSBGRU06IG9yaWdpbmFsIEIgdmVyc3VzIGVxdWl2YWxlbnQgd2VpZ2h0ZWQgRSwgc2FtZSBtZXNoIGFuZCBzcGFyc2UgTFUiLAogICJleGVjdXRpb25fZW52aXJvbm1lbnQiOiAiYWN0dWFsIG1hY2hpbmUgcnVubmluZyB0aGlzIHNjcmlwdDsgbm90IG5lY2Vzc2FyaWx5IEdvb2dsZSBDb2xhYiIsCiAgInB5dGhvbiI6ICIzLjEyLjE0IChtYWluLCBBdWcgMjUgMjAyNiwgMTQ6MDA6NDkpIFtDbGFuZyAyMi4xLjMgXSIsCiAgIm51bXB5IjogIjIuMy41IiwKICAic2NpcHkiOiAiMS4xNy4wIiwKICAicGxhdGZvcm0iOiAiTGludXgtNi4xOC40NC14ODZfNjQtd2l0aC1nbGliYzIuMzkiLAogICJjcHUiOiAiQU1EIEVQWUMgOVY3NCA4MC1Db3JlIFByb2Nlc3NvciIsCiAgImxvZ2ljYWxfY3B1X2NvdW50IjogOSwKICAibnVtZXJpY2FsX3RocmVhZHMiOiAxLAogICJ0aHJlYWRwb29sX2luZm9fZHVyaW5nX3J1biI6IFsKICAgIHsKICAgICAgInVzZXJfYXBpIjogImJsYXMiLAogICAgICAiaW50ZXJuYWxfYXBpIjogIm9wZW5ibGFzIiwKICAgICAgIm51bV90aHJlYWRzIjogMSwKICAgICAgInByZWZpeCI6ICJsaWJzY2lweV9vcGVuYmxhcyIsCiAgICAgICJmaWxlcGF0aCI6ICIvb3B0L2NvZGV4L3J1bnRpbWVzL2NvZGV4LXByaW1hcnktcnVudGltZS9kZXBlbmRlbmNpZXMvcHl0aG9uL2xpYi9weXRob24zLjEyL3NpdGUtcGFja2FnZXMvbnVtcHkubGlicy9saWJzY2lweV9vcGVuYmxhczY0Xy1mZGRlNTc3OC5zbyIsCiAgICAgICJ2ZXJzaW9uIjogIjAuMy4zMCIsCiAgICAgICJ0aHJlYWRpbmdfbGF5ZXIiOiAicHRocmVhZHMiLAogICAgICAiYXJjaGl0ZWN0dXJlIjogIlNreWxha2VYIgogICAgfSwKICAgIHsKICAgICAgInVzZXJfYXBpIjogImJsYXMiLAogICAgICAiaW50ZXJuYWxfYXBpIjogIm9wZW5ibGFzIiwKICAgICAgIm51bV90aHJlYWRzIjogMSwKICAgICAgInByZWZpeCI6ICJsaWJzY2lweV9vcGVuYmxhcyIsCiAgICAgICJmaWxlcGF0aCI6ICIvb3B0L2NvZGV4L3J1bnRpbWVzL2NvZGV4LXByaW1hcnktcnVudGltZS9kZXBlbmRlbmNpZXMvcHl0aG9uL2xpYi9weXRob24zLjEyL3NpdGUtcGFja2FnZXMvc2NpcHkubGlicy9saWJzY2lweV9vcGVuYmxhcy02Y2RjM2I0YS5zbyIsCiAgICAgICJ2ZXJzaW9uIjogIjAuMy4zMCIsCiAgICAgICJ0aHJlYWRpbmdfbGF5ZXIiOiAicHRocmVhZHMiLAogICAgICAiYXJjaGl0ZWN0dXJlIjogIlNreWxha2VYIgogICAgfQogIF0sCiAgIkEiOiBbCiAgICBbCiAgICAgIDIuMCwKICAgICAgMS4wCiAgICBdLAogICAgWwogICAgICAxLjAsCiAgICAgIDEuMAogICAgXQogIF0sCiAgIkciOiBbCiAgICBbCiAgICAgIDAuMCwKICAgICAgMS4wCiAgICBdLAogICAgWwogICAgICAtMi4wLAogICAgICAtMy4wCiAgICBdCiAgXSwKICAiY29lZmZpY2llbnRzIjogewogICAgIlMiOiBbCiAgICAgIFsKICAgICAgICAtMC40MTA5NTg5MDQxMDk1ODksCiAgICAgICAgLTAuNDkzMTUwNjg0OTMxNTA2OAogICAgICBdLAogICAgICBbCiAgICAgICAgLTAuNDkzMTUwNjg0OTMxNTA2OCwKICAgICAgICAtMS4xOTE3ODA4MjE5MTc4MDgKICAgICAgXQogICAgXSwKICAgICJIIjogWwogICAgICBbCiAgICAgICAgMS4zMTUwNjg0OTMxNTA2ODQ5LAogICAgICAgIDMuMTc4MDgyMTkxNzgwODIxNwogICAgICBdLAogICAgICBbCiAgICAgICAgLTEuMDk1ODkwNDEwOTU4OTA0LAogICAgICAgIC0xLjMxNTA2ODQ5MzE1MDY4NQogICAgICBdCiAgICBdLAogICAgIkMiOiBbCiAgICAgIFsKICAgICAgICAwLjAsCiAgICAgICAgMS4zMzMzMzMzMzMzMzMzMzMzCiAgICAgIF0sCiAgICAgIFsKICAgICAgICAtMS4zMzMzMzMzMzMzMzMzMzMzLAogICAgICAgIDAuMAogICAgICBdCiAgICBdLAogICAgIk0iOiBbCiAgICAgIFsKICAgICAgICAxLjAsCiAgICAgICAgMS44MzMzMzMzMzMzMzMzMzMzCiAgICAgIF0sCiAgICAgIFsKICAgICAgICAtMC44MzMzMzMzMzMzMzMzMzMzLAogICAgICAgIDAuNQogICAgICBdCiAgICBdCiAgfSwKICAiZXhhY3Rfc29sdXRpb24iOiAiKDEteF4yKSgxLXleMikiLAogICJmb3JjaW5nIjogIjMteF4yLTgqeV4yLTYqeCp5LTIqeCp5XjMrNCp4XjMqeSs2KnheMip5XjIiLAogICJkb21haW4iOiAiKC0xLDEpXjIiLAogICJtZXNoIjogInVuaWZvcm0gbi1ieS1uIHNxdWFyZXM7IHNvdXRod2VzdC10by1ub3J0aGVhc3QgZGlhZ29uYWxzIiwKICAic2l6ZXMiOiBbCiAgICA4LAogICAgMTYsCiAgICAzMiwKICAgIDY0LAogICAgMTI4LAogICAgMjU2LAogICAgNTEyCiAgXSwKICAicmVwZXRpdGlvbnMiOiA3LAogICJ3YXJtdXBzX3Blcl9tZXNoX3Blcl9tZXRob2QiOiAxLAogICJhc3NlbWJseV9kdWZmeV9vcmRlciI6IDgsCiAgImVycm9yX2R1ZmZ5X29yZGVyIjogOCwKICAiYmF0Y2hfc2l6ZSI6IDQwOTYsCiAgInNvbHZlciI6ICJzY2lweS5zcGFyc2UubGluYWxnLnNwbHU7IFN1cGVyTFU7IHBlcm1jX3NwZWM9Q09MQU1EOyBkaWFnX3Bpdm90X3RocmVzaD0xLjAiLAogICJ0aW1pbmdfdW5pdCI6ICJzZWNvbmRzIiwKICAidGltaW5nX3N0YXRpc3RpYyI6ICJtZWRpYW4gb2YgdG90YWwgdGltZSBvdmVyIHJlcGV0aXRpb25zIiwKICAidGltaW5nX2luY2x1ZGVzIjogWwogICAgIkU6IGNvbXB1dGUgUyxILEMsTTsgQjogY29lZmZpY2llbnQgcHJlcGFyYXRpb249MCIsCiAgICAiZ2VvbWV0cnkgYW5kIG1hdHJpeC9sb2FkIGFzc2VtYmx5LCBpbmNsdWRpbmcgZGVuc2l0eSBldmFsdWF0aW9uIiwKICAgICJzcGFyc2UgbWF0cml4IGNvbnN0cnVjdGlvbiBhbmQgaG9tb2dlbmVvdXMgRGlyaWNobGV0IGV4dHJhY3Rpb24iLAogICAgImZyZXNoIHNwYXJzZSBMVSBmYWN0b3JpemF0aW9uIGFuZCB0cmlhbmd1bGFyIHNvbHZlcyIKICBdLAogICJ0aW1pbmdfZXhjbHVkZXMiOiBbCiAgICAiY29tbW9uIG1lc2gvY29ubmVjdGl2aXR5IGNyZWF0aW9uIiwKICAgICJxdWFkcmF0dXJlLXJ1bGUgY29uc3RydWN0aW9uIiwKICAgICJ3YXJtdXAiLAogICAgImdhcmJhZ2UgY29sbGVjdGlvbiIsCiAgICAiZ2xvYmFsIG5vZGFsIG91dHB1dCBleHBhbnNpb24iLAogICAgImVycm9yL3Jlc2lkdWFsIGV2YWx1YXRpb24iLAogICAgIm91dHB1dCBhbmQgcGxvdHRpbmciCiAgXSwKICAiY29tcG9uZW50X21lZGlhbnNfbm90ZSI6ICJFYWNoIGNvbXBvbmVudCBpcyBtZWRpYW4gc2VwYXJhdGVseTsgdGhlaXIgc3VtIG5lZWQgbm90IGVxdWFsIG1lZGlhbiB0b3RhbC4iLAogICJIMV9lcnJvcl9kZWZpbml0aW9uIjogInNxcnQoTDJfZXJyb3JeMiArIGdyYWRpZW50X0wyX2Vycm9yXjIpLCB1bndlaWdodGVkIiwKICAicmVsYXRpdmVfZXJyb3JfZGVmaW5pdGlvbiI6ICJsMi8oMTYvMTUpLCBoMS8oMTYqc3FydCg2KS8xNSk7IGRpbWVuc2lvbmxlc3MgcmF0aW9zIiwKICAiY29lZmZpY2llbnRfY29uc3RydWN0aW9uIjogIkV4cGFuZGVkIHBsYW5hciBTIGZvcm11bGE7IEM9W1swLC1ldGEvKDIqdGF1KV0sW2V0YS8oMip0YXUpLDBdXSwgbm8gUyBpbnZlcnNlIiwKICAidmFsaWRhdGlvbiI6IHsKICAgICJjb2VmZmljaWVudF9pZGVudGl0eV9kZWZlY3RzIjogewogICAgICAiU19zeW1tZXRyeSI6IDAuMCwKICAgICAgInJpY2NhdGkiOiAyLjIyMDQ0NjA0OTI1MDMxM2UtMTYsCiAgICAgICJ0cmFjZSI6IDAuMCwKICAgICAgIkNfYW50aXN5bW1ldHJ5IjogMC4wLAogICAgICAicG90ZW50aWFsIjogNC45NjUwNjgzMDY0OTQ1NDZlLTE2LAogICAgICAiZmFjdG9yaXphdGlvbiI6IDQuMDY4OTg2NTEyNzYxNTMzNmUtMTYsCiAgICAgICJTX2V4YWN0IjogMi4yMjA0NDYwNDkyNTAzMTNlLTE2LAogICAgICAiQ19leGFjdCI6IDAuMAogICAgfSwKICAgICJtYW51ZmFjdHVyZWRfZm9yY2luZ19tYXhfYWJzb2x1dGVfZGVmZWN0IjogMS43NzYzNTY4Mzk0MDAyNTA1ZS0xNSwKICAgICJ3ZWlnaHRlZF9kcmlmdF9wb3RlbnRpYWxfcmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiAxLjk1NDg5MDYxMDY2ODQ4NjhlLTE2LAogICAgIm9yaWdpbmFsX2VuZXJneV9yZWxhdGl2ZV9kZWZlY3QiOiAyLjg4NzA4NjMwMDUxMzM0ODRlLTE2LAogICAgImV4YWN0X3NvbHV0aW9uX25vcm1fc3F1YXJlZF9kZWZlY3RzIjogewogICAgICAiTDJfc3F1YXJlZCI6IDQuNDQwODkyMDk4NTAwNjI2ZS0xNiwKICAgICAgImdyYWRpZW50X3NxdWFyZWQiOiAwLjAsCiAgICAgICJIMV9zcXVhcmVkIjogOC44ODE3ODQxOTcwMDEyNTJlLTE2CiAgICB9LAogICAgInF1YWRyYXR1cmVfY2hlY2tzIjogWwogICAgICB7CiAgICAgICAgIm4iOiA4LAogICAgICAgICJmb3JtdWxhdGlvbiI6ICJCIiwKICAgICAgICAiYmFzZWxpbmVfb3JkZXIiOiA4LAogICAgICAgICJoaWdoZXJfb3JkZXIiOiAxMiwKICAgICAgICAicmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiAwLjAsCiAgICAgICAgInJlbGF0aXZlX2xvYWRfZGlmZmVyZW5jZSI6IDMuMjMwMTM1MTg0MjMzOTY4NWUtMTYsCiAgICAgICAgInJlbGF0aXZlX25vZGFsX2RpZmZlcmVuY2UiOiAzLjg1NzU3MzU0MjUyMzM4MDZlLTE2CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAibiI6IDgsCiAgICAgICAgImZvcm11bGF0aW9uIjogIkUiLAogICAgICAgICJiYXNlbGluZV9vcmRlciI6IDgsCiAgICAgICAgImhpZ2hlcl9vcmRlciI6IDEyLAogICAgICAgICJyZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDMuMDE2NjgwNzc5MDM3MjI0ZS0xNiwKICAgICAgICAicmVsYXRpdmVfbG9hZF9kaWZmZXJlbmNlIjogMy4yMTQ1Mzc0NzUzOTAxMTk4ZS0xNiwKICAgICAgICAicmVsYXRpdmVfbm9kYWxfZGlmZmVyZW5jZSI6IDUuNjM4MzAzMDkzNzA2NDUyZS0xNgogICAgICB9LAogICAgICB7CiAgICAgICAgIm4iOiAzMiwKICAgICAgICAiZm9ybXVsYXRpb24iOiAiQiIsCiAgICAgICAgImJhc2VsaW5lX29yZGVyIjogOCwKICAgICAgICAiaGlnaGVyX29yZGVyIjogMTIsCiAgICAgICAgInJlbGF0aXZlX21hdHJpeF9kaWZmZXJlbmNlIjogMC4wLAogICAgICAgICJyZWxhdGl2ZV9sb2FkX2RpZmZlcmVuY2UiOiAyLjc0MTc4OTYzNzAxMTU2ZS0xNiwKICAgICAgICAicmVsYXRpdmVfbm9kYWxfZGlmZmVyZW5jZSI6IDMuMzIxMzIxMjc5MzIwNTZlLTE2CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAibiI6IDMyLAogICAgICAgICJmb3JtdWxhdGlvbiI6ICJFIiwKICAgICAgICAiYmFzZWxpbmVfb3JkZXIiOiA4LAogICAgICAgICJoaWdoZXJfb3JkZXIiOiAxMiwKICAgICAgICAicmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiAzLjIzODQ5MDQwNzA2ODk4OGUtMTYsCiAgICAgICAgInJlbGF0aXZlX2xvYWRfZGlmZmVyZW5jZSI6IDIuNjkzOTg3NDk3NzY3NzQ0ZS0xNiwKICAgICAgICAicmVsYXRpdmVfbm9kYWxfZGlmZmVyZW5jZSI6IDEuMTMxMTg2Njk0MzAwMjk3NWUtMTUKICAgICAgfQogICAgXQogIH0KfQ=='}, 'results/negative/timings_raw.csv': {'sha256': 'f5912bf124b0bc318aa88c7f6a82a03c756591f8000df859deb7a86293c08ec7', 'base64': 'bixuZG9mLG1ldGhvZCxyZXBldGl0aW9uLHBvc2l0aW9uX2luX3BhaXIsY29lZmZpY2llbnRfcyxhc3NlbWJseV9zLHNvbHZlX3MsdG90YWxfcyxzY2FsZWRfYWxnZWJyYWljX3Jlc2lkdWFsDQo4LDQ5LEIsMSwxLDAuMCwwLjAwMjI1MDkwNTk5OTk3MjAxOTMsMC4wMDAxNjAzMTAwMDI2ODQxMjc1NCwwLjAwMjQxMTIxNjAwMjY1NjE0NywxLjU4NTc0OTc0MjA3NzQ3NGUtMTYNCjgsNDksRSwxLDIsOS43MTE1MDAwMjQ0MjMwMDNlLTA1LDAuMDAyODY1OTc5MDA0MTI3NDM5LDAuMDAwMTM0MDEwOTk1NjgwMDAwNjMsMC4wMDMwOTcxMDUwMDAwNTE2Njk4LDEuNDAyOTY0NzIxMjMxMDczOGUtMTYNCjgsNDksRSwyLDEsMC4wMDAxNzIxNjgwMDE4ODE4MDA2LDAuMDAyMzk5OTU4OTk3NTU3NDA1NCwwLjAwMDEzMTc0ODAwNTUxNDk2NDQ2LDAuMDAyNzAzODc1MDA0OTU0MTcwNCwxLjQwMjk2NDcyMTIzMTA3MzhlLTE2DQo4LDQ5LEIsMiwyLDAuMCwwLjAwMzY1Njg2NDk5NTcxODkzMzYsMC4wMDAzNDU2NzgwMDMyMTM4MzQsMC4wMDQwMDI1NDI5OTg5MzI3NjgsMS41ODU3NDk3NDIwNzc0NzRlLTE2DQo4LDQ5LEIsMywxLDAuMCwwLjAwMjI3OTE0ODAwMDUyNTMxMDYsMC4wMDAxMzM5MTA5OTQ5MTg1NTEzLDAuMDAyNDEzMDU4OTk1NDQzODYyLDEuNTg1NzQ5NzQyMDc3NDc0ZS0xNg0KOCw0OSxFLDMsMiw4LjAyMjAwMDExMDk0MTAwOGUtMDUsMC4wMDI0OTk5MzAwMDAzNDAxMDA0LDAuMDAwMTI2ODkwMDAxNDExMDg0MDksMC4wMDI3MDcwNDAwMDI4NjA1OTQ1LDEuNDAyOTY0NzIxMjMxMDczOGUtMTYNCjgsNDksRSw0LDEsNy44NTM3OTk2MTU5ODY2NDVlLTA1LDAuMDAyMjQ1NjM4MDAyOTMyNTM3MywwLjAwMDEyNzkwMTk5OTgwMzcyNTYzLDAuMDAyNDUyMDc3OTk4ODk2MTI5NCwxLjQwMjk2NDcyMTIzMTA3MzhlLTE2DQo4LDQ5LEIsNCwyLDAuMCwwLjAwMjI1NzY2NzAwMTY4NjA1OSwwLjAwMDExODc3Nzk5OTg4NzI0MjkxLDAuMDAyMzc2NDQ1MDAxNTczMzAyLDEuNTg1NzQ5NzQyMDc3NDc0ZS0xNg0KOCw0OSxCLDUsMSwwLjAsMC4wMDI4MTI4Nzg5OTk5NzU1MSwwLjAwMDIxNTQ0MzAwMzk4MDkwODU0LDAuMDAzMDI4MzIyMDAzOTU2NDE4NSwxLjU4NTc0OTc0MjA3NzQ3NGUtMTYNCjgsNDksRSw1LDIsOC4zOTc2MDAzMjI0OTYxNDZlLTA1LDAuMDAyNDk2NzE1MDAyMDUyOTUxNiwwLjAwMDEyNDI5NTk5ODgzMDM0ODI1LDAuMDAyNzA0OTg3MDA0MTA4MjYxMywxLjQwMjk2NDcyMTIzMTA3MzhlLTE2DQo4LDQ5LEUsNiwxLDUuMzYwMTAwNDc2NTM2OWUtMDUsMC4wMDI1NTQ5MzE5OTgwMjAwMzgsMC4wMDAxMTI3NzkwMDE5MzgxNzkxNCwwLjAwMjcyMTMxMjAwNDcyMzU4NiwxLjQwMjk2NDcyMTIzMTA3MzhlLTE2DQo4LDQ5LEIsNiwyLDAuMCwwLjAwMjE0NjcyOTk5NTcyOTQwMTcsMC4wMDAxMTE4MDY5OTk0ODQ1NDI3NiwwLjAwMjI1ODUzNjk5NTIxMzk0NDUsMS41ODU3NDk3NDIwNzc0NzRlLTE2DQo4LDQ5LEIsNywxLDAuMCwwLjAwMjAyODQzMjAwNTMxNzg4MTcsMC4wMDAxMjA4NzA5OTQ1Nzg1ODEzLDAuMDAyMTQ5MzAyOTk5ODk2NDYzLDEuNTg1NzQ5NzQyMDc3NDc0ZS0xNg0KOCw0OSxFLDcsMiw0Ljg5MTMwMDM0MTExNDQwMmUtMDUsMC4wMDIxNzAxNjQ5OTY2NDE2ODgsMC4wMDAxMDkzMTQwMDE3NDc0MDcwMiwwLjAwMjMyODM5MjAwMTgwMDIzOSwxLjQwMjk2NDcyMTIzMTA3MzhlLTE2DQoxNiwyMjUsRSwxLDEsOC41NDA4OTk1ODAxMTE3MjdlLTA1LDAuMDA5MDEwMjkzMDA0Mzk0MDQzLDAuMDAwMzcyODgwMDAwMjEzNjI2NzcsMC4wMDk0Njg1ODIwMDA0MDg3ODcsMS41NDU0NDUwMzA3MzgxNDFlLTE2DQoxNiwyMjUsQiwxLDIsMC4wLDAuMDA5MjgwMjcwMDAxOTA5MjMsMC4wMDA2NjA2NTA5OTk2MzQ1MjY3LDAuMDA5OTQwOTIxMDAxNTQzNzU3LDEuMjg5NTA0MDM5OTIwOTc1MmUtMTYNCjE2LDIyNSxCLDIsMSwwLjAsMC4wMDgwMDM5NzUwMDI3OTM1OTUsMC4wMDAzNzk1Njg5OTkwNTExMTY0LDAuMDA4MzgzNTQ0MDAxODQ0NzEyLDEuMjg5NTA0MDM5OTIwOTc1MmUtMTYNCjE2LDIyNSxFLDIsMiwwLjAwMDEzMzkzMTAwMzgwMTk5MDMsMC4wMDgwNDgxNTA5OTQwMDAwMjUsMC4wMDAzOTk3MDkwMDI1MjIyODYwMywwLjAwODU4MTc5MTAwMDMyNDMwMSwxLjU0NTQ0NTAzMDczODE0MWUtMTYNCjE2LDIyNSxFLDMsMSw2LjM2MTU5OTY2MjM0MTE3N2UtMDUsMC4wMDc4NDc1MTAwMDE3Mzg1NTYsMC4wMDA0MTc2MzYwMDE2NTcyMjg5LDAuMDA4MzI4NzYyMDAwMDE5MTk2LDEuNTQ1NDQ1MDMwNzM4MTQxZS0xNg0KMTYsMjI1LEIsMywyLDAuMCwwLjAwNzU5MzE0ODAwMjM0Mjc4NywwLjAwMDQ0MTQyMjAwMjA0MDc4MTEsMC4wMDgwMzQ1NzAwMDQzODM1NjgsMS4yODk1MDQwMzk5MjA5NzUyZS0xNg0KMTYsMjI1LEIsNCwxLDAuMCwwLjAwNzk2ODg3MTk5OTc2NDgxNSwwLjAwMDQ3NTkyMjk5OTYxMjUyMTQsMC4wMDg0NDQ3OTQ5OTkzNzczMzYsMS4yODk1MDQwMzk5MjA5NzUyZS0xNg0KMTYsMjI1LEUsNCwyLDYuMjA3Mjk5ODg0NDA4NzEyZS0wNSwwLjAwODEyNzc0OTk5OTQwMzAyMiwwLjAwMDQzMTc5ODAwNTU5MzAwOTMsMC4wMDg2MjE2MjEwMDM4NDAxMTksMS41NDU0NDUwMzA3MzgxNDFlLTE2DQoxNiwyMjUsRSw1LDEsOC4yOTg1MDAzMjQ1OTk0NDVlLTA1LDAuMDA3OTk4MzI2MDAwMjMzNjQ2LDAuMDAwNjQ3MDA5OTk3OTE5NjY3NSwwLjAwODcyODMyMTAwMTM5OTMwOCwxLjU0NTQ0NTAzMDczODE0MWUtMTYNCjE2LDIyNSxCLDUsMiwwLjAsMC4wMDcyMjM3MzM5OTkxMjYxMjUsMC4wMDA1MTQ3MjE5OTk5MzQ0MDg4LDAuMDA3NzM4NDU1OTk5MDYwNTM0LDEuMjg5NTA0MDM5OTIwOTc1MmUtMTYNCjE2LDIyNSxCLDYsMSwwLjAsMC4wMDY3NDA0MjAwMDE3MzM1ODI1LDAuMDAwMzkxODg4MDAyOTIzMjI0MSwwLjAwNzEzMjMwODAwNDY1NjgwNywxLjI4OTUwNDAzOTkyMDk3NTJlLTE2DQoxNiwyMjUsRSw2LDIsOC41MzQ4MDAxNDQ4NDM3NTdlLTA1LDAuMDA5MTIyNzQyOTk0ODA0ODY3LDAuMDAwNDQ2OTQwMDAwOTgzODg2NCwwLjAwOTY1NTAzMDk5NzIzNzE5LDEuNTQ1NDQ1MDMwNzM4MTQxZS0xNg0KMTYsMjI1LEUsNywxLDYuNjU5OTk5NzgwODIyNTQ1ZS0wNSwwLjAwNzYzMzc3OTk5ODI4NjYyNywwLjAwMDM5NTcxMzAwMjk0NDgzMDgsMC4wMDgwOTYwOTI5OTkwMzk2ODMsMS41NDU0NDUwMzA3MzgxNDFlLTE2DQoxNiwyMjUsQiw3LDIsMC4wLDAuMDA2OTg2NjA4MDAxMTcwNjgsMC4wMDAzODgyMzI5OTgzNzUzMTUyLDAuMDA3Mzc0ODQwOTk5NTQ1OTk1LDEuMjg5NTA0MDM5OTIwOTc1MmUtMTYNCjMyLDk2MSxCLDEsMSwwLjAsMC4wMjgyNDMxMDk5OTg1ODQyNiwwLjAwMTg4ODcxMjk5OTA2MTY4MTMsMC4wMzAxMzE4MjI5OTc2NDU5NCwyLjUxMDU3ODA0MzE1ODQ4NjNlLTE2DQozMiw5NjEsRSwxLDIsOS4wNzk1OTk4NDAzMzM2ODVlLTA1LDAuMDI5MzM1NDA5MDAyNjY5NjIsMC4wMDE4NzMzOTkwMDAwMzI4MDUsMC4wMzEyOTk2MDQwMDExMDU3NiwxLjc1MTg2MjU0NTMxMDAxNTZlLTE2DQozMiw5NjEsRSwyLDEsOS44OTQ4MDAzMTQyMTQwOGUtMDUsMC4wMzA4MTAzMjA5OTcyNjE0NDIsMC4wMDIxMDA2MzEwMDUzMDUzMzEsMC4wMzMwMDk5MDAwMDU3MDg5MTQsMS43NTE4NjI1NDUzMTAwMTU2ZS0xNg0KMzIsOTYxLEIsMiwyLDAuMCwwLjAzMzk1NTAyNzAwMDA0NzI2NiwwLjAwMTc4NTQ2ODAwNDUyNzUwMTgsMC4wMzU3NDA0OTUwMDQ1NzQ3NywyLjUxMDU3ODA0MzE1ODQ4NjNlLTE2DQozMiw5NjEsQiwzLDEsMC4wLDAuMDMwODg2Nzg2MDAwMzM3NDUsMC4wMDIyNjc2NTA5OTc5NjQ2NjUzLDAuMDMzMTU0NDM2OTk4MzAyMTIsMi41MTA1NzgwNDMxNTg0ODYzZS0xNg0KMzIsOTYxLEUsMywyLDAuMDAwMTk1NDYyOTk4ODE3Njc0ODIsMC4wMzI0MTUzNzgwMDIwNDI0MTYsMC4wMDE4MjcxNTAwMDExOTA2MDI4LDAuMDM0NDM3OTkxMDAyMDUwNjk0LDEuNzUxODYyNTQ1MzEwMDE1NmUtMTYNCjMyLDk2MSxFLDQsMSw4LjA5NDA5OTc1NzcyNzIzNmUtMDUsMC4wMzE3MzM3NDYwMDAwOTk5MywwLjAwMjgwOTQ0Mzk5ODU1Nzk4MSwwLjAzNDYyNDEzMDk5NjIzNTE4LDEuNzUxODYyNTQ1MzEwMDE1NmUtMTYNCjMyLDk2MSxCLDQsMiwwLjAsMC4wMzI5OTc4MTA5OTQ4NTY5OSwwLjAwMjMxODAyNzAwMDAwMTE2NiwwLjAzNTMxNTgzNzk5NDg1ODE2LDIuNTEwNTc4MDQzMTU4NDg2M2UtMTYNCjMyLDk2MSxCLDUsMSwwLjAsMC4wMzM5NjA2MDUwMDM4MTI4MTYsMC4wMDM3ODIyODI5OTgxNDUwOTIzLDAuMDM3NzQyODg4MDAxOTU3OTEsMi41MTA1NzgwNDMxNTg0ODYzZS0xNg0KMzIsOTYxLEUsNSwyLDAuMDAwMTA5NDQ0MDAxMjgyMDk5NjMsMC4wMzUyNDE1NTcwMDE0MjcyOCwwLjAwMTk4MDQyOTk5NTQ1NTc3OSwwLjAzNzMzMTQzMDk5ODE2NTE2LDEuNzUxODYyNTQ1MzEwMDE1NmUtMTYNCjMyLDk2MSxFLDYsMSw3Ljk1NDg5OTc3NDYzMTQ4NWUtMDUsMC4wMzQ3OTk1MzM5OTgwMTE2MSwwLjAwMjExMzAwOTk5OTQ0Nzk2OCwwLjAzNjk5MjA5Mjk5NTIwNTg5NCwxLjc1MTg2MjU0NTMxMDAxNTZlLTE2DQozMiw5NjEsQiw2LDIsMC4wLDAuMDMyNDg0MzExOTk2MDg2MDM1LDAuMDAxNzg3OTAxMDA0NzE4MTUxLDAuMDM0MjcyMjEzMDAwODA0MTg2LDIuNTEwNTc4MDQzMTU4NDg2M2UtMTYNCjMyLDk2MSxCLDcsMSwwLjAsMC4wMzA1MjQ1NjE5OTY1NzE3MiwwLjAwMjE2OTgzNTAwMjg2MDA1NDQsMC4wMzI2OTQzOTY5OTk0MzE3NzQsMi41MTA1NzgwNDMxNTg0ODYzZS0xNg0KMzIsOTYxLEUsNywyLDAuMDAwMTEzOTgwOTk3NDExOTk5ODUsMC4wNDI0MzQ2MzUwMDQ2MTA3NTUsMC4wMDM4NzEwMjU5OTc3OTYwOTYsMC4wNDY0MTk2NDE5OTk4MTg4NSwxLjc1MTg2MjU0NTMxMDAxNTZlLTE2DQo2NCwzOTY5LEUsMSwxLDAuMDAwMTEyNTU4OTk4ODA3Nzk5MDcsMC4xNTExMzg2OTUwMDA1NTQ0MywwLjAxMzE5OTcyNzk5NjgzNDU1NCwwLjE2NDQ1MDk4MTk5NjE5NjgsNC45NjAyNDg2MDE0NjY1MjVlLTE2DQo2NCwzOTY5LEIsMSwyLDAuMCwwLjEzNzYwNzU3MDk5OTUzNDgsMC4wMTE5OTMwMjgwMDUyMzU4MjgsMC4xNDk2MDA1OTkwMDQ3NzA2MywzLjQ3MTk1MDMxNzgyMzAyZS0xNg0KNjQsMzk2OSxCLDIsMSwwLjAsMC4xNDQxMTIwODIwMDE4MTEzNiwwLjAxMjA3NjYxMTk5NDQxNjQ1MywwLjE1NjE4ODY5Mzk5NjIyNzgsMy40NzE5NTAzMTc4MjMwMmUtMTYNCjY0LDM5NjksRSwyLDIsMC4wMDAxMDE2NjE5OTM1Mzk4OTIxNCwwLjE0MDI2MjkzNDAwMDkyOTgyLDAuMDA5OTk3MTY0MDAxMTA2Mjg1LDAuMTUwMzYxNzU5OTk1NTc2LDQuOTYwMjQ4NjAxNDY2NTI1ZS0xNg0KNjQsMzk2OSxFLDMsMSw5LjU0NzMwMDU3ODU2MDY4ZS0wNSwwLjEzMjY3MTU3Nzk5NDE4NzgzLDAuMDEyODE0ODYwMDA1MTY0NTE5LDAuMTQ1NTgxOTExMDA1MTM3OTUsNC45NjAyNDg2MDE0NjY1MjVlLTE2DQo2NCwzOTY5LEIsMywyLDAuMCwwLjEyODY3NDcyMzAwNDIyMzc4LDAuMDEwNjkwNzg0MDAxNzgxMTQsMC4xMzkzNjU1MDcwMDYwMDQ5MiwzLjQ3MTk1MDMxNzgyMzAyZS0xNg0KNjQsMzk2OSxCLDQsMSwwLjAsMC4xMTM5NzI5NjMwMDEyODgwMSwwLjAxMDk1MzkwODk5NzE4MDQwNiwwLjEyNDkyNjg3MTk5ODQ2ODQxLDMuNDcxOTUwMzE3ODIzMDJlLTE2DQo2NCwzOTY5LEUsNCwyLDcuODc2Nzk5NjQ1NjAwODRlLTA1LDAuMTUxMzAzMTcwOTk4Mjg5NzIsMC4wMTcxMjc5NjUwMDIxNDYwODQsMC4xNjg1MDk5MDM5OTY4OTE4LDQuOTYwMjQ4NjAxNDY2NTI1ZS0xNg0KNjQsMzk2OSxFLDUsMSwwLjAwMDE2MDkxMTAwNDA1OTAxNjcsMC4xNDMyMzQ4MDU5OTcwNTM2MywwLjAxMjI2ODMyOTk5OTY0MjQ0NywwLjE1NTY2NDA0NzAwMDc1NTEsNC45NjAyNDg2MDE0NjY1MjVlLTE2DQo2NCwzOTY5LEIsNSwyLDAuMCwwLjEzNzQyNDQ3NjAwMDg4NzgsMC4wMTE4NTk5NzkwMDAyOTE3ODcsMC4xNDkyODQ0NTUwMDExNzk2LDMuNDcxOTUwMzE3ODIzMDJlLTE2DQo2NCwzOTY5LEIsNiwxLDAuMCwwLjE1NTkxMjM1MDk5NDE4NTU0LDAuMDE2NTUxNTY1MDAwNTU0NTQ3LDAuMTcyNDYzOTE1OTk0NzQwMSwzLjQ3MTk1MDMxNzgyMzAyZS0xNg0KNjQsMzk2OSxFLDYsMiwwLjAwMDE1ODUwNjk5ODU1OTQ2MDA0LDAuMTU1NTI0NDUzOTk5NTMyODUsMC4wMTI5NDE2MTEwMDY2ODEzNDIsMC4xNjg2MjQ1NzIwMDQ3NzM2NSw0Ljk2MDI0ODYwMTQ2NjUyNWUtMTYNCjY0LDM5NjksRSw3LDEsMC4wMDAxMTA3NjcwMDMyMDcwMDE4NCwwLjE0Njc4NDc0NDk5NjY5MzkyLDAuMDExMTIxNDcwMDAwOTQ0MSwwLjE1ODAxNjk4MjAwMDg0NTAyLDQuOTYwMjQ4NjAxNDY2NTI1ZS0xNg0KNjQsMzk2OSxCLDcsMiwwLjAsMC4xNDAzMTMzMzUwMDMxNTY2OCwwLjAxMDc1MDIzMjk5OTI3MTI0OCwwLjE1MTA2MzU2ODAwMjQyNzkzLDMuNDcxOTUwMzE3ODIzMDJlLTE2DQoxMjgsMTYxMjksQiwxLDEsMC4wLDAuNTA2ODQwMzAzOTk4NjE1MSwwLjA2NTQ3ODMwODAwMjYzOTEsMC41NzIzMTg2MTIwMDEyNTQyLDQuNzk4MDI2ODk0NDc1ODQyZS0xNg0KMTI4LDE2MTI5LEUsMSwyLDAuMDAwMTE2MTMzOTk2OTI1NzgyNDEsMC41MjU4NzYxNDQwMDU0NTE0LDAuMDU2MjUyODk4MDAwODIxMDA2LDAuNTgyMjQ1MTc2MDAzMTk4MSw0LjM2NjcyNjEwNjExNzU1ODRlLTE2DQoxMjgsMTYxMjksRSwyLDEsOC40NjY3MDAwOTE5NTgwNDZlLTA1LDAuNDkxNzcxNTE1MDAwNDM2OSwwLjA2MzAxMjU0NTAwNDMxMjQxLDAuNTU0ODY4NzI3MDA1NjY4OSw0LjM2NjcyNjEwNjExNzU1ODRlLTE2DQoxMjgsMTYxMjksQiwyLDIsMC4wLDAuNDY4MDc5OTkyOTk5MjY0NSwwLjA3MjA5NTc1Nzk5Nzg4NTgxLDAuNTQwMTc1NzUwOTk3MTUwMyw0Ljc5ODAyNjg5NDQ3NTg0MmUtMTYNCjEyOCwxNjEyOSxCLDMsMSwwLjAsMC40NjQ4Nzc4OTk5OTgxMDA1LDAuMDcwNzU5NTE1MDAxNTU0OTcsMC41MzU2Mzc0MTQ5OTk2NTU0LDQuNzk4MDI2ODk0NDc1ODQyZS0xNg0KMTI4LDE2MTI5LEUsMywyLDcuNjk5NTAwNTY1NjQ5OTQyZS0wNSwwLjUwMDU2MTI2Mjk5NzUwNzYsMC4wNjA4MzE1MTUwMDIwNjQ0MSwwLjU2MTQ2OTc3MzAwNTIyODUsNC4zNjY3MjYxMDYxMTc1NTg0ZS0xNg0KMTI4LDE2MTI5LEUsNCwxLDguODM5MjAwMDE3OTczNzhlLTA1LDAuNDkzNTg2OTcyOTk2MTI2ODYsMC4wNTkzNDU5NTIwMDIzMzE2MTQsMC41NTMwMjEzMTY5OTg2MzgyLDQuMzY2NzI2MTA2MTE3NTU4NGUtMTYNCjEyOCwxNjEyOSxCLDQsMiwwLjAsMC40NDE4NzY5MTM5OTU2NTY2NSwwLjA2MTA5NDkxOTk5ODE0MDU0LDAuNTAyOTcxODMzOTkzNzk3Miw0Ljc5ODAyNjg5NDQ3NTg0MmUtMTYNCjEyOCwxNjEyOSxCLDUsMSwwLjAsMC40NTg1MjI4NzI5OTkzMTA1LDAuMDU5NDQ2MzM4MDAzOTA4NzUsMC41MTc5NjkyMTEwMDMyMTkyLDQuNzk4MDI2ODk0NDc1ODQyZS0xNg0KMTI4LDE2MTI5LEUsNSwyLDkuNDE2MTk5NDYzODg1Mjc4ZS0wNSwwLjUwNzU0NDczNzAwMzYyNzIsMC4wNjE5ODU1NjUwMDAyMTM2OCwwLjU2OTYyNDQ2Mzk5ODQ3OTgsNC4zNjY3MjYxMDYxMTc1NTg0ZS0xNg0KMTI4LDE2MTI5LEUsNiwxLDkuNzExNjAwNDMyNjM4MDc5ZS0wNSwwLjQ5MjA3ODYzNTk5NDQwMzE2LDAuMDY3NjUxMjYyMDA0ODAxNTQsMC41NTk4MjcwMTQwMDM1MzExLDQuMzY2NzI2MTA2MTE3NTU4NGUtMTYNCjEyOCwxNjEyOSxCLDYsMiwwLjAsMC40NTk5MzY5NDk5OTkyNDYzLDAuMDYzMjgzMTQxMDAzOTIxNjMsMC41MjMyMjAwOTEwMDMxNjc5LDQuNzk4MDI2ODk0NDc1ODQyZS0xNg0KMTI4LDE2MTI5LEIsNywxLDAuMCwwLjQ2NDUyNDkyMTk5NzUwODUsMC4wNjU0NjQyMjMwMDAxNjI3NSwwLjUyOTk4OTE0NDk5NzY3MTIsNC43OTgwMjY4OTQ0NzU4NDJlLTE2DQoxMjgsMTYxMjksRSw3LDIsOC4wMTkwMDAyMzM2MTY2OGUtMDUsMC41MDY4NTA4NzEwMDI4MjU0LDAuMDY4MDUyNDMzOTk4NjczMjIsMC41NzQ5ODM0OTUwMDM4MzQ4LDQuMzY2NzI2MTA2MTE3NTU4NGUtMTYNCjI1Niw2NTAyNSxFLDEsMSw5LjIxMDc5OTkwODAzMjY0MWUtMDUsMi4xMjI0NjQxMTE5OTg4NTIsMC40NTE3Mjc3MTEwMDYyNDIzLDIuNTc0MjgzOTMxMDA0MTc0NSw1LjcyMDAxNzY3NTY0ODY3NGUtMTYNCjI1Niw2NTAyNSxCLDEsMiwwLjAsMS44OTkzODM1Njk5OTQ4MjUsMC40MTQ4MzY3ODMwMDU3Mzk1NiwyLjMxNDIyMDM1MzAwMDU2NDUsNS42ODYzMTU0MTUwNzc3MDFlLTE2DQoyNTYsNjUwMjUsQiwyLDEsMC4wLDEuNzQzMTM4OTEyOTk5NzExNSwwLjQxNTI5MTEwNTAwNDE0MzksMi4xNTg0MzAwMTgwMDM4NTU0LDUuNjg2MzE1NDE1MDc3NzAxZS0xNg0KMjU2LDY1MDI1LEUsMiwyLDAuMDAwMTU5MTA5MDA0MDE2NDk5OTcsMS44NzM2NDMxMjA5OTgxNjc0LDAuMzk3ODM0MDgyOTk5OTA2OTcsMi4yNzE2MzYzMTMwMDIwOTEsNS43MjAwMTc2NzU2NDg2NzRlLTE2DQoyNTYsNjUwMjUsRSwzLDEsOC43NTkxOTk0MDg4MTQzMTFlLTA1LDEuODUyNTMwNDgwMDA1OTUzLDAuNDEzMDQzNzc3OTk2NzkwNiwyLjI2NTY2MTg0OTk5NjgzMTcsNS43MjAwMTc2NzU2NDg2NzRlLTE2DQoyNTYsNjUwMjUsQiwzLDIsMC4wLDEuNzA3MjQ2NTUxMDA2MzkyLDAuNDIyMzY4NTAzOTk3ODU5MSwyLjEyOTYxNTA1NTAwNDI1MSw1LjY4NjMxNTQxNTA3NzcwMWUtMTYNCjI1Niw2NTAyNSxCLDQsMSwwLjAsMS44NTYzMjg5MDk5OTg0MTA0LDAuNDY3ODAxMTI1OTk2NjI5NywyLjMyNDEzMDAzNTk5NTA0LDUuNjg2MzE1NDE1MDc3NzAxZS0xNg0KMjU2LDY1MDI1LEUsNCwyLDAuMDAwMTAxNjYyMDAwODE1ODQ5NzUsMi4wNTM3MDI4NTgwMDAxNzksMC40NDY3Nzk4MjQwMDAyNTk0LDIuNTAwNTg0MzQ0MDAxMjU0LDUuNzIwMDE3Njc1NjQ4Njc0ZS0xNg0KMjU2LDY1MDI1LEUsNSwxLDAuMDAwMTA1NzM3OTk5NTQ3MjcyOTIsMS45ODc4MDcxNzY5OTk5MjI0LDAuNDg0MTQ1NTk1OTk3Njg4MzYsMi40NzIwNTg1MTA5OTcxNTgsNS43MjAwMTc2NzU2NDg2NzRlLTE2DQoyNTYsNjUwMjUsQiw1LDIsMC4wLDIuMDYwNDA4MzU4MDAzNzY5NSwwLjUyODYzMzMxNzAwMDY0MDEsMi41ODkwNDE2NzUwMDQ0MDk2LDUuNjg2MzE1NDE1MDc3NzAxZS0xNg0KMjU2LDY1MDI1LEIsNiwxLDAuMCwxLjg4MDIzMDEyODAwMTEzMDcsMC40NDY1NDMxMTU5OTg3OTI5LDIuMzI2NzczMjQzOTk5OTIzNiw1LjY4NjMxNTQxNTA3NzcwMWUtMTYNCjI1Niw2NTAyNSxFLDYsMiwwLjAwMDEyNTg2OTAwMjY1ODg3MzgsMi4wMjQ1NzM5NDg5OTgyOTUyLDAuNDQyNDY3NTAzMDAyMjYyNywyLjQ2NzE2NzMyMTAwMzIxNyw1LjcyMDAxNzY3NTY0ODY3NGUtMTYNCjI1Niw2NTAyNSxFLDcsMSwwLjAwMDEyMzI0Mzk5NzIyMjc0MzkzLDIuMDQ5MDg2MjAyOTk4Mjk2NCwwLjQ3MzU5OTk5MTAwNDQ0ODQsMi41MjI4MDk0Mzc5OTk5Njc1LDUuNzIwMDE3Njc1NjQ4Njc0ZS0xNg0KMjU2LDY1MDI1LEIsNywyLDAuMCwxLjg4NDQyMzgxMjAwMDAyMjcsMC41MTg3NzAwMzA5OTc5ODQ2LDIuNDAzMTkzODQyOTk4MDA3Myw1LjY4NjMxNTQxNTA3NzcwMWUtMTYNCjUxMiwyNjExMjEsQiwxLDEsMC4wLDcuMjA3ODYxMjkwMDAwNDM3LDQuMzczNTk0ODMyOTk5NTE1LDExLjU4MTQ1NjEyMjk5OTk1Myw5LjExNzQ3MjEzNjc5MTgxNmUtMTYNCjUxMiwyNjExMjEsRSwxLDIsOS45OTEwMDAxMTU0MDU3NjhlLTA1LDguMDkxOTc2MDYwMDAwODg5LDQuMjk1ODUwOTUyOTk5ODk5LDEyLjM4NzkyNjkyMzAwMTk0MiwxLjA3ODk5NTQ4NDQxMjgxNzllLTE1DQo1MTIsMjYxMTIxLEUsMiwxLDkuNDI4MjAwNDI4Mzc0MTEyZS0wNSw4LjAwNjAwOTEwNjAwMDQyLDQuNDQ5NjcwNTEwOTk5NTY3LDEyLjQ1NTc3Mzg5OTAwNDI3LDEuMDc4OTk1NDg0NDEyODE3OWUtMTUNCjUxMiwyNjExMjEsQiwyLDIsMC4wLDcuMTAzNDcyODAyOTk5MDIyLDQuMzAzOTIwOTgwMDA0NDc3LDExLjQwNzM5Mzc4MzAwMzUsOS4xMTc0NzIxMzY3OTE4MTZlLTE2DQo1MTIsMjYxMTIxLEIsMywxLDAuMCw3LjIzNjA5NjQ5OTAwMDI2ODUsNC4yNzYxOTU5MzMwMDE4NzMsMTEuNTEyMjkyNDMyMDAyMTQxLDkuMTE3NDcyMTM2NzkxODE2ZS0xNg0KNTEyLDI2MTEyMSxFLDMsMiwwLjAwMDEwNjQ0ODk5ODg0OTM3MzMxLDguNDA4NDM4NzQzMDA0ODA4LDMuOTUwNjk3OTYzOTk5MTE4NywxMi4zNTkyNDMxNTYwMDI3NzYsMS4wNzg5OTU0ODQ0MTI4MTc5ZS0xNQ0KNTEyLDI2MTEyMSxFLDQsMSwwLjAwMDEwMDcyMTAwMTIxNzYwNzQxLDcuNzg2OTU3NjI1OTk4MTg0LDMuODIxOTAxOTIxMDAwOTcyLDExLjYwODk2MDI2ODAwMDM3NCwxLjA3ODk5NTQ4NDQxMjgxNzllLTE1DQo1MTIsMjYxMTIxLEIsNCwyLDAuMCw3LjUwNzA0Mzk1NjAwMDQxNiw0LjMxNjA1MzM0OTk5NzY3NCwxMS44MjMwOTczMDU5OTgwOSw5LjExNzQ3MjEzNjc5MTgxNmUtMTYNCjUxMiwyNjExMjEsQiw1LDEsMC4wLDYuODIxMDAyMjcwOTk3MjY5LDMuOTQyMDMxMzA0MDAyNzM5NSwxMC43NjMwMzM1NzUwMDAwMDksOS4xMTc0NzIxMzY3OTE4MTZlLTE2DQo1MTIsMjYxMTIxLEUsNSwyLDkuNzMzNjAwMDE4MDgwMzI0ZS0wNSw3LjYzMjA0NTk3OTAwMDM5NSw0LjExNzg0OTY3NzAwMDc3NywxMS43NDk5OTI5OTIwMDEzNTMsMS4wNzg5OTU0ODQ0MTI4MTc5ZS0xNQ0KNTEyLDI2MTEyMSxFLDYsMSwwLjAwMDEwMjExMzAwMTA0ODU2NDkxLDcuNzM2NDkzNTQ3OTk5OTk4Niw0LjQxNDgzMTM1NTk5OTkyMiwxMi4xNTE0MjcwMTcwMDA5NywxLjA3ODk5NTQ4NDQxMjgxNzllLTE1DQo1MTIsMjYxMTIxLEIsNiwyLDAuMCw3LjE1NzQ3MzgxNzk5ODgzNCwzLjk0NTQ3MzkwNjAwMjAwNzQsMTEuMTAyOTQ3NzI0MDAwODQyLDkuMTE3NDcyMTM2NzkxODE2ZS0xNg0KNTEyLDI2MTEyMSxCLDcsMSwwLjAsNy4xOTM5MjQ2NTc5OTMxOTMsNC40MDUxNzM1NzMwMDI2ODUsMTEuNTk5MDk4MjMwOTk1ODc4LDkuMTE3NDcyMTM2NzkxODE2ZS0xNg0KNTEyLDI2MTEyMSxFLDcsMiwwLjAwMDE3NTk4NDAwMTU0MzgzODUzLDcuODA2NDg2NDI0MDAwNDY4LDQuMTEyNTkyNzM1OTk3MTY4LDExLjkxOTI1NTE0Mzk5OTE4LDEuMDc4OTk1NDg0NDEyODE3OWUtMTUNCg=='}, 'results/noncoercive/convergence.csv': {'sha256': '391f6c95ccaed215b55232a9426ed4726bac07573e3b5218906d17b089a1807d', 'base64': '...'}, 'results/noncoercive/metadata.json': {'sha256': '633ae4b0a4a0275112db6fae31fcf369cfe94e7ddd1753795d321cceb21ead65', 'base64': 'ewogICJzdGFydGVkX3V0YyI6ICIyMDI2LTEwLTE2VDE0OjMwOjM1LjM4NDkzNyswMDowMCIsCiAgImNvbXBsZXRlZF91dGMiOiAiMjAyNi0xMC0xNlQxNDo1MjowNi41NDk1MjQrMDA6MDAiLAogICJzb3VyY2Vfc2hhMjU2IjogIjk1MWY2NWUyODJhZDFiYTNlMGE2N2MxNGQyZDViMzM2MDUyYmE5MDg3NjI5MzdlYWVhZTJmMTA2NGRhMmYzODEiLAogICJkZXNjcmlwdGlvbiI6ICJOb25jb2VyY2l2ZSBPVSBmYW1pbHk6IG9yaWdpbmFsIEIgdmVyc3VzIGVxdWl2YWxlbnQgd2VpZ2h0ZWQgRSwgUDEgRkVNLCBzYW1lIG1lc2ggYW5kIHNwYXJzZSBMVSIsCiAgImV4ZWN1dGlvbl9lbnZpcm9ubWVudCI6ICJhY3R1YWwgbWFjaGluZSBydW5uaW5nIHRoaXMgc2NyaXB0OyBub3QgbmVjZXNzYXJpbHkgR29vZ2xlIENvbGFiIiwKICAicHl0aG9uIjogIjMuMTIuMTQgKG1haW4sIEF1ZyAyNSAyMDI2LCAxNDowMDo0OSkgW0NsYW5nIDIyLjEuMyBdIiwKICAibnVtcHkiOiAiMi4zLjUiLAogICJzY2lweSI6ICIxLjE3LjAiLAogICJwbGF0Zm9ybSI6ICJMaW51eC02LjE4LjQ0LXg4Nl82NC13aXRoLWdsaWJjMi4zOSIsCiAgImNwdSI6ICJBTUQgRVBZQyA5Vjc0IDgwLUNvcmUgUHJvY2Vzc29yIiwKICAibG9naWNhbF9jcHVfY291bnQiOiA5LAogICJudW1lcmljYWxfdGhyZWFkcyI6IDEsCiAgInRocmVhZHBvb2xfaW5mb19kdXJpbmdfcnVuIjogWwogICAgewogICAgICAidXNlcl9hcGkiOiAiYmxhcyIsCiAgICAgICJpbnRlcm5hbF9hcGkiOiAib3BlbmJsYXMiLAogICAgICAibnVtX3RocmVhZHMiOiAxLAogICAgICAicHJlZml4IjogImxpYnNjaXB5X29wZW5ibGFzIiwKICAgICAgImZpbGVwYXRoIjogIi9vcHQvY29kZXgvcnVudGltZXMvY29kZXgtcHJpbWFyeS1ydW50aW1lL2RlcGVuZGVuY2llcy9weXRob24vbGliL3B5dGhvbjMuMTIvc2l0ZS1wYWNrYWdlcy9udW1weS5saWJzL2xpYnNjaXB5X29wZW5ibGFzNjRfLWZkZGU1Nzc4LnNvIiwKICAgICAgInZlcnNpb24iOiAiMC4zLjMwIiwKICAgICAgInRocmVhZGluZ19sYXllciI6ICJwdGhyZWFkcyIsCiAgICAgICJhcmNoaXRlY3R1cmUiOiAiU2t5bGFrZVgiCiAgICB9LAogICAgewogICAgICAidXNlcl9hcGkiOiAiYmxhcyIsCiAgICAgICJpbnRlcm5hbF9hcGkiOiAib3BlbmJsYXMiLAogICAgICAibnVtX3RocmVhZHMiOiAxLAogICAgICAicHJlZml4IjogImxpYnNjaXB5X29wZW5ibGFzIiwKICAgICAgImZpbGVwYXRoIjogIi9vcHQvY29kZXgvcnVudGltZXMvY29kZXgtcHJpbWFyeS1ydW50aW1lL2RlcGVuZGVuY2llcy9weXRob24vbGliL3B5dGhvbjMuMTIvc2l0ZS1wYWNrYWdlcy9zY2lweS5saWJzL2xpYnNjaXB5X29wZW5ibGFzLTZjZGMzYjRhLnNvIiwKICAgICAgInZlcnNpb24iOiAiMC4zLjMwIiwKICAgICAgInRocmVhZGluZ19sYXllciI6ICJwdGhyZWFkcyIsCiAgICAgICJhcmNoaXRlY3R1cmUiOiAiU2t5bGFrZVgiCiAgICB9CiAgXSwKICAiQSI6IFsKICAgIFsKICAgICAgMS4wLAogICAgICAwLjAKICAgIF0sCiAgICBbCiAgICAgIDAuMCwKICAgICAgMi4wCiAgICBdCiAgXSwKICAiR19iYXNlIjogWwogICAgWwogICAgICAtMS4wLAogICAgICAxLjAKICAgIF0sCiAgICBbCiAgICAgIC0xLjAsCiAgICAgIC0yLjAKICAgIF0KICBdLAogICJjb2VmZmljaWVudHMiOiB7CiAgICAiNCI6IHsKICAgICAgIlMiOiBbCiAgICAgICAgWwogICAgICAgICAgLTQuMCwKICAgICAgICAgIC0wLjAKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC0wLjAsCiAgICAgICAgICAtNC4wCiAgICAgICAgXQogICAgICBdLAogICAgICAiSCI6IFsKICAgICAgICBbCiAgICAgICAgICAwLjAsCiAgICAgICAgICA0LjAKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC00LjAsCiAgICAgICAgICAwLjAKICAgICAgICBdCiAgICAgIF0sCiAgICAgICJDIjogWwogICAgICAgIFsKICAgICAgICAgIDAuMCwKICAgICAgICAgIDAuNQogICAgICAgIF0sCiAgICAgICAgWwogICAgICAgICAgLTAuNSwKICAgICAgICAgIDAuMAogICAgICAgIF0KICAgICAgXSwKICAgICAgIk0iOiBbCiAgICAgICAgWwogICAgICAgICAgMC41LAogICAgICAgICAgMC41CiAgICAgICAgXSwKICAgICAgICBbCiAgICAgICAgICAtMC41LAogICAgICAgICAgMS4wCiAgICAgICAgXQogICAgICBdCiAgICB9LAogICAgIjgiOiB7CiAgICAgICJTIjogWwogICAgICAgIFsKICAgICAgICAgIC04LjAsCiAgICAgICAgICAtMC4wCiAgICAgICAgXSwKICAgICAgICBbCiAgICAgICAgICAtMC4wLAogICAgICAgICAgLTguMAogICAgICAgIF0KICAgICAgXSwKICAgICAgIkgiOiBbCiAgICAgICAgWwogICAgICAgICAgMC4wLAogICAgICAgICAgOC4wCiAgICAgICAgXSwKICAgICAgICBbCiAgICAgICAgICAtOC4wLAogICAgICAgICAgMC4wCiAgICAgICAgXQogICAgICBdLAogICAgICAiQyI6IFsKICAgICAgICBbCiAgICAgICAgICAwLjAsCiAgICAgICAgICAwLjUKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC0wLjUsCiAgICAgICAgICAwLjAKICAgICAgICBdCiAgICAgIF0sCiAgICAgICJNIjogWwogICAgICAgIFsKICAgICAgICAgIDAuNSwKICAgICAgICAgIDAuNQogICAgICAgIF0sCiAgICAgICAgWwogICAgICAgICAgLTAuNSwKICAgICAgICAgIDEuMAogICAgICAgIF0KICAgICAgXQogICAgfSwKICAgICIxMiI6IHsKICAgICAgIlMiOiBbCiAgICAgICAgWwogICAgICAgICAgLTEyLjAsCiAgICAgICAgICAtMC4wCiAgICAgICAgXSwKICAgICAgICBbCiAgICAgICAgICAtMC4wLAogICAgICAgICAgLTEyLjAKICAgICAgICBdCiAgICAgIF0sCiAgICAgICJIIjogWwogICAgICAgIFsKICAgICAgICAgIDAuMCwKICAgICAgICAgIDEyLjAKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC0xMi4wLAogICAgICAgICAgMC4wCiAgICAgICAgXQogICAgICBdLAogICAgICAiQyI6IFsKICAgICAgICBbCiAgICAgICAgICAwLjAsCiAgICAgICAgICAwLjUKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC0wLjUsCiAgICAgICAgICAwLjAKICAgICAgICBdCiAgICAgIF0sCiAgICAgICJNIjogWwogICAgICAgIFsKICAgICAgICAgIDAuNSwKICAgICAgICAgIDAuNQogICAgICAgIF0sCiAgICAgICAgWwogICAgICAgICAgLTAuNSwKICAgICAgICAgIDEuMAogICAgICAgIF0KICAgICAgXQogICAgfSwKICAgICIxNiI6IHsKICAgICAgIlMiOiBbCiAgICAgICAgWwogICAgICAgICAgLTE2LjAsCiAgICAgICAgICAtMC4wCiAgICAgICAgXSwKICAgICAgICBbCiAgICAgICAgICAtMC4wLAogICAgICAgICAgLTE2LjAKICAgICAgICBdCiAgICAgIF0sCiAgICAgICJIIjogWwogICAgICAgIFsKICAgICAgICAgIDAuMCwKICAgICAgICAgIDE2LjAKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC0xNi4wLAogICAgICAgICAgMC4wCiAgICAgICAgXQogICAgICBdLAogICAgICAiQyI6IFsKICAgICAgICBbCiAgICAgICAgICAwLjAsCiAgICAgICAgICAwLjUKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC0wLjUsCiAgICAgICAgICAwLjAKICAgICAgICBdCiAgICAgIF0sCiAgICAgICJNIjogWwogICAgICAgIFsKICAgICAgICAgIDAuNSwKICAgICAgICAgIDAuNQogICAgICAgIF0sCiAgICAgICAgWwogICAgICAgICAgLTAuNSwKICAgICAgICAgIDEuMAogICAgICAgIF0KICAgICAgXQogICAgfSwKICAgICIyMCI6IHsKICAgICAgIlMiOiBbCiAgICAgICAgWwogICAgICAgICAgLTIwLjAsCiAgICAgICAgICAtMC4wCiAgICAgICAgXSwKICAgICAgICBbCiAgICAgICAgICAtMC4wLAogICAgICAgICAgLTIwLjAKICAgICAgICBdCiAgICAgIF0sCiAgICAgICJIIjogWwogICAgICAgIFsKICAgICAgICAgIDAuMCwKICAgICAgICAgIDIwLjAKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC0yMC4wLAogICAgICAgICAgMC4wCiAgICAgICAgXQogICAgICBdLAogICAgICAiQyI6IFsKICAgICAgICBbCiAgICAgICAgICAwLjAsCiAgICAgICAgICAwLjUKICAgICAgICBdLAogICAgICAgIFsKICAgICAgICAgIC0wLjUsCiAgICAgICAgICAwLjAKICAgICAgICBdCiAgICAgIF0sCiAgICAgICJNIjogWwogICAgICAgIFsKICAgICAgICAgIDAuNSwKICAgICAgICAgIDAuNQogICAgICAgIF0sCiAgICAgICAgWwogICAgICAgICAgLTAuNSwKICAgICAgICAgIDEuMAogICAgICAgIF0KICAgICAgXQogICAgfQogIH0sCiAgImRlbnNpdHkiOiAiZXhwKC1rKih4XjIreV4yKSkiLAogICJkZW5zaXR5X2NvbnRyYXN0IjogewogICAgIjQiOiAyOTgwLjk1Nzk4NzA0MTcyODMsCiAgICAiOCI6IDg4ODYxMTAuNTIwNTA3ODcyLAogICAgIjEyIjogMjY0ODkxMjIxMjkuODQzNDcsCiAgICAiMTYiOiA3ODk2Mjk2MDE4MjY4MC42OSwKICAgICIyMCI6IDIuMzUzODUyNjY4MzcwMTk5N2UrMTcKICB9LAogICJleGFjdF9zb2x1dGlvbiI6ICIoMS14XjIpKDEteV4yKSIsCiAgImZvcmNpbmciOiAiMy0oMisyKmspKnheMi0oMSs0KmspKnleMis2KmsqeF4yKnleMisyKmsqeF4zKnktMiprKngqeV4zIiwKICAiZG9tYWluIjogIigtMSwxKV4yIiwKICAibWVzaCI6ICJ1bmlmb3JtIG4tYnktbiBzcXVhcmVzOyBzb3V0aHdlc3QtdG8tbm9ydGhlYXN0IGRpYWdvbmFscyIsCiAgImtfdmFsdWVzIjogWwogICAgNCwKICAgIDgsCiAgICAxMiwKICAgIDE2LAogICAgMjAKICBdLAogICJzaXplcyI6IFsKICAgIDgsCiAgICAxNiwKICAgIDMyLAogICAgNjQsCiAgICAxMjgsCiAgICAyNTYsCiAgICA1MTIKICBdLAogICJyZXBldGl0aW9ucyI6IDcsCiAgIndhcm11cHNfcGVyX2tfbWVzaF9tZXRob2QiOiAxLAogICJhc3NlbWJseV9kdWZmeV9vcmRlciI6IDgsCiAgImVycm9yX2R1ZmZ5X29yZGVyIjogOCwKICAiYmF0Y2hfc2l6ZSI6IDQwOTYsCiAgInNvbHZlciI6ICJzY2lweS5zcGFyc2UubGluYWxnLnNwbHU7IFN1cGVyTFU7IHBlcm1jX3NwZWM9Q09MQU1EOyBkaWFnX3Bpdm90X3RocmVzaD0xLjAiLAogICJ0aW1pbmdfdW5pdCI6ICJzZWNvbmRzIiwKICAidGltaW5nX3N0YXRpc3RpYyI6ICJtZWRpYW4gb2YgdG90YWwgdGltZSBvdmVyIHJlcGV0aXRpb25zIiwKICAidGltaW5nX2luY2x1ZGVzIjogWwogICAgIkU6IGNvbXB1dGUgUyxILEMsTTsgQjogY29lZmZpY2llbnQgcHJlcGFyYXRpb249MCIsCiAgICAiZ2VvbWV0cnkgYW5kIG1hdHJpeC9sb2FkIGFzc2VtYmx5LCBpbmNsdWRpbmcgZGVuc2l0eSBldmFsdWF0aW9uIiwKICAgICJzcGFyc2UgbWF0cml4IGNvbnN0cnVjdGlvbiBhbmQgaG9tb2dlbmVvdXMgRGlyaWNobGV0IGV4dHJhY3Rpb24iLAogICAgImZyZXNoIHNwYXJzZSBMVSBmYWN0b3JpemF0aW9uIGFuZCB0cmlhbmd1bGFyIHNvbHZlcyIKICBdLAogICJ0aW1pbmdfZXhjbHVkZXMiOiBbCiAgICAiY29tbW9uIG1lc2gvY29ubmVjdGl2aXR5IGNyZWF0aW9uIiwKICAgICJxdWFkcmF0dXJlLXJ1bGUgY29uc3RydWN0aW9uIiwKICAgICJ3YXJtdXAiLAogICAgImdhcmJhZ2UgY29sbGVjdGlvbiIsCiAgICAiZ2xvYmFsIG5vZGFsIG91dHB1dCBleHBhbnNpb24iLAogICAgImVycm9yL3Jlc2lkdWFsIGV2YWx1YXRpb24iLAogICAgIm91dHB1dCBhbmQgcGxvdHRpbmciCiAgXSwKICAiY29tcG9uZW50X21lZGlhbnNfbm90ZSI6ICJFYWNoIGNvbXBvbmVudCBpcyBtZWRpYW4gc2VwYXJhdGVseTsgdGhlaXIgc3VtIG5lZWQgbm90IGVxdWFsIG1lZGlhbiB0b3RhbC4iLAogICJIMV9lcnJvcl9kZWZpbml0aW9uIjogInNxcnQoTDJfZXJyb3JeMiArIGdyYWRpZW50X0wyX2Vycm9yXjIpLCB1bndlaWdodGVkIiwKICAicmVsYXRpdmVfZXJyb3JfZGVmaW5pdGlvbiI6ICJsMi8oMTYvMTUpLCBoMS8oMTYqc3FydCg2KS8xNSk7IGRpbWVuc2lvbmxlc3MgcmF0aW9zIiwKICAiY29lZmZpY2llbnRfY29uc3RydWN0aW9uIjogIkV4cGFuZGVkIHBsYW5hciBTIGZvcm11bGE7IEM9W1swLC1ldGEvKDIqdGF1KV0sW2V0YS8oMip0YXUpLDBdXSwgbm8gUyBpbnZlcnNlIiwKICAidmFsaWRhdGlvbiI6IHsKICAgICJwYXJhbWV0ZXJfY2hlY2tzIjogWwogICAgICB7CiAgICAgICAgImsiOiA0LAogICAgICAgICJjb2VmZmljaWVudF9pZGVudGl0eV9kZWZlY3RzIjogewogICAgICAgICAgIlNfc3ltbWV0cnkiOiAwLjAsCiAgICAgICAgICAicmljY2F0aSI6IDAuMCwKICAgICAgICAgICJ0cmFjZSI6IDAuMCwKICAgICAgICAgICJDX2FudGlzeW1tZXRyeSI6IDAuMCwKICAgICAgICAgICJwb3RlbnRpYWwiOiAwLjAsCiAgICAgICAgICAiZmFjdG9yaXphdGlvbiI6IDAuMCwKICAgICAgICAgICJTX2V4YWN0IjogMC4wLAogICAgICAgICAgIkNfZXhhY3QiOiAwLjAKICAgICAgICB9LAogICAgICAgICJtYW51ZmFjdHVyZWRfZm9yY2luZ19tYXhfYWJzb2x1dGVfZGVmZWN0IjogNy4xMDU0MjczNTc2MDEwMDJlLTE1LAogICAgICAgICJ3ZWlnaHRlZF9kcmlmdF9wb3RlbnRpYWxfcmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiA0LjA3NzM3Mzg5NDE4MjUwM2UtMTYsCiAgICAgICAgIm9yaWdpbmFsX2VuZXJneV9yZWxhdGl2ZV9kZWZlY3QiOiAxLjMzNDU1MzgxOTQ2ODA0M2UtMTYsCiAgICAgICAgIndlaWdodGVkX2VuZXJneV9yZWxhdGl2ZV9kZWZlY3QiOiAwLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJrIjogOCwKICAgICAgICAiY29lZmZpY2llbnRfaWRlbnRpdHlfZGVmZWN0cyI6IHsKICAgICAgICAgICJTX3N5bW1ldHJ5IjogMC4wLAogICAgICAgICAgInJpY2NhdGkiOiAwLjAsCiAgICAgICAgICAidHJhY2UiOiAwLjAsCiAgICAgICAgICAiQ19hbnRpc3ltbWV0cnkiOiAwLjAsCiAgICAgICAgICAicG90ZW50aWFsIjogMC4wLAogICAgICAgICAgImZhY3Rvcml6YXRpb24iOiAwLjAsCiAgICAgICAgICAiU19leGFjdCI6IDAuMCwKICAgICAgICAgICJDX2V4YWN0IjogMC4wCiAgICAgICAgfSwKICAgICAgICAibWFudWZhY3R1cmVkX2ZvcmNpbmdfbWF4X2Fic29sdXRlX2RlZmVjdCI6IDEuMjg3ODU4NzA4NTY1MTgxNmUtMTQsCiAgICAgICAgIndlaWdodGVkX2RyaWZ0X3BvdGVudGlhbF9yZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDUuNTgyMzcyNDc2NDMwMjE5ZS0xNiwKICAgICAgICAib3JpZ2luYWxfZW5lcmd5X3JlbGF0aXZlX2RlZmVjdCI6IDEuNTcxMDY0NTY0NDU2MTc5ZS0xNiwKICAgICAgICAid2VpZ2h0ZWRfZW5lcmd5X3JlbGF0aXZlX2RlZmVjdCI6IDIuMDI2MDU1Mjc5NjQ4OTEwM2UtMTYKICAgICAgfSwKICAgICAgewogICAgICAgICJrIjogMTIsCiAgICAgICAgImNvZWZmaWNpZW50X2lkZW50aXR5X2RlZmVjdHMiOiB7CiAgICAgICAgICAiU19zeW1tZXRyeSI6IDAuMCwKICAgICAgICAgICJyaWNjYXRpIjogMC4wLAogICAgICAgICAgInRyYWNlIjogMC4wLAogICAgICAgICAgIkNfYW50aXN5bW1ldHJ5IjogMC4wLAogICAgICAgICAgInBvdGVudGlhbCI6IDAuMCwKICAgICAgICAgICJmYWN0b3JpemF0aW9uIjogMC4wLAogICAgICAgICAgIlNfZXhhY3QiOiAwLjAsCiAgICAgICAgICAiQ19leGFjdCI6IDAuMAogICAgICAgIH0sCiAgICAgICAgIm1hbnVmYWN0dXJlZF9mb3JjaW5nX21heF9hYnNvbHV0ZV9kZWZlY3QiOiAxLjQyMTA4NTQ3MTUyMDIwMDRlLTE0LAogICAgICAgICJ3ZWlnaHRlZF9kcmlmdF9wb3RlbnRpYWxfcmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiA4LjI1NTUzOTc5NTQ2NzYxNGUtMTYsCiAgICAgICAgIm9yaWdpbmFsX2VuZXJneV9yZWxhdGl2ZV9kZWZlY3QiOiAwLjAsCiAgICAgICAgIndlaWdodGVkX2VuZXJneV9yZWxhdGl2ZV9kZWZlY3QiOiAwLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJrIjogMTYsCiAgICAgICAgImNvZWZmaWNpZW50X2lkZW50aXR5X2RlZmVjdHMiOiB7CiAgICAgICAgICAiU19zeW1tZXRyeSI6IDAuMCwKICAgICAgICAgICJyaWNjYXRpIjogMC4wLAogICAgICAgICAgInRyYWNlIjogMC4wLAogICAgICAgICAgIkNfYW50aXN5bW1ldHJ5IjogMC4wLAogICAgICAgICAgInBvdGVudGlhbCI6IDAuMCwKICAgICAgICAgICJmYWN0b3JpemF0aW9uIjogMC4wLAogICAgICAgICAgIlNfZXhhY3QiOiAwLjAsCiAgICAgICAgICAiQ19leGFjdCI6IDAuMAogICAgICAgIH0sCiAgICAgICAgIm1hbnVmYWN0dXJlZF9mb3JjaW5nX21heF9hYnNvbHV0ZV9kZWZlY3QiOiAzLjgxOTE2NzIwNDcxMDUzODVlLTE0LAogICAgICAgICJ3ZWlnaHRlZF9kcmlmdF9wb3RlbnRpYWxfcmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiAxLjA1NDI3OTQ3NzUxNDA3MzRlLTE1LAogICAgICAgICJvcmlnaW5hbF9lbmVyZ3lfcmVsYXRpdmVfZGVmZWN0IjogMy40Mzg1Nzc5ODk2NDcwNTU0ZS0xNiwKICAgICAgICAid2VpZ2h0ZWRfZW5lcmd5X3JlbGF0aXZlX2RlZmVjdCI6IDIuOTIxMzIwODQyMTYzMzA2M2UtMTYKICAgICAgfSwKICAgICAgewogICAgICAgICJrIjogMjAsCiAgICAgICAgImNvZWZmaWNpZW50X2lkZW50aXR5X2RlZmVjdHMiOiB7CiAgICAgICAgICAiU19zeW1tZXRyeSI6IDAuMCwKICAgICAgICAgICJyaWNjYXRpIjogMC4wLAogICAgICAgICAgInRyYWNlIjogMC4wLAogICAgICAgICAgIkNfYW50aXN5bW1ldHJ5IjogMC4wLAogICAgICAgICAgInBvdGVudGlhbCI6IDAuMCwKICAgICAgICAgICJmYWN0b3JpemF0aW9uIjogMC4wLAogICAgICAgICAgIlNfZXhhY3QiOiAwLjAsCiAgICAgICAgICAiQ19leGFjdCI6IDAuMAogICAgICAgIH0sCiAgICAgICAgIm1hbnVmYWN0dXJlZF9mb3JjaW5nX21heF9hYnNvbHV0ZV9kZWZlY3QiOiAyLjg0MjE3MDk0MzA0MDQwMWUtMTQsCiAgICAgICAgIndlaWdodGVkX2RyaWZ0X3BvdGVudGlhbF9yZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDEuMjE4MTgzMTg1MjQwNzI1ZS0xNSwKICAgICAgICAib3JpZ2luYWxfZW5lcmd5X3JlbGF0aXZlX2RlZmVjdCI6IDIuMzAxOTMxNjA0OTcwNDYyZS0xNSwKICAgICAgICAid2VpZ2h0ZWRfZW5lcmd5X3JlbGF0aXZlX2RlZmVjdCI6IDUuMDY3MzUzOTY0NTMyMDY3ZS0xNgogICAgICB9CiAgICBdLAogICAgImV4YWN0X3NvbHV0aW9uX25vcm1fc3F1YXJlZF9kZWZlY3RzIjogewogICAgICAiTDJfc3F1YXJlZCI6IDQuNDQwODkyMDk4NTAwNjI2ZS0xNiwKICAgICAgImdyYWRpZW50X3NxdWFyZWQiOiAwLjAsCiAgICAgICJIMV9zcXVhcmVkIjogOC44ODE3ODQxOTcwMDEyNTJlLTE2CiAgICB9LAogICAgInF1YWRyYXR1cmVfY2hlY2tzIjogWwogICAgICB7CiAgICAgICAgImsiOiAyMCwKICAgICAgICAibiI6IDgsCiAgICAgICAgIm1ldGhvZCI6ICJCIiwKICAgICAgICAiYmFzZV9vcmRlciI6IDgsCiAgICAgICAgImNvbXBhcmlzb25fb3JkZXIiOiAxMiwKICAgICAgICAicmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiAwLjAsCiAgICAgICAgInJlbGF0aXZlX2xvYWRfZGlmZmVyZW5jZSI6IDMuMzU3MDUxMjM2NTUyMzcxN2UtMTYsCiAgICAgICAgInJlbGF0aXZlX25vZGFsX2RpZmZlcmVuY2UiOiA1LjE1MTQ2OTk2MzcxOTIxMmUtMTYsCiAgICAgICAgInJlbGF0aXZlX0wyX2Vycm9yX2RpZmZlcmVuY2UiOiA5LjU0NzkxODAxMTc3NjM0NmUtMTUsCiAgICAgICAgInJlbGF0aXZlX0gxX2Vycm9yX2RpZmZlcmVuY2UiOiAxLjc3NjM1NjgzOTQwMDI1MDVlLTE1CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAiayI6IDIwLAogICAgICAgICJuIjogOCwKICAgICAgICAibWV0aG9kIjogIkUiLAogICAgICAgICJiYXNlX29yZGVyIjogOCwKICAgICAgICAiY29tcGFyaXNvbl9vcmRlciI6IDEyLAogICAgICAgICJyZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDIuMTc2NDc4NDY3MzIwMjY3M2UtMTIsCiAgICAgICAgInJlbGF0aXZlX2xvYWRfZGlmZmVyZW5jZSI6IDkuMTQxNjgxMjc0NjE4ODI2ZS0xMSwKICAgICAgICAicmVsYXRpdmVfbm9kYWxfZGlmZmVyZW5jZSI6IDEuMTk2MzQ3NDYxMDQwNTg2NmUtMDcsCiAgICAgICAgInJlbGF0aXZlX0wyX2Vycm9yX2RpZmZlcmVuY2UiOiA1LjcxNDg3NTgzMDM0MjY5MmUtMDcsCiAgICAgICAgInJlbGF0aXZlX0gxX2Vycm9yX2RpZmZlcmVuY2UiOiAyLjg3MTY5MDQ0NTQzODQ0NmUtMDcKICAgICAgfSwKICAgICAgewogICAgICAgICJrIjogMjAsCiAgICAgICAgIm4iOiAzMiwKICAgICAgICAibWV0aG9kIjogIkIiLAogICAgICAgICJiYXNlX29yZGVyIjogOCwKICAgICAgICAiY29tcGFyaXNvbl9vcmRlciI6IDEyLAogICAgICAgICJyZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDAuMCwKICAgICAgICAicmVsYXRpdmVfbG9hZF9kaWZmZXJlbmNlIjogMi42NzY4NDYwMzAwMjI1MzA1ZS0xNiwKICAgICAgICAicmVsYXRpdmVfbm9kYWxfZGlmZmVyZW5jZSI6IDIuMzYxMDYyMTkyNzY2MzA1NmUtMTQsCiAgICAgICAgInJlbGF0aXZlX0wyX2Vycm9yX2RpZmZlcmVuY2UiOiAyLjM1MzY3MjgxMjIwNTMzMmUtMTQsCiAgICAgICAgInJlbGF0aXZlX0gxX2Vycm9yX2RpZmZlcmVuY2UiOiAyLjM1MzY3MjgxMjIwNTMzMmUtMTQKICAgICAgfSwKICAgICAgewogICAgICAgICJrIjogMjAsCiAgICAgICAgIm4iOiAzMiwKICAgICAgICAibWV0aG9kIjogIkUiLAogICAgICAgICJiYXNlX29yZGVyIjogOCwKICAgICAgICAiY29tcGFyaXNvbl9vcmRlciI6IDEyLAogICAgICAgICJyZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDIuNzMwNTQxOTgzODI0ODcxNmUtMTYsCiAgICAgICAgInJlbGF0aXZlX2xvYWRfZGlmZmVyZW5jZSI6IDIuOTIxODU2NTE0NzgxMzk4ZS0xNiwKICAgICAgICAicmVsYXRpdmVfbm9kYWxfZGlmZmVyZW5jZSI6IDUuNDkxMjM5MTE0Mzg3NTA4ZS0wOSwKICAgICAgICAicmVsYXRpdmVfTDJfZXJyb3JfZGlmZmVyZW5jZSI6IDIuNjY1MTE2NTk4NTIxNDIyZS0wNywKICAgICAgICAicmVsYXRpdmVfSDFfZXJyb3JfZGlmZmVyZW5jZSI6IDguODQ3OTIyNTUwMTgxNDA4ZS0wOAogICAgICB9CiAgICBdCiAgfQp9'}, 'results/noncoercive/timings_raw.csv': {'sha256': 'f034a3196b385e223c634e5cb4a49bdc65672a5dd8c3b743232dff98b135c64d', 'base64': '...'}, 'results/noncoercive/validation.json': {'sha256': 'c7e970f8e61055f6610cc116dcbc932e6715b04fa678b7a41a93718e34a7b845', 'base64': 'ewogICJwYXJhbWV0ZXJfY2hlY2tzIjogWwogICAgewogICAgICAiayI6IDQsCiAgICAgICJjb2VmZmljaWVudF9pZGVudGl0eV9kZWZlY3RzIjogewogICAgICAgICJTX3N5bW1ldHJ5IjogMC4wLAogICAgICAgICJyaWNjYXRpIjogMC4wLAogICAgICAgICJ0cmFjZSI6IDAuMCwKICAgICAgICAiQ19hbnRpc3ltbWV0cnkiOiAwLjAsCiAgICAgICAgInBvdGVudGlhbCI6IDAuMCwKICAgICAgICAiZmFjdG9yaXphdGlvbiI6IDAuMCwKICAgICAgICAiU19leGFjdCI6IDAuMCwKICAgICAgICAiQ19leGFjdCI6IDAuMAogICAgICB9LAogICAgICAibWFudWZhY3R1cmVkX2ZvcmNpbmdfbWF4X2Fic29sdXRlX2RlZmVjdCI6IDcuMTA1NDI3MzU3NjAxMDAyZS0xNSwKICAgICAgIndlaWdodGVkX2RyaWZ0X3BvdGVudGlhbF9yZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDQuMDc3MzczODk0MTgyNTAzZS0xNiwKICAgICAgIm9yaWdpbmFsX2VuZXJneV9yZWxhdGl2ZV9kZWZlY3QiOiAxLjMzNDU1MzgxOTQ2ODA0M2UtMTYsCiAgICAgICJ3ZWlnaHRlZF9lbmVyZ3lfcmVsYXRpdmVfZGVmZWN0IjogMC4wCiAgICB9LAogICAgewogICAgICAiayI6IDgsCiAgICAgICJjb2VmZmljaWVudF9pZGVudGl0eV9kZWZlY3RzIjogewogICAgICAgICJTX3N5bW1ldHJ5IjogMC4wLAogICAgICAgICJyaWNjYXRpIjogMC4wLAogICAgICAgICJ0cmFjZSI6IDAuMCwKICAgICAgICAiQ19hbnRpc3ltbWV0cnkiOiAwLjAsCiAgICAgICAgInBvdGVudGlhbCI6IDAuMCwKICAgICAgICAiZmFjdG9yaXphdGlvbiI6IDAuMCwKICAgICAgICAiU19leGFjdCI6IDAuMCwKICAgICAgICAiQ19leGFjdCI6IDAuMAogICAgICB9LAogICAgICAibWFudWZhY3R1cmVkX2ZvcmNpbmdfbWF4X2Fic29sdXRlX2RlZmVjdCI6IDEuMjg3ODU4NzA4NTY1MTgxNmUtMTQsCiAgICAgICJ3ZWlnaHRlZF9kcmlmdF9wb3RlbnRpYWxfcmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiA1LjU4MjM3MjQ3NjQzMDIxOWUtMTYsCiAgICAgICJvcmlnaW5hbF9lbmVyZ3lfcmVsYXRpdmVfZGVmZWN0IjogMS41NzEwNjQ1NjQ0NTYxNzllLTE2LAogICAgICAid2VpZ2h0ZWRfZW5lcmd5X3JlbGF0aXZlX2RlZmVjdCI6IDIuMDI2MDU1Mjc5NjQ4OTEwM2UtMTYKICAgIH0sCiAgICB7CiAgICAgICJrIjogMTIsCiAgICAgICJjb2VmZmljaWVudF9pZGVudGl0eV9kZWZlY3RzIjogewogICAgICAgICJTX3N5bW1ldHJ5IjogMC4wLAogICAgICAgICJyaWNjYXRpIjogMC4wLAogICAgICAgICJ0cmFjZSI6IDAuMCwKICAgICAgICAiQ19hbnRpc3ltbWV0cnkiOiAwLjAsCiAgICAgICAgInBvdGVudGlhbCI6IDAuMCwKICAgICAgICAiZmFjdG9yaXphdGlvbiI6IDAuMCwKICAgICAgICAiU19leGFjdCI6IDAuMCwKICAgICAgICAiQ19leGFjdCI6IDAuMAogICAgICB9LAogICAgICAibWFudWZhY3R1cmVkX2ZvcmNpbmdfbWF4X2Fic29sdXRlX2RlZmVjdCI6IDEuNDIxMDg1NDcxNTIwMjAwNGUtMTQsCiAgICAgICJ3ZWlnaHRlZF9kcmlmdF9wb3RlbnRpYWxfcmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiA4LjI1NTUzOTc5NTQ2NzYxNGUtMTYsCiAgICAgICJvcmlnaW5hbF9lbmVyZ3lfcmVsYXRpdmVfZGVmZWN0IjogMC4wLAogICAgICAid2VpZ2h0ZWRfZW5lcmd5X3JlbGF0aXZlX2RlZmVjdCI6IDAuMAogICAgfSwKICAgIHsKICAgICAgImsiOiAxNiwKICAgICAgImNvZWZmaWNpZW50X2lkZW50aXR5X2RlZmVjdHMiOiB7CiAgICAgICAgIlNfc3ltbWV0cnkiOiAwLjAsCiAgICAgICAgInJpY2NhdGkiOiAwLjAsCiAgICAgICAgInRyYWNlIjogMC4wLAogICAgICAgICJDX2FudGlzeW1tZXRyeSI6IDAuMCwKICAgICAgICAicG90ZW50aWFsIjogMC4wLAogICAgICAgICJmYWN0b3JpemF0aW9uIjogMC4wLAogICAgICAgICJTX2V4YWN0IjogMC4wLAogICAgICAgICJDX2V4YWN0IjogMC4wCiAgICAgIH0sCiAgICAgICJtYW51ZmFjdHVyZWRfZm9yY2luZ19tYXhfYWJzb2x1dGVfZGVmZWN0IjogMy44MTkxNjcyMDQ3MTA1Mzg1ZS0xNCwKICAgICAgIndlaWdodGVkX2RyaWZ0X3BvdGVudGlhbF9yZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDEuMDU0Mjc5NDc3NTE0MDczNGUtMTUsCiAgICAgICJvcmlnaW5hbF9lbmVyZ3lfcmVsYXRpdmVfZGVmZWN0IjogMy40Mzg1Nzc5ODk2NDcwNTU0ZS0xNiwKICAgICAgIndlaWdodGVkX2VuZXJneV9yZWxhdGl2ZV9kZWZlY3QiOiAyLjkyMTMyMDg0MjE2MzMwNjNlLTE2CiAgICB9LAogICAgewogICAgICAiayI6IDIwLAogICAgICAiY29lZmZpY2llbnRfaWRlbnRpdHlfZGVmZWN0cyI6IHsKICAgICAgICAiU19zeW1tZXRyeSI6IDAuMCwKICAgICAgICAicmljY2F0aSI6IDAuMCwKICAgICAgICAidHJhY2UiOiAwLjAsCiAgICAgICAgIkNfYW50aXN5bW1ldHJ5IjogMC4wLAogICAgICAgICJwb3RlbnRpYWwiOiAwLjAsCiAgICAgICAgImZhY3Rvcml6YXRpb24iOiAwLjAsCiAgICAgICAgIlNfZXhhY3QiOiAwLjAsCiAgICAgICAgIkNfZXhhY3QiOiAwLjAKICAgICAgfSwKICAgICAgIm1hbnVmYWN0dXJlZF9mb3JjaW5nX21heF9hYnNvbHV0ZV9kZWZlY3QiOiAyLjg0MjE3MDk0MzA0MDQwMWUtMTQsCiAgICAgICJ3ZWlnaHRlZF9kcmlmdF9wb3RlbnRpYWxfcmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiAxLjIxODE4MzE4NTI0MDcyNWUtMTUsCiAgICAgICJvcmlnaW5hbF9lbmVyZ3lfcmVsYXRpdmVfZGVmZWN0IjogMi4zMDE5MzE2MDQ5NzA0NjJlLTE1LAogICAgICAid2VpZ2h0ZWRfZW5lcmd5X3JlbGF0aXZlX2RlZmVjdCI6IDUuMDY3MzUzOTY0NTMyMDY3ZS0xNgogICAgfQogIF0sCiAgImV4YWN0X3NvbHV0aW9uX25vcm1fc3F1YXJlZF9kZWZlY3RzIjogewogICAgIkwyX3NxdWFyZWQiOiA0LjQ0MDg5MjA5ODUwMDYyNmUtMTYsCiAgICAiZ3JhZGllbnRfc3F1YXJlZCI6IDAuMCwKICAgICJIMV9zcXVhcmVkIjogOC44ODE3ODQxOTcwMDEyNTJlLTE2CiAgfSwKICAicXVhZHJhdHVyZV9jaGVja3MiOiBbCiAgICB7CiAgICAgICJrIjogMjAsCiAgICAgICJuIjogOCwKICAgICAgIm1ldGhvZCI6ICJCIiwKICAgICAgImJhc2Vfb3JkZXIiOiA4LAogICAgICAiY29tcGFyaXNvbl9vcmRlciI6IDEyLAogICAgICAicmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiAwLjAsCiAgICAgICJyZWxhdGl2ZV9sb2FkX2RpZmZlcmVuY2UiOiAzLjM1NzA1MTIzNjU1MjM3MTdlLTE2LAogICAgICAicmVsYXRpdmVfbm9kYWxfZGlmZmVyZW5jZSI6IDUuMTUxNDY5OTYzNzE5MjEyZS0xNiwKICAgICAgInJlbGF0aXZlX0wyX2Vycm9yX2RpZmZlcmVuY2UiOiA5LjU0NzkxODAxMTc3NjM0NmUtMTUsCiAgICAgICJyZWxhdGl2ZV9IMV9lcnJvcl9kaWZmZXJlbmNlIjogMS43NzYzNTY4Mzk0MDAyNTA1ZS0xNQogICAgfSwKICAgIHsKICAgICAgImsiOiAyMCwKICAgICAgIm4iOiA4LAogICAgICAibWV0aG9kIjogIkUiLAogICAgICAiYmFzZV9vcmRlciI6IDgsCiAgICAgICJjb21wYXJpc29uX29yZGVyIjogMTIsCiAgICAgICJyZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDIuMTc2NDc4NDY3MzIwMjY3M2UtMTIsCiAgICAgICJyZWxhdGl2ZV9sb2FkX2RpZmZlcmVuY2UiOiA5LjE0MTY4MTI3NDYxODgyNmUtMTEsCiAgICAgICJyZWxhdGl2ZV9ub2RhbF9kaWZmZXJlbmNlIjogMS4xOTYzNDc0NjEwNDA1ODY2ZS0wNywKICAgICAgInJlbGF0aXZlX0wyX2Vycm9yX2RpZmZlcmVuY2UiOiA1LjcxNDg3NTgzMDM0MjY5MmUtMDcsCiAgICAgICJyZWxhdGl2ZV9IMV9lcnJvcl9kaWZmZXJlbmNlIjogMi44NzE2OTA0NDU0Mzg0NDZlLTA3CiAgICB9LAogICAgewogICAgICAiayI6IDIwLAogICAgICAibiI6IDMyLAogICAgICAibWV0aG9kIjogIkIiLAogICAgICAiYmFzZV9vcmRlciI6IDgsCiAgICAgICJjb21wYXJpc29uX29yZGVyIjogMTIsCiAgICAgICJyZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDAuMCwKICAgICAgInJlbGF0aXZlX2xvYWRfZGlmZmVyZW5jZSI6IDIuNjc2ODQ2MDMwMDIyNTMwNWUtMTYsCiAgICAgICJyZWxhdGl2ZV9ub2RhbF9kaWZmZXJlbmNlIjogMi4zNjEwNjIxOTI3NjYzMDU2ZS0xNCwKICAgICAgInJlbGF0aXZlX0wyX2Vycm9yX2RpZmZlcmVuY2UiOiAyLjM1MzY3MjgxMjIwNTMzMmUtMTQsCiAgICAgICJyZWxhdGl2ZV9IMV9lcnJvcl9kaWZmZXJlbmNlIjogMi4zNTM2NzI4MTIyMDUzMzJlLTE0CiAgICB9LAogICAgewogICAgICAiayI6IDIwLAogICAgICAibiI6IDMyLAogICAgICAibWV0aG9kIjogIkUiLAogICAgICAiYmFzZV9vcmRlciI6IDgsCiAgICAgICJjb21wYXJpc29uX29yZGVyIjogMTIsCiAgICAgICJyZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDIuNzMwNTQxOTgzODI0ODcxNmUtMTYsCiAgICAgICJyZWxhdGl2ZV9sb2FkX2RpZmZlcmVuY2UiOiAyLjkyMTg1NjUxNDc4MTM5OGUtMTYsCiAgICAgICJyZWxhdGl2ZV9ub2RhbF9kaWZmZXJlbmNlIjogNS40OTEyMzkxMTQzODc1MDhlLTA5LAogICAgICAicmVsYXRpdmVfTDJfZXJyb3JfZGlmZmVyZW5jZSI6IDIuNjY1MTE2NTk4NTIxNDIyZS0wNywKICAgICAgInJlbGF0aXZlX0gxX2Vycm9yX2RpZmZlcmVuY2UiOiA4Ljg0NzkyMjU1MDE4MTQwOGUtMDgKICAgIH0KICBdCn0='}, 'results/positive/convergence.csv': {'sha256': '844a92e516a3b8d3f0396eb23e91df64747f5e2c2bd9f8bbc33afefc733a3606', 'base64': 'bixoLG5kb2YsdHJpYW5nbGVzLG1ldGhvZCxsMixoMSxsMl9yZWxhdGl2ZSxoMV9yZWxhdGl2ZSxoMV9zZW1pbm9ybSxjb2VmZl9tZWRpYW5fcyxhc3NlbWJseV9tZWRpYW5fcyxzb2x2ZV9tZWRpYW5fcyx0b3RhbF9tZWRpYW5fcyx0b3RhbF9xMjVfcyx0b3RhbF9xNzVfcyxtaW5fdG90YWxfcyxtYXhfdG90YWxfcyxzY2FsZWRfYWxnZWJyYWljX3Jlc2lkdWFsLG1hdHJpeF9ubnoNCjgsMC4zNTM1NTMzOTA1OTMyNzM4LDQ5LDEyOCxCLDAuMDI4NTcyNDgzOTg1NTk2Mjc0LDAuNDg3MjYzNDY3MTc3NTc2NTcsMC4wMjY3ODY3MDM3MzY0OTY1MDgsMC4xODY0OTE2OTc2MzgxOTQsMC40ODY0MjUwMTk1MDk0ODgzLDAuMCwwLjAwMjE0Mzc2NTk5NjE1MjA2OTQsMC4wMDAxMTk5Mzk5OTk0MjIwNTg0NiwwLjAwMjI2MjYwNDk5NzY2Nzk0OTYsMC4wMDIyNTQ1MzI0OTczMTc5OTYsMC4wMDIyOTQ1MjgwMDM0MDM2ODQsMC4wMDIxNDM1NzcwMDMxNTMwNDEsMC4wMDI4OTQ5MTQwMDMyNjM2OTMzLDguMzE3NDIyNzIwMjQyMzQzZS0xNywyODkNCjgsMC4zNTM1NTMzOTA1OTMyNzM4LDQ5LDEyOCxFLDAuMDM0MjAyMjUyODg4OTg5NjEsMC40ODc4MzUzODgxNzczMDQxLDAuMDMyMDY0NjEyMDgzNDI3NzYsMC4xODY3MTA1OTA1NDc5NjE5NiwwLjQ4NjYzNDk0NzIxOTU5NTYsOC40ODI4MDAzMzA5NjY3MTFlLTA1LDAuMDAyNTM1NDk0MDAwNzY3MzU3NiwwLjAwMDE0ODUwMjAwMzg2NzE3OTE2LDAuMDAyODQ5Njc1OTk4NzMxOTgyLDAuMDAyNjAxMjEyNTAxNzU4NzA5NSwwLjAwMzIyNjQ2MTQ5NjUyNDI4LDAuMDAyNTcxMDE4MDA1MjU1NjA5OCwwLjAwNDI5MTY1OTk5OTU4MDU4OCw0LjUyODM4MDY1NDk1NTMwM2UtMTcsMjg5DQoxNiwwLjE3Njc3NjY5NTI5NjYzNjksMjI1LDUxMixCLDAuMDA3MTM0OTI4NDM5MDY3MTI1LDAuMjQzNTAzODM1ODQyOTcxMDMsMC4wMDY2ODg5OTU0MTE2MjU0MywwLjA5MzE5Njg5ODE2MDI2ODE5LDAuMjQzMzk5MjgyNzk3NjQ5MSwwLjAsMC4wMDc0NjcyNTM5OTk0OTM1NTA1LDAuMDAwMzczNzM5OTk2NTc1NzUwNCwwLjAwNzk2ODY3NzAwNDEwMDc1NSwwLjAwNzU5NDkwNjAwMTAzMTYzMiwwLjAwODA2Mjc3ODAwMDc0ODk0OSwwLjAwNzM2MDg0NTAwMzg1OTE0LDAuMDA4ODExMzYyMDAwNDE1MTAyLDIuMzgwMDkzMjEwMzk1Mjk3NmUtMTYsMTQ1Nw0KMTYsMC4xNzY3NzY2OTUyOTY2MzY5LDIyNSw1MTIsRSwwLjAwODY1OTY5NzkyNzUwMDIyNSwwLjI0MzU4MDkzNDgzMzQwODMsMC4wMDgxMTg0NjY4MDcwMzE0NjIsMC4wOTMyMjY0MDY0NzA2MjA1NCwwLjI0MzQyNjk1Mjk5ODQ3NDU4LDcuMDY2NTk5OTM3Mzg4NjczZS0wNSwwLjAwODIwNzU2NDk5ODg4Mzc1NCwwLjAwMDQwMDc3MTAwMTI5NTY1MjI0LDAuMDA4NzQxNDI3MDAxOTUwNzk3LDAuMDA4NjE2NDc0MDAyMzM3NzgyLDAuMDA5MzMyMjczMDAyNjU1NzExLDAuMDA4NTA5NDM5MDAxNTAwMTQxLDAuMDA5MzkxNTQyMDAwMjMwNDAyLDMuODM2Mzg1MzA3MjEwMDczZS0xNywxNDU3DQozMiwwLjA4ODM4ODM0NzY0ODMxODQ1LDk2MSwyMDQ4LEIsMC4wMDE3ODMyNjk1ODIwMjc5MzcyLDAuMTIxNzI1ODM3NDI4MzQ0ODQsMC4wMDE2NzE4MTUyMzMxNTExOTEyLDAuMDQ2NTg4NDY3MjIwNjkxMTU0LDAuMTIxNzEyNzc0Mzc5ODA2MzgsMC4wLDAuMDMyODc1OTY5OTk4NzEzMTIsMC4wMDE3ODcyMzA5OTgxNjEyNDg5LDAuMDM0ODM5OTI3MDAwNTc1NywwLjAzMTE1NzkwMjAwMTcyMzQxNCwwLjAzNzAxNjAyMjAwMDg5ODM3NCwwLjAyODk3OTUxMzk5NzkxMDU0NCwwLjA0MTMzNDYyMjAwMTE1OTAwNSwyLjA3MTMxNTYyMDg4OTc3MzNlLTE2LDY0ODENCjMyLDAuMDg4Mzg4MzQ3NjQ4MzE4NDUsOTYxLDIwNDgsRSwwLjAwMjE3MzY1NjQyNzM2MDM5MDUsMC4xMjE3MzU5MTI2MjIxMzMyNCwwLjAwMjAzNzgwMjkwMDY1MDM2NiwwLjA0NjU5MjMyMzMyNzU0MTM0NiwwLjEyMTcxNjUwNTIwNjQ4MTU2LDcuNzEwNjAwMDM4OTkwMzc4ZS0wNSwwLjAzMTQ2NzMyNTk5OTk4MzU4NiwwLjAwMTk1Mjk3MDk5NzA4MTEzODIsMC4wMzMzNzQ4MDgwMDEwOTkxNTQsMC4wMzMxODQ4ODg1MDM0NTYwMiwwLjAzMzg5MzkwMjUwMjMxMzcyLDAuMDMyOTI0OTMzOTk5MzExMTgsMC4wMzU1MjI1NjIwMDA1NTEyNiwyLjU0MDY3NzgwODIxNDI1NjhlLTE3LDY0ODENCjY0LDAuMDQ0MTk0MTczODI0MTU5MjIsMzk2OSw4MTkyLEIsMC4wMDA0NDU3OTI5MDYyNDcyOTMwNiwwLjA2MDg1OTMwMTk5OTM5NjM3NiwwLjAwMDQxNzkzMDg0OTYwNjgzNzI1LDAuMDIzMjkyODQ5Mzc1MDcyNjksMC4wNjA4NTc2NjkyNjYzOTk1NCwwLjAsMC4xMTQ1ODczMzgwMDEwMDE2MywwLjAwOTYwNTA0MTk5NjExMDIzLDAuMTI0MjQyNzM1MDAwNzM0NjgsMC4xMTk4NDQzMzk5OTg5NjI4MiwwLjEyODI4ODY4MTAwMDQxMjksMC4xMTY1NzM0MTIwMDA0MzQ0NywwLjEzOTYxODMxMTAwNDU5Njc2LDQuNzY1MzcyNzc1NTA0MDY1ZS0xNiwyNzI4MQ0KNjQsMC4wNDQxOTQxNzM4MjQxNTkyMiwzOTY5LDgxOTIsRSwwLjAwMDU0NDAyMjM2NDA0Mjk2MzQsMC4wNjA4NjA1ODMyODQ5NjIzMzQsMC4wMDA1MTAwMjA5NjYyOTAyNzgyLDAuMDIzMjkzMzM5NzY1MDQ5NDY4LDAuMDYwODU4MTUxNzc0ODcxMTk0LDkuMzgzOTk5NzEzNDYzNzA5ZS0wNSwwLjEzMDAzNTkxNzAwNTQ0MzQ4LDAuMDEwMDMzNzM1MDAxMzQzMzAyLDAuMTQwNDkzMTk5MDAzNjg1NzMsMC4xMzY4MDA3NTM1MDIwMjk3LDAuMTQxMjg2OTM0NTAwMzg1NTQsMC4xMzI4NDYyODAwMDExMDM4OCwwLjE0NDczMjQ1ODk5NzE2ODU4LDIuMjcwNjI1NjgyMjgyOTk5NmUtMTcsMjcyODENCjEyOCwwLjAyMjA5NzA4NjkxMjA3OTYxLDE2MTI5LDMyNzY4LEIsMC4wMDAxMTE0NDY4ODM3NzQxNjQ5OCwwLjAzMDQyOTE4NjgyNjY5MTQ3NiwwLjAwMDEwNDQ4MTQ1MzUzODI3OTY2LDAuMDExNjQ2MjQ3MDMzMzE0NjUzLDAuMDMwNDI4OTgyNzM4OTI0OTI1LDAuMCwwLjQzOTc2MjE4Njk5MzkwMywwLjA2MDc1NjI3NzAwMTkyODU0LDAuNTAzMjI0ODg3OTk0MjkyNywwLjQ5NjEyMzgxMDk5NzIzMjgsMC41MTgwMDIwMjQ0OTg4NTU1LDAuNDc5NTg5MDQ0OTk2ODI4MywwLjU0NTAxNTE5MDk5NzIxOTIsNC4zMDkzNTQwMjYxNDQxODI2ZS0xNiwxMTE4ODkNCjEyOCwwLjAyMjA5NzA4NjkxMjA3OTYxLDE2MTI5LDMyNzY4LEUsMC4wMDAxMzYwNDU0ODQ3ODkxMzY1MiwwLjAzMDQyOTM0Nzg2NzUyNTk2NywwLjAwMDEyNzU0MjY0MTk4OTgxNTUsMC4wMTE2NDYzMDg2Njg5MTk2OTMsMC4wMzA0MjkwNDM3NDU1NTYyNCw5LjMxMTAwMDAzMDcyMDYwNmUtMDUsMC40OTI5NDM1NDM5OTg5OTAyLDAuMDYxMDA1MTYwOTk4MzIwMjEsMC41NTIyNjQ2OTc5OTUyNjk3LDAuNTQ0NjU5MDQxMDAyODg3NSwwLjU3MDc2MjUxMzk5ODQ4ODQsMC41Mjg4ODUwNTUwMDU5NTc4LDAuNjgyMTg0NjA1OTk3NDYzMiwyLjY3ODY2MDUxNDMyNDExMjdlLTE3LDExMTg4OQ0KMjU2LDAuMDExMDQ4NTQzNDU2MDM5ODA2LDY1MDI1LDEzMTA3MixCLDIuNzg2MTY0Mzc2NTI2MjcwNmUtMDUsMC4wMTUyMTQ1MzQ5OTUyNDAyOTgsMi42MTIwMjkxMDI5OTMzNzllLTA1LDAuMDA1ODIzMTAxMTU4MTMzODc5LDAuMDE1MjE0NTA5NDg0Mzc3MDI0LDAuMCwxLjg3MzAyOTA0NjAwMDczODIsMC40NTY5MjE1OTY5OTY4MzU0MywyLjMxNDU0NDQ3NjAwMTE3NCwyLjI1MDAyNzk2MzQ5ODcyMzcsMi40MDgzMDk5MjIwMDA2MjYsMi4xMDE1NzAyMzIwMDAwODU1LDIuNTQxODI5Mzk4MDAzNzU2LDUuNDk0OTQ5NTYzNDQ1NTUxZS0xNiw0NTMxMzcNCjI1NiwwLjAxMTA0ODU0MzQ1NjAzOTgwNiw2NTAyNSwxMzEwNzIsRSwzLjQwMTM5MjA2MDQyNDM4NWUtMDUsMC4wMTUyMTQ1NTUxNTcwMDM0NCwzLjE4ODgwNTA1NjY0Nzg2MTRlLTA1LDAuMDA1ODIzMTA4ODc0NzAxMzgsMC4wMTUyMTQ1MTcxMzU5MDM2MjcsMC4wMDAxMDU1Nzg5OTgwNDU1MzAxNywyLjA2MDg1MzEwNTk5ODgzMSwwLjQ1MzgwOTI5ODAwMDEyMzgsMi40OTQ1MDQ3Mjc5OTc5OTE3LDIuNDExMzQyMjkyMDAxNzkxLDIuNjQ1NjQ2OTgxMDAwMDI3LDIuMzQyNDM4MzExMDAzMjA2OCwyLjc4Nzc0OTM1OTAwNTgyOTMsMy43MDQyODM3NzQ5NjQ1MTg1ZS0xNyw0NTMxMzcNCjUxMiwwLjAwNTUyNDI3MTcyODAxOTkwMywyNjExMjEsNTI0Mjg4LEIsNi45NjU0MDY1MzUxMjM3NzhlLTA2LDAuMDA3NjA3MjYwMTgyNjEyNzM5LDYuNTMwMDY4NjI2Njc4NTQyZS0wNiwwLjAwMjkxMTU0Nzc3OTM3Mzg3NCwwLjAwNzYwNzI1Njk5Mzc1NzgwMywwLjAsNy41MDg1NTc2OTAwMDE5MDgsNC43MDMxNTQ5MzYwMDM1NTcsMTIuMzk5MzE3MDE0OTk4ODE1LDEyLjAzNzYxNDY1MTAwMjUxNCwxMi41MzYzNDAwMjU5OTkzOCwxMS4zMjM1ODQ1MzY5OTk5ODgsMTIuNjc4NjU3MjE0MDA0NDg4LDEuMDM5ODc1MjUzNDIwMzcwNGUtMTUsMTgyMzc2MQ0KNTEyLDAuMDA1NTI0MjcxNzI4MDE5OTAzLDI2MTEyMSw1MjQyODgsRSw4LjUwMzY0MTIwNzQyMzAxN2UtMDYsMC4wMDc2MDcyNjI3MDM5MDYxNTYsNy45NzIxNjM2MzE5NTkwNzhlLTA2LDAuMDAyOTExNTQ4NzQ0MzU1NDkzLDAuMDA3NjA3MjU3OTUxMDgzODA3LDkuODQxODAwMDU2MTY1MDg0ZS0wNSw4LjQ4NDI1MDMwNTAwMjIwNiw0Ljc4NzU4MzMzMTk5OTQ4NSwxMy4zMjYzMjE0MDAwMDM1MDgsMTIuNTg0Mjg3OTI0NDk5MjIyLDEzLjYxMTg2MTY5MzQ5ODIxMywxMS43MzA4OTQ1MzUwMDE0MjYsMTQuMjA0OTIyNTQzOTk4NjkxLDUuNTkxODg2MjI3NTY2NjkyZS0xNywxODIzNzYxDQo='}, 'results/positive/metadata.json': {'sha256': 'daf8593adc65e45bb564b08f07a6dc907a6e4d362bec4706afcfd99e11f1ffa7', 'base64': 'ewogICJzdGFydGVkX3V0YyI6ICIyMDI2LTEwLTE2VDE0OjIyOjAwLjQxNDgyMSswMDowMCIsCiAgImNvbXBsZXRlZF91dGMiOiAiMjAyNi0xMC0xNlQxNDoyNjoyNS45NzU1ODQrMDA6MDAiLAogICJzb3VyY2Vfc2hhMjU2IjogIjQxYWUxZjg2MDYzZTgyZmNmZmNlN2U5ZDE4Mzk1OGM2YTYxNDc3YWRkNTM5MWU4ZDg3MGMwMTJmNjBhODE1N2QiLAogICJkZXNjcmlwdGlvbiI6ICJQb3NpdGl2ZS10cmFjZSBPVTogUDEgb3JpZ2luYWwgQiB2ZXJzdXMgd2VpZ2h0ZWQgRSwgc2FtZSBtZXNoIGFuZCBzcGFyc2UgTFUiLAogICJleGVjdXRpb25fZW52aXJvbm1lbnQiOiAiYWN0dWFsIG1hY2hpbmUgcnVubmluZyB0aGlzIHNjcmlwdDsgbm90IG5lY2Vzc2FyaWx5IEdvb2dsZSBDb2xhYiIsCiAgInB5dGhvbiI6ICIzLjEyLjE0IChtYWluLCBBdWcgMjUgMjAyNiwgMTQ6MDA6NDkpIFtDbGFuZyAyMi4xLjMgXSIsCiAgIm51bXB5IjogIjIuMy41IiwKICAic2NpcHkiOiAiMS4xNy4wIiwKICAicGxhdGZvcm0iOiAiTGludXgtNi4xOC40NC14ODZfNjQtd2l0aC1nbGliYzIuMzkiLAogICJjcHUiOiAiQU1EIEVQWUMgOVY3NCA4MC1Db3JlIFByb2Nlc3NvciIsCiAgImxvZ2ljYWxfY3B1X2NvdW50IjogOSwKICAibnVtZXJpY2FsX3RocmVhZHMiOiAxLAogICJ0aHJlYWRwb29sX2luZm9fZHVyaW5nX3J1biI6IFsKICAgIHsKICAgICAgInVzZXJfYXBpIjogImJsYXMiLAogICAgICAiaW50ZXJuYWxfYXBpIjogIm9wZW5ibGFzIiwKICAgICAgIm51bV90aHJlYWRzIjogMSwKICAgICAgInByZWZpeCI6ICJsaWJzY2lweV9vcGVuYmxhcyIsCiAgICAgICJmaWxlcGF0aCI6ICIvb3B0L2NvZGV4L3J1bnRpbWVzL2NvZGV4LXByaW1hcnktcnVudGltZS9kZXBlbmRlbmNpZXMvcHl0aG9uL2xpYi9weXRob24zLjEyL3NpdGUtcGFja2FnZXMvbnVtcHkubGlicy9saWJzY2lweV9vcGVuYmxhczY0Xy1mZGRlNTc3OC5zbyIsCiAgICAgICJ2ZXJzaW9uIjogIjAuMy4zMCIsCiAgICAgICJ0aHJlYWRpbmdfbGF5ZXIiOiAicHRocmVhZHMiLAogICAgICAiYXJjaGl0ZWN0dXJlIjogIlNreWxha2VYIgogICAgfSwKICAgIHsKICAgICAgInVzZXJfYXBpIjogImJsYXMiLAogICAgICAiaW50ZXJuYWxfYXBpIjogIm9wZW5ibGFzIiwKICAgICAgIm51bV90aHJlYWRzIjogMSwKICAgICAgInByZWZpeCI6ICJsaWJzY2lweV9vcGVuYmxhcyIsCiAgICAgICJmaWxlcGF0aCI6ICIvb3B0L2NvZGV4L3J1bnRpbWVzL2NvZGV4LXByaW1hcnktcnVudGltZS9kZXBlbmRlbmNpZXMvcHl0aG9uL2xpYi9weXRob24zLjEyL3NpdGUtcGFja2FnZXMvc2NpcHkubGlicy9saWJzY2lweV9vcGVuYmxhcy02Y2RjM2I0YS5zbyIsCiAgICAgICJ2ZXJzaW9uIjogIjAuMy4zMCIsCiAgICAgICJ0aHJlYWRpbmdfbGF5ZXIiOiAicHRocmVhZHMiLAogICAgICAiYXJjaGl0ZWN0dXJlIjogIlNreWxha2VYIgogICAgfQogIF0sCiAgIkEiOiBbCiAgICBbCiAgICAgIDIuMCwKICAgICAgMS4wCiAgICBdLAogICAgWwogICAgICAxLjAsCiAgICAgIDIuMAogICAgXQogIF0sCiAgIkciOiBbCiAgICBbCiAgICAgIDMuMCwKICAgICAgNC4wCiAgICBdLAogICAgWwogICAgICAxLjAsCiAgICAgIDMuMAogICAgXQogIF0sCiAgImNvZWZmaWNpZW50cyI6IHsKICAgICJTIjogWwogICAgICBbCiAgICAgICAgMS4wLAogICAgICAgIDAuNQogICAgICBdLAogICAgICBbCiAgICAgICAgMC41LAogICAgICAgIDEuNQogICAgICBdCiAgICBdLAogICAgIkgiOiBbCiAgICAgIFsKICAgICAgICAwLjUsCiAgICAgICAgMS41CiAgICAgIF0sCiAgICAgIFsKICAgICAgICAtMS4wLAogICAgICAgIC0wLjUKICAgICAgXQogICAgXSwKICAgICJDIjogWwogICAgICBbCiAgICAgICAgMC4wLAogICAgICAgIC0wLjUKICAgICAgXSwKICAgICAgWwogICAgICAgIDAuNSwKICAgICAgICAwLjAKICAgICAgXQogICAgXSwKICAgICJNIjogWwogICAgICBbCiAgICAgICAgMS4wLAogICAgICAgIDAuMAogICAgICBdLAogICAgICBbCiAgICAgICAgMS4wLAogICAgICAgIDEuMAogICAgICBdCiAgICBdCiAgfSwKICAiZXhhY3Rfc29sdXRpb24iOiAiKDEteF4yKSgxLXleMikiLAogICJmb3JjaW5nIjogIjQrNCp4XjIrNCp5XjIrNip4KnktMTIqeF4yKnleMi04KngqeV4zLTIqeF4zKnkiLAogICJkb21haW4iOiAiKC0xLDEpXjIiLAogICJtZXNoIjogInVuaWZvcm0gbi1ieS1uIHNxdWFyZXM7IHNvdXRod2VzdC10by1ub3J0aGVhc3QgZGlhZ29uYWxzIiwKICAic2l6ZXMiOiBbCiAgICA4LAogICAgMTYsCiAgICAzMiwKICAgIDY0LAogICAgMTI4LAogICAgMjU2LAogICAgNTEyCiAgXSwKICAicmVwZXRpdGlvbnMiOiA3LAogICJ3YXJtdXBzX3Blcl9tZXNoX3Blcl9tZXRob2QiOiAxLAogICJhc3NlbWJseV9kdWZmeV9vcmRlciI6IDgsCiAgImVycm9yX2R1ZmZ5X29yZGVyIjogOCwKICAiYmF0Y2hfc2l6ZSI6IDQwOTYsCiAgInNvbHZlciI6ICJzY2lweS5zcGFyc2UubGluYWxnLnNwbHU7IFN1cGVyTFU7IHBlcm1jX3NwZWM9Q09MQU1EOyBkaWFnX3Bpdm90X3RocmVzaD0xLjAiLAogICJ0aW1pbmdfdW5pdCI6ICJzZWNvbmRzIiwKICAidGltaW5nX3N0YXRpc3RpYyI6ICJtZWRpYW4gb2YgdG90YWwgdGltZSBvdmVyIHJlcGV0aXRpb25zIiwKICAidGltaW5nX2luY2x1ZGVzIjogWwogICAgIkU6IGNvbXB1dGUgUyxILEMsTTsgQjogY29lZmZpY2llbnQgcHJlcGFyYXRpb249MCIsCiAgICAiZ2VvbWV0cnkgYW5kIG1hdHJpeC9sb2FkIGFzc2VtYmx5LCBpbmNsdWRpbmcgZGVuc2l0eSBldmFsdWF0aW9uIiwKICAgICJzcGFyc2UgbWF0cml4IGNvbnN0cnVjdGlvbiBhbmQgaG9tb2dlbmVvdXMgRGlyaWNobGV0IGV4dHJhY3Rpb24iLAogICAgImZyZXNoIHNwYXJzZSBMVSBmYWN0b3JpemF0aW9uIGFuZCB0cmlhbmd1bGFyIHNvbHZlcyIKICBdLAogICJ0aW1pbmdfZXhjbHVkZXMiOiBbCiAgICAiY29tbW9uIG1lc2gvY29ubmVjdGl2aXR5IGNyZWF0aW9uIiwKICAgICJxdWFkcmF0dXJlLXJ1bGUgY29uc3RydWN0aW9uIiwKICAgICJ3YXJtdXAiLAogICAgImdhcmJhZ2UgY29sbGVjdGlvbiIsCiAgICAiZ2xvYmFsIG5vZGFsIG91dHB1dCBleHBhbnNpb24iLAogICAgImVycm9yL3Jlc2lkdWFsIGV2YWx1YXRpb24iLAogICAgIm91dHB1dCBhbmQgcGxvdHRpbmciCiAgXSwKICAiY29tcG9uZW50X21lZGlhbnNfbm90ZSI6ICJFYWNoIGNvbXBvbmVudCBpcyBtZWRpYW4gc2VwYXJhdGVseTsgdGhlaXIgc3VtIG5lZWQgbm90IGVxdWFsIG1lZGlhbiB0b3RhbC4iLAogICJIMV9lcnJvcl9kZWZpbml0aW9uIjogInNxcnQoTDJfZXJyb3JeMiArIGdyYWRpZW50X0wyX2Vycm9yXjIpLCB1bndlaWdodGVkIiwKICAicmVsYXRpdmVfZXJyb3JfZGVmaW5pdGlvbiI6ICJsMi8oMTYvMTUpLCBoMS8oMTYqc3FydCg2KS8xNSk7IGRpbWVuc2lvbmxlc3MgcmF0aW9zIiwKICAiY29lZmZpY2llbnRfY29uc3RydWN0aW9uIjogIkV4cGFuZGVkIHBsYW5hciBTIGZvcm11bGE7IEM9W1swLC1ldGEvKDIqdGF1KV0sW2V0YS8oMip0YXUpLDBdXSwgbm8gUyBpbnZlcnNlIiwKICAidmFsaWRhdGlvbiI6IHsKICAgICJjb2VmZmljaWVudF9pZGVudGl0eV9kZWZlY3RzIjogewogICAgICAiU19zeW1tZXRyeSI6IDAuMCwKICAgICAgInJpY2NhdGkiOiAwLjAsCiAgICAgICJ0cmFjZSI6IDAuMCwKICAgICAgIkNfYW50aXN5bW1ldHJ5IjogMC4wLAogICAgICAicG90ZW50aWFsIjogMC4wLAogICAgICAiZmFjdG9yaXphdGlvbiI6IDAuMCwKICAgICAgIlNfZXhhY3QiOiAwLjAsCiAgICAgICJDX2V4YWN0IjogMC4wCiAgICB9LAogICAgIm1hbnVmYWN0dXJlZF9mb3JjaW5nX21heF9hYnNvbHV0ZV9kZWZlY3QiOiAzLjU1MjcxMzY3ODgwMDUwMWUtMTUsCiAgICAid2VpZ2h0ZWRfZHJpZnRfcG90ZW50aWFsX3JlbGF0aXZlX21hdHJpeF9kaWZmZXJlbmNlIjogMS4yNzczNjgwNTM1ODA3MzFlLTE2LAogICAgIm9yaWdpbmFsX2VuZXJneV9yZWxhdGl2ZV9kZWZlY3QiOiAxLjcxNDkwNTg4MDM4NDYwNWUtMTYsCiAgICAiZXhhY3Rfc29sdXRpb25fbm9ybV9zcXVhcmVkX2RlZmVjdHMiOiB7CiAgICAgICJMMl9zcXVhcmVkIjogNC40NDA4OTIwOTg1MDA2MjZlLTE2LAogICAgICAiZ3JhZGllbnRfc3F1YXJlZCI6IDAuMCwKICAgICAgIkgxX3NxdWFyZWQiOiA4Ljg4MTc4NDE5NzAwMTI1MmUtMTYKICAgIH0sCiAgICAicXVhZHJhdHVyZV9jaGVja3MiOiBbCiAgICAgIHsKICAgICAgICAibiI6IDgsCiAgICAgICAgImZvcm11bGF0aW9uIjogIkIiLAogICAgICAgICJiYXNlbGluZV9vcmRlciI6IDgsCiAgICAgICAgImhpZ2hlcl9vcmRlciI6IDEyLAogICAgICAgICJyZWxhdGl2ZV9tYXRyaXhfZGlmZmVyZW5jZSI6IDAuMCwKICAgICAgICAicmVsYXRpdmVfbG9hZF9kaWZmZXJlbmNlIjogMy4wNzYwNDczMTY0ODU2NThlLTE2LAogICAgICAgICJyZWxhdGl2ZV9ub2RhbF9kaWZmZXJlbmNlIjogMy43MTk2NTE1Nzk1MTQzOTM1ZS0xNgogICAgICB9LAogICAgICB7CiAgICAgICAgIm4iOiA4LAogICAgICAgICJmb3JtdWxhdGlvbiI6ICJFIiwKICAgICAgICAiYmFzZWxpbmVfb3JkZXIiOiA4LAogICAgICAgICJoaWdoZXJfb3JkZXIiOiAxMiwKICAgICAgICAicmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiA0LjA4MDg1NDQ5MTExNjUwNWUtMTYsCiAgICAgICAgInJlbGF0aXZlX2xvYWRfZGlmZmVyZW5jZSI6IDguMzExOTc0MjIwMTQ5NDM4ZS0xNSwKICAgICAgICAicmVsYXRpdmVfbm9kYWxfZGlmZmVyZW5jZSI6IDkuNzE3MTM3ODA1MzM0ODc4ZS0xNgogICAgICB9LAogICAgICB7CiAgICAgICAgIm4iOiAzMiwKICAgICAgICAiZm9ybXVsYXRpb24iOiAiQiIsCiAgICAgICAgImJhc2VsaW5lX29yZGVyIjogOCwKICAgICAgICAiaGlnaGVyX29yZGVyIjogMTIsCiAgICAgICAgInJlbGF0aXZlX21hdHJpeF9kaWZmZXJlbmNlIjogMC4wLAogICAgICAgICJyZWxhdGl2ZV9sb2FkX2RpZmZlcmVuY2UiOiAyLjcxNDAzNTE0MTUwNTNlLTE2LAogICAgICAgICJyZWxhdGl2ZV9ub2RhbF9kaWZmZXJlbmNlIjogMi4zMDMzOTg2ODE3OTk5NzJlLTE2CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAibiI6IDMyLAogICAgICAgICJmb3JtdWxhdGlvbiI6ICJFIiwKICAgICAgICAiYmFzZWxpbmVfb3JkZXIiOiA4LAogICAgICAgICJoaWdoZXJfb3JkZXIiOiAxMiwKICAgICAgICAicmVsYXRpdmVfbWF0cml4X2RpZmZlcmVuY2UiOiAzLjA5NzIyNzgyMjQxMTg3NDNlLTE2LAogICAgICAgICJyZWxhdGl2ZV9sb2FkX2RpZmZlcmVuY2UiOiAyLjg1NDA2NzA2OTQ3MjI1MTNlLTE2LAogICAgICAgICJyZWxhdGl2ZV9ub2RhbF9kaWZmZXJlbmNlIjogNy40NDIzNjczMTgxNTAzMzFlLTE2CiAgICAgIH0KICAgIF0KICB9Cn0='}, 'results/positive/timings_raw.csv': {'sha256': '4bef361275dd5bc21c6abd265dd02eeed73ff8dc072b317eccef5d951d792c2b', 'base64': 'bixuZG9mLG1ldGhvZCxyZXBldGl0aW9uLHBvc2l0aW9uX2luX3BhaXIsY29lZmZpY2llbnRfcyxhc3NlbWJseV9zLHNvbHZlX3MsdG90YWxfcyxzY2FsZWRfYWxnZWJyYWljX3Jlc2lkdWFsDQo4LDQ5LEIsMSwxLDAuMCwwLjAwMjc1NDYxNDAwNDUxNjA0MjgsMC4wMDAxNDAyOTk5OTg3NDc2NTA1MywwLjAwMjg5NDkxNDAwMzI2MzY5MzMsOC4zMTc0MjI3MjAyNDIzNDNlLTE3DQo4LDQ5LEUsMSwyLDguNDgyODAwMzMwOTY2NzExZS0wNSwwLjAwMzI1ODAxNzk5NzcyMzA3MjgsMC4wMDA5NDg4MTM5OTg1NDc4NDgzLDAuMDA0MjkxNjU5OTk5NTgwNTg4LDQuNTI4MzgwNjU0OTU1MzAzZS0xNw0KOCw0OSxFLDIsMSwwLjAwMDEzNTQxMzk5Njc1ODg3MDc4LDAuMDAyNTM1NDk0MDAwNzY3MzU3NiwwLjAwMDE3ODc2ODAwMTIwNTc1MzU0LDAuMDAyODQ5Njc1OTk4NzMxOTgyLDQuNTI4MzgwNjU0OTU1MzAzZS0xNw0KOCw0OSxCLDIsMiwwLjAsMC4wMDIxMzMyMzA5OTg4NzkyOTg2LDAuMDAwMTI2MTQ4OTk2MDU5NzgyOCwwLjAwMjI1OTM3OTk5NDkzOTA4MTQsOC4zMTc0MjI3MjAyNDIzNDNlLTE3DQo4LDQ5LEIsMywxLDAuMCwwLjAwMjE0Mzc2NTk5NjE1MjA2OTQsMC4wMDAxMTg4MzkwMDE1MTU4ODAyMywwLjAwMjI2MjYwNDk5NzY2Nzk0OTYsOC4zMTc0MjI3MjAyNDIzNDNlLTE3DQo4LDQ5LEUsMywyLDUuNjc0NDk5Njk4MjE2MDk1NmUtMDUsMC4wMDMyMzA4Mjc5OTg3NzczODYsMC4wMDAxMjc1NzEwMDE5Mzk5NDEyLDAuMDAzNDE1MTQzOTk3Njk5NDg4LDQuNTI4MzgwNjU0OTU1MzAzZS0xNw0KOCw0OSxFLDQsMSwwLjAwMDEyMTM5MzAwMDg4MTY1MzI4LDAuMDAyMzIxMTcyMDAzNTc2MDY2MywwLjAwMDEyODQ1MzAwMDc5Nzg5MDEzLDAuMDAyNTcxMDE4MDA1MjU1NjA5OCw0LjUyODM4MDY1NDk1NTMwM2UtMTcNCjgsNDksQiw0LDIsMC4wLDAuMDAyMTI5NzQ1MDAwMjc0ODUyLDAuMDAwMTE5OTM5OTk5NDIyMDU4NDYsMC4wMDIyNDk2ODQ5OTk2OTY5MTA0LDguMzE3NDIyNzIwMjQyMzQzZS0xNw0KOCw0OSxCLDUsMSwwLjAsMC4wMDIxODA5NjIwMDExNDgxNDU2LDAuMDAwMTIxMTYyMDAzNzc5MzE4MTgsMC4wMDIzMDIxMjQwMDQ5Mjc0NjQsOC4zMTc0MjI3MjAyNDIzNDNlLTE3DQo4LDQ5LEUsNSwyLDYuOTA1NDAwMzY4ODUwNjc4ZS0wNSwwLjAwMjM0NTMzODk5NDkyODE3LDAuMDAwMTgxNTUyMDAwODY3NjY4NTQsMC4wMDI1OTU5NDQ5OTk0ODQzNDUzLDQuNTI4MzgwNjU0OTU1MzAzZS0xNw0KOCw0OSxFLDYsMSwwLjAwMDE4NDY1Njk5Mzk1MTY0ODQ3LDAuMDAyNzMxOTMwMDA2MTIwODE5NiwwLjAwMDEyMTE5MTk5NTI3NjYwMzg1LDAuMDAzMDM3Nzc4OTk1MzQ5MDcyLDQuNTI4MzgwNjU0OTU1MzAzZS0xNw0KOCw0OSxCLDYsMiwwLjAsMC4wMDIwMzAwNDYwMDE4OTE2MDU2LDAuMDAwMTEzNTMxMDAxMjYxNDM1NDUsMC4wMDIxNDM1NzcwMDMxNTMwNDEsOC4zMTc0MjI3MjAyNDIzNDNlLTE3DQo4LDQ5LEIsNywxLDAuMCwwLjAwMjE3MTAzODAwMjQxNjAyNiwwLjAwMDExNTg5Mzk5OTQ2Mzg3ODU3LDAuMDAyMjg2OTMyMDAxODc5OTA0NCw4LjMxNzQyMjcyMDI0MjM0M2UtMTcNCjgsNDksRSw3LDIsNS40MzkxMDA2NDE1MjQ0MTllLTA1LDAuMDAyNDAzNTg2OTkzNzUwNjUwNCwwLjAwMDE0ODUwMjAwMzg2NzE3OTE2LDAuMDAyNjA2NDgwMDA0MDMzMDczOCw0LjUyODM4MDY1NDk1NTMwM2UtMTcNCjE2LDIyNSxFLDEsMSw5LjY3MDUwMDAwMzI2NzA4ZS0wNSwwLjAwNzk5MjU2Mjk5Nzk2OTc5OSwwLjAwMDQyMDE3MTAwMzQ5NzY3MTM0LDAuMDA4NTA5NDM5MDAxNTAwMTQxLDMuODM2Mzg1MzA3MjEwMDczZS0xNw0KMTYsMjI1LEIsMSwyLDAuMCwwLjAwNjk5NDk1NjAwNTY1NTY5NSwwLjAwMDM2NTg4ODk5ODIwMzQ0NTIsMC4wMDczNjA4NDUwMDM4NTkxNCwyLjM4MDA5MzIxMDM5NTI5NzZlLTE2DQoxNiwyMjUsQiwyLDEsMC4wLDAuMDA4NDM4MTYyMDAzNTg1NjAzLDAuMDAwMzczMTk5OTk2ODI5NDk4NTYsMC4wMDg4MTEzNjIwMDA0MTUxMDIsMi4zODAwOTMyMTAzOTUyOTc2ZS0xNg0KMTYsMjI1LEUsMiwyLDkuOTU1OTk5ODQ4ODk4NWUtMDUsMC4wMDg4NTk5NDM5OTg2ODY1MDQsMC4wMDAzODI1MjQwMDU1NDQ4Mzc1NiwwLjAwOTM0MjAyODAwMjcyMDMyNiwzLjgzNjM4NTMwNzIxMDA3M2UtMTcNCjE2LDIyNSxFLDMsMSw2Ljg3ODMwMDMzNzEyMDg5ZS0wNSwwLjAwODA4ODYwNjk5OTA4MTA5NCwwLjAwMDQ4MDI4MDk5OTkwOTE1NTA3LDAuMDA4NjM3NjcxMDAyMzYxNDU4LDMuODM2Mzg1MzA3MjEwMDczZS0xNw0KMTYsMjI1LEIsMywyLDAuMCwwLjAwNzY5MDMzODk5OTk2NDI5NywwLjAwMDM2ODg3NDAwMzQ3MDQwOTY2LDAuMDA4MDU5MjEzMDAzNDM0NzA2LDIuMzgwMDkzMjEwMzk1Mjk3NmUtMTYNCjE2LDIyNSxCLDQsMSwwLjAsMC4wMDc0NjcyNTM5OTk0OTM1NTA1LDAuMDAwNTAxNDIzMDA0NjA3MjA0MywwLjAwNzk2ODY3NzAwNDEwMDc1NSwyLjM4MDA5MzIxMDM5NTI5NzZlLTE2DQoxNiwyMjUsRSw0LDIsNi4zODQ1OTk2OTE5NTUzNzNlLTA1LDAuMDA4MTM1Mjc2OTk5MzgyNzY4LDAuMDAwMzk2MTU0MDA2MDExNzg0MSwwLjAwODU5NTI3NzAwMjMxNDEwNiwzLjgzNjM4NTMwNzIxMDA3M2UtMTcNCjE2LDIyNSxFLDUsMSw3LjA2NjU5OTkzNzM4ODY3M2UtMDUsMC4wMDg5NDA1OTYwMDI1MDMyOTgsMC4wMDAzODAyNzk5OTgzNTMyMTY4LDAuMDA5MzkxNTQyMDAwMjMwNDAyLDMuODM2Mzg1MzA3MjEwMDczZS0xNw0KMTYsMjI1LEIsNSwyLDAuMCwwLjAwNzQwNTA3MTAwMjcyMjI1MiwwLjAwMDM3MzczOTk5NjU3NTc1MDQsMC4wMDc3Nzg4MTA5OTkyOTgwMDMsMi4zODAwOTMyMTAzOTUyOTc2ZS0xNg0KMTYsMjI1LEIsNiwxLDAuMCwwLjAwNzYzODk0MDk5NTA5MzQzNSwwLjAwMDQyNzQwMjAwMjk2OTc1NjcsMC4wMDgwNjYzNDI5OTgwNjMxOTIsMi4zODAwOTMyMTAzOTUyOTc2ZS0xNg0KMTYsMjI1LEUsNiwyLDcuMDU5NjAwNDY2MTYzODI5ZS0wNSwwLjAwODg1MTE1MDk5NjYzMzgwNSwwLjAwMDQwMDc3MTAwMTI5NTY1MjI0LDAuMDA5MzIyNTE4MDAyNTkxMDk2LDMuODM2Mzg1MzA3MjEwMDczZS0xNw0KMTYsMjI1LEUsNywxLDcuNDgwMjk5OTczMzk0NzIyZS0wNSwwLjAwODIwNzU2NDk5ODg4Mzc1NCwwLjAwMDQ1OTA1OTAwMzMzMzA5NTQ2LDAuMDA4NzQxNDI3MDAxOTUwNzk3LDMuODM2Mzg1MzA3MjEwMDczZS0xNw0KMTYsMjI1LEIsNywyLDAuMCwwLjAwNzAzMjExMjAwMDE2MDg1MSwwLjAwMDM3ODg4OTAwMjYwNDQxMDA1LDAuMDA3NDExMDAxMDAyNzY1MjYxLDIuMzgwMDkzMjEwMzk1Mjk3NmUtMTYNCjMyLDk2MSxCLDEsMSwwLjAsMC4wMzk2MDU5MTc5OTg0MTQ5NSwwLjAwMTcyODcwNDAwMjc0NDA1MjYsMC4wNDEzMzQ2MjIwMDExNTkwMDUsMi4wNzEzMTU2MjA4ODk3NzMzZS0xNg0KMzIsOTYxLEUsMSwyLDAuMDAwMzYyMzkzOTk5MjM5NDI5ODMsMC4wMzI2MTE4NjQwMDA1MDA2NDUsMC4wMDI1NDgzMDQwMDA4MTExODU3LDAuMDM1NTIyNTYyMDAwNTUxMjYsMi41NDA2Nzc4MDgyMTQyNTY4ZS0xNw0KMzIsOTYxLEUsMiwxLDguODU5Mjk5ODUwODgyOTYyZS0wNSwwLjAzMDczNTAzODAwMjYwMzY4NywwLjAwMjEwMTMwMjk5ODE5ODY2MiwwLjAzMjkyNDkzMzk5OTMxMTE4LDIuNTQwNjc3ODA4MjE0MjU2OGUtMTcNCjMyLDk2MSxCLDIsMiwwLjAsMC4wMzI4NzU5Njk5OTg3MTMxMiwwLjAwMjMwMzI0NjAwMTI0NzMxNjYsMC4wMzUxNzkyMTU5OTk5NjA0MzQsMi4wNzEzMTU2MjA4ODk3NzMzZS0xNg0KMzIsOTYxLEIsMywxLDAuMCwwLjAyNzYyNTU5MjAwMzQ2MDA0MiwwLjAwMTc4NzIzMDk5ODE2MTI0ODksMC4wMjk0MTI4MjMwMDE2MjEyOSwyLjA3MTMxNTYyMDg4OTc3MzNlLTE2DQozMiw5NjEsRSwzLDIsNy4zMDA5MDAzMjQ0ODA2MDhlLTA1LDAuMDMxMTc0MTE3MDAwNzczNTUsMC4wMDE3NTc2MzcwMDA5OTE5NjI5LDAuMDMzMDA0NzYzMDA1MDEwMzIsMi41NDA2Nzc4MDgyMTQyNTY4ZS0xNw0KMzIsOTYxLEUsNCwxLDcuMzAzMDAwMTY1ODQ4MDYxZS0wNSwwLjAzMTQ2NzMyNTk5OTk4MzU4NiwwLjAwMTgzNDQ1MTk5OTQ1NzA4NzQsMC4wMzMzNzQ4MDgwMDEwOTkxNTQsMi41NDA2Nzc4MDgyMTQyNTY4ZS0xNw0KMzIsOTYxLEIsNCwyLDAuMCwwLjAyNjgxOTI2MTk5ODc1MzA2MywwLjAwMjE2MDI1MTk5OTE1NzQ4MSwwLjAyODk3OTUxMzk5NzkxMDU0NCwyLjA3MTMxNTYyMDg4OTc3MzNlLTE2DQozMiw5NjEsQiw1LDEsMC4wLDAuMDMzMTI2NjM2MDAwODE1NzgsMC4wMDE3MTMyOTA5OTk3NTk5MiwwLjAzNDgzOTkyNzAwMDU3NTcsMi4wNzEzMTU2MjA4ODk3NzMzZS0xNg0KMzIsOTYxLEUsNSwyLDcuMDUwNjAwMTA2NTk1MDg0ZS0wNSwwLjAzMTE5NjUwMDAwMDA3ODIzLDAuMDAyMDk4MDA4MDAwNzU3NTQ1MiwwLjAzMzM2NTAxNDAwMTkwMTczLDIuNTQwNjc3ODA4MjE0MjU2OGUtMTcNCjMyLDk2MSxFLDYsMSw3LjcxMDYwMDAzODk5MDM3OGUtMDUsMC4wMzE4MDI3MDkwMDMzODY1NSwwLjAwMTg4NTI2ODk5NzI4NDU4MzcsMC4wMzM3NjUwODQwMDEwNjEwNCwyLjU0MDY3NzgwODIxNDI1NjhlLTE3DQozMiw5NjEsQiw2LDIsMC4wLDAuMDMxMTgwOTU2OTk3NTU5NDA2LDAuMDAxNzIyMDI0MDA0MjY2MTMxNywwLjAzMjkwMjk4MTAwMTgyNTU0LDIuMDcxMzE1NjIwODg5NzczM2UtMTYNCjMyLDk2MSxCLDcsMSwwLjAsMC4wMzcwMjQ5NDQ5OTg4NDc3ODYsMC4wMDE4Mjc4ODMwMDI5ODg1Mjg1LDAuMDM4ODUyODI4MDAxODM2MzE1LDIuMDcxMzE1NjIwODg5NzczM2UtMTYNCjMyLDk2MSxFLDcsMiw5LjE3MDgwMDMzMTA0ODY2N2UtMDUsMC4wMzE5NzgwNDIwMDMxNzQ3OCwwLjAwMTk1Mjk3MDk5NzA4MTEzODIsMC4wMzQwMjI3MjEwMDM1NjY0LDIuNTQwNjc3ODA4MjE0MjU2OGUtMTcNCjY0LDM5NjksRSwxLDEsOS4zODM5OTk3MTM0NjM3MDllLTA1LDAuMTMxMjM1MzE4MDAzNzQ2MjIsMC4wMDk4NzU1NDc5OTkzOTk3MTQsMC4xNDEyMDQ3MDYwMDAyODA1NywyLjI3MDYyNTY4MjI4Mjk5OTZlLTE3DQo2NCwzOTY5LEIsMSwyLDAuMCwwLjEzMDAyOTc2NDAwNDU3NTU1LDAuMDA5NTg4NTQ3MDAwMDIxMjA4LDAuMTM5NjE4MzExMDA0NTk2NzYsNC43NjUzNzI3NzU1MDQwNjVlLTE2DQo2NCwzOTY5LEIsMiwxLDAuMCwwLjEyMDk1Njk3NDAwMjE4MzQxLDAuMDA4ODY2Nzk0OTk2NzIwMjczLDAuMTI5ODIzNzY4OTk4OTAzNyw0Ljc2NTM3Mjc3NTUwNDA2NWUtMTYNCjY0LDM5NjksRSwyLDIsMC4wMDAyMzI2Mjg5OTc3NjQ1NTAxNSwwLjEyNzM2OTYyNzAwMDAwNzMsMC4wMTAwMzM3MzUwMDEzNDMzMDIsMC4xMzc2MzU5OTA5OTkxMTUxNywyLjI3MDYyNTY4MjI4Mjk5OTZlLTE3DQo2NCwzOTY5LEUsMywxLDcuNjk5NDk5ODM4MDU0MThlLTA1LDAuMTI0MDk0MTY3MDAzOTMxOTQsMC4wMDg2NzUxMTc5OTg3OTEzOTMsMC4xMzI4NDYyODAwMDExMDM4OCwyLjI3MDYyNTY4MjI4Mjk5OTZlLTE3DQo2NCwzOTY5LEIsMywyLDAuMCwwLjEwNjk2ODM3MDAwNDMyNDI0LDAuMDA5NjA1MDQxOTk2MTEwMjMsMC4xMTY1NzM0MTIwMDA0MzQ0Nyw0Ljc2NTM3Mjc3NTUwNDA2NWUtMTYNCjY0LDM5NjksQiw0LDEsMC4wLDAuMTExODI2MjMzOTk1MDg4ODQsMC4wMDk2ODgzMjcwMDE2NDA1NzMsMC4xMjE1MTQ1NjA5OTY3Mjk0MSw0Ljc2NTM3Mjc3NTUwNDA2NWUtMTYNCjY0LDM5NjksRSw0LDIsOS4zODgwMDAwMzQ5NTk5ODdlLTA1LDAuMTMzNTY1OTE0MDAyMjQ3MTUsMC4wMTEwNzI2NjQ5OTQ1NzE4MjQsMC4xNDQ3MzI0NTg5OTcxNjg1OCwyLjI3MDYyNTY4MjI4Mjk5OTZlLTE3DQo2NCwzOTY5LEUsNSwxLDkuNzM5NzAwMTgwOTQ0MDU1ZS0wNSwwLjEzMDAzNTkxNzAwNTQ0MzQ4LDAuMDEwMzU5ODg0OTk2NDMyODExLDAuMTQwNDkzMTk5MDAzNjg1NzMsMi4yNzA2MjU2ODIyODI5OTk2ZS0xNw0KNjQsMzk2OSxCLDUsMiwwLjAsMC4xMTY3Mjc0MDg5OTY2NjY3OCwwLjAxMDAyNjE4NDAwNTI1NTM0NSwwLjEyNjc1MzU5MzAwMTkyMjEzLDQuNzY1MzcyNzc1NTA0MDY1ZS0xNg0KNjQsMzk2OSxCLDYsMSwwLjAsMC4xMDg2NDY3MDI5OTYzMzg3MiwwLjAwOTUyNzQxNjAwNDg1NzUxNCwwLjExODE3NDExOTAwMTE5NjIzLDQuNzY1MzcyNzc1NTA0MDY1ZS0xNg0KNjQsMzk2OSxFLDYsMiw4LjM4OTYwMDQwNzA5OTM1MWUtMDUsMC4xMjU1ODg4MzQ5OTg3Nzk5MywwLjAxMDI5Mjc4NTAwMjA5MzI5NiwwLjEzNTk2NTUxNjAwNDk0NDIyLDIuMjcwNjI1NjgyMjgyOTk5NmUtMTcNCjY0LDM5NjksRSw3LDEsNy41NTUyOTk4MTY4ODU5NjNlLTA1LDAuMTMxMzc2NTcwMDAzMzQwMDIsMC4wMDk5MTcwMzk5OTg5ODE2MzYsMC4xNDEzNjkxNjMwMDA0OTA1MiwyLjI3MDYyNTY4MjI4Mjk5OTZlLTE3DQo2NCwzOTY5LEIsNywyLDAuMCwwLjExNDU4NzMzODAwMTAwMTYzLDAuMDA5NjU1Mzk2OTk5NzMzMDU3LDAuMTI0MjQyNzM1MDAwNzM0NjgsNC43NjUzNzI3NzU1MDQwNjVlLTE2DQoxMjgsMTYxMjksQiwxLDEsMC4wLDAuNDg1MTE1MTkyMDAxMTAzNCwwLjA1OTg5OTk5ODk5NjExNTc1NSwwLjU0NTAxNTE5MDk5NzIxOTIsNC4zMDkzNTQwMjYxNDQxODI2ZS0xNg0KMTI4LDE2MTI5LEUsMSwyLDguMzg4NTk5OTYyOTI3NDAxZS0wNSwwLjYxMzQzMTAwMTk5ODgzMzQsMC4wNjg2Njk3MTc5OTkwMDA1NywwLjY4MjE4NDYwNTk5NzQ2MzIsMi42Nzg2NjA1MTQzMjQxMTI3ZS0xNw0KMTI4LDE2MTI5LEUsMiwxLDAuMDAwMTA1ODA5MDA1NjE3NjI5NzQsMC41MTczMjAzOTM5OTkzOTgyLDAuMDY0MTYzMjQ1OTk1NTg3MiwwLjU4MTU4OTQ0OTAwMDYwMywyLjY3ODY2MDUxNDMyNDExMjdlLTE3DQoxMjgsMTYxMjksQiwyLDIsMC4wLDAuNDM5NzYyMTg2OTkzOTAzLDAuMDYzNDYyNzAxMDAwMzg5NjQsMC41MDMyMjQ4ODc5OTQyOTI3LDQuMzA5MzU0MDI2MTQ0MTgyNmUtMTYNCjEyOCwxNjEyOSxCLDMsMSwwLjAsMC40NjczODQ1OTQwMDUxMjUxLDAuMDYzODQ3Mzg1OTk0NDQ5NzIsMC41MzEyMzE5Nzk5OTk1NzQ4LDQuMzA5MzU0MDI2MTQ0MTgyNmUtMTYNCjEyOCwxNjEyOSxFLDMsMiw5LjYyNTQ5OTY2MDYxNDg3OGUtMDUsMC40OTI5NDM1NDM5OTg5OTAyLDAuMDU5MjI0ODk4OTk5NjczMzE0LDAuNTUyMjY0Njk3OTk1MjY5NywyLjY3ODY2MDUxNDMyNDExMjdlLTE3DQoxMjgsMTYxMjksRSw0LDEsMC4wMDAxMDQwNjY5OTU4NDU2NDE5NCwwLjQ5ODgyNjM1MTAwMjIwNzk2LDAuMDYxMDA1MTYwOTk4MzIwMjEsMC41NTk5MzU1Nzg5OTYzNzM4LDIuNjc4NjYwNTE0MzI0MTEyN2UtMTcNCjEyOCwxNjEyOSxCLDQsMiwwLjAsMC40Mzc2NDAwODg5OTU5NzY4NCwwLjA2MDc1NjI3NzAwMTkyODU0LDAuNDk4Mzk2MzY1OTk3OTA1NCw0LjMwOTM1NDAyNjE0NDE4MjZlLTE2DQoxMjgsMTYxMjksQiw1LDEsMC4wLDAuNDQzMTYyODMyMDAxNjg5OCwwLjA2MTYwOTIzNjk5NjQ0NjI3LDAuNTA0NzcyMDY4OTk4MTM2MSw0LjMwOTM1NDAyNjE0NDE4MjZlLTE2DQoxMjgsMTYxMjksRSw1LDIsOC45MDc0MDA0NzkwNzQ1NjhlLTA1LDAuNDg0NTc1MDU2OTk0Nzk5NSwwLjA1NjkzNTc2NTAwMTk4MTA1NCwwLjU0MTU5OTg5NjAwMTU3MTMsMi42Nzg2NjA1MTQzMjQxMTI3ZS0xNw0KMTI4LDE2MTI5LEUsNiwxLDkuMzExMDAwMDMwNzIwNjA2ZS0wNSwwLjQ3NzUzOTM4MDAwMDk2MjQsMC4wNzAwODU2OTYwMDI5MzQxMywwLjU0NzcxODE4NjAwNDIwMzcsMi42Nzg2NjA1MTQzMjQxMTI3ZS0xNw0KMTI4LDE2MTI5LEIsNiwyLDAuMCwwLjQxOTI0MTM2MDk5OTg3NjYzLDAuMDYwMzQ3NjgzOTk2OTUxNjcsMC40Nzk1ODkwNDQ5OTY4MjgzLDQuMzA5MzU0MDI2MTQ0MTgyNmUtMTYNCjEyOCwxNjEyOSxCLDcsMSwwLjAsMC40MzMzODgxMzg5OTc1NDMxNCwwLjA2MDQ2MzExNjk5OTAxNzA3NSwwLjQ5Mzg1MTI1NTk5NjU2MDIsNC4zMDkzNTQwMjYxNDQxODI2ZS0xNg0KMTI4LDE2MTI5LEUsNywyLDcuOTY4MDAwMTM2MzE1ODIzZS0wNSwwLjQ3MzcyMjgzMTAwNDEyNDYsMC4wNTUwODI1NDQwMDA0NzAwOCwwLjUyODg4NTA1NTAwNTk1NzgsMi42Nzg2NjA1MTQzMjQxMTI3ZS0xNw0KMjU2LDY1MDI1LEUsMSwxLDAuMDAwMjM2NjA1MDAzMDEwNDgxNiwxLjkzNzgxMDUyNTk5NjYxMDUsMC40MDQzOTExODAwMDM1ODU3NiwyLjM0MjQzODMxMTAwMzIwNjgsMy43MDQyODM3NzQ5NjQ1MTg1ZS0xNw0KMjU2LDY1MDI1LEIsMSwyLDAuMCwxLjY5NjUyNzY0NDk5OTE1MDcsMC40MDUwNDI1ODcwMDA5MzQ5LDIuMTAxNTcwMjMyMDAwMDg1NSw1LjQ5NDk0OTU2MzQ0NTU1MWUtMTYNCjI1Niw2NTAyNSxCLDIsMSwwLjAsMS43ODczNzkzNDIwMDAzNDk0LDAuNDE5NjAzNDUxMDAyMzMxLDIuMjA2OTgyNzkzMDAyNjgwNCw1LjQ5NDk0OTU2MzQ0NTU1MWUtMTYNCjI1Niw2NTAyNSxFLDIsMiwwLjAwMDEwNTU3ODk5ODA0NTUzMDE3LDEuOTMwMDA1MTQwMDA1OTYwMSwwLjQzMDk4NTAzNjAwMDQ5MywyLjM2MTA5NTc1NTAwNDQ5ODYsMy43MDQyODM3NzQ5NjQ1MTg1ZS0xNw0KMjU2LDY1MDI1LEUsMywxLDAuMDAwMTA1MTc3OTk4MTkzNTM5NjgsMi4wNjA4NTMxMDU5OTg4MzEsMC40MzM1NDY0NDQwMDA5NjczLDIuNDk0NTA0NzI3OTk3OTkxNywzLjcwNDI4Mzc3NDk2NDUxODVlLTE3DQoyNTYsNjUwMjUsQiwzLDIsMC4wLDEuOTY2NDkyMzQ5OTk0OTUxLDAuNDU4NzMyNTY2MDAzMTYzOCwyLjQyNTIyNDkxNTk5ODExNDcsNS40OTQ5NDk1NjM0NDU1NTFlLTE2DQoyNTYsNjUwMjUsQiw0LDEsMC4wLDEuODczMDI5MDQ2MDAwNzM4MiwwLjQyMDA0NDA4Nzk5NDAyODgsMi4yOTMwNzMxMzM5OTQ3NjcsNS40OTQ5NDk1NjM0NDU1NTFlLTE2DQoyNTYsNjUwMjUsRSw0LDIsOS43MzM2MDAwMTgwODAzMjRlLTA1LDEuOTY4Njc3OTEyMDAyMDIyLDAuNDkyODEzNTgwOTk2ODgwOCwyLjQ2MTU4ODgyODk5OTA4MzUsMy43MDQyODM3NzQ5NjQ1MTg1ZS0xNw0KMjU2LDY1MDI1LEUsNSwxLDAuMDAwMTE0MzkxOTk0NDI5NzUyMjMsMi4xNzY0Nzc1NzEwMDM0ODgzLDAuNDUzODA5Mjk4MDAwMTIzOCwyLjYzMDQwMTI2MDk5ODA0MiwzLjcwNDI4Mzc3NDk2NDUxODVlLTE3DQoyNTYsNjUwMjUsQiw1LDIsMC4wLDEuOTMzMzcxNTI2OTk2Nzc5NywwLjQ1ODAyMzQwMTAwNjM1Nzc0LDIuMzkxMzk0OTI4MDAzMTM3NSw1LjQ5NDk0OTU2MzQ0NTU1MWUtMTYNCjI1Niw2NTAyNSxCLDYsMSwwLjAsMS44NTc2MjI4NzkwMDQzMzg4LDAuNDU2OTIxNTk2OTk2ODM1NDMsMi4zMTQ1NDQ0NzYwMDExNzQsNS40OTQ5NDk1NjM0NDU1NTFlLTE2DQoyNTYsNjUwMjUsRSw2LDIsOS4yMzc4MDAyNTkxNDMxMTRlLTA1LDIuMTcxMDUyMTgzOTk5NTYzNiwwLjQ4OTc0ODEzODk5OTg1NzUsMi42NjA4OTI3MDEwMDIwMTI1LDMuNzA0MjgzNzc0OTY0NTE4NWUtMTcNCjI1Niw2NTAyNSxFLDcsMSwwLjAwMDEwOTEwNDAwMzA1ODc0NjQ2LDIuMjg5MDE0NzQwOTk5ODQ2MywwLjQ5ODYyNTUxNDAwMjkyNDMsMi43ODc3NDkzNTkwMDU4MjkzLDMuNzA0MjgzNzc0OTY0NTE4NWUtMTcNCjI1Niw2NTAyNSxCLDcsMiwwLjAsMS44ODkyNDgyMTIwMDYzNTA1LDAuNjUyNTgxMTg1OTk3NDA1NiwyLjU0MTgyOTM5ODAwMzc1Niw1LjQ5NDk0OTU2MzQ0NTU1MWUtMTYNCjUxMiwyNjExMjEsQiwxLDEsMC4wLDcuNDc2NjE2ODY1MDAyMTQzLDQuNzcxMTMwNTYxOTk1OTYsMTIuMjQ3NzQ3NDI2OTk4MTAzLDEuMDM5ODc1MjUzNDIwMzcwNGUtMTUNCjUxMiwyNjExMjEsRSwxLDIsMC4wMDAxNzU4ODQwMDA3ODIzODkyLDguNTczMDEzMTc2OTk2MDI4LDQuNzg3NTgzMzMxOTk5NDg1LDEzLjM2MDc3MjM5Mjk5NjI5Niw1LjU5MTg4NjIyNzU2NjY5MmUtMTcNCjUxMiwyNjExMjEsRSwyLDEsOS43NzQ2OTk3MTk4NTU1NjJlLTA1LDguMjY2MzQ2Mzk3OTk4ODYsNC4zODQwNDg2NTcwMDQzMTQsMTIuNjUwNDkyODAyMDAwMzcyLDUuNTkxODg2MjI3NTY2NjkyZS0xNw0KNTEyLDI2MTEyMSxCLDIsMiwwLjAsNy4xNzg5Mzg1NjMwMDM2MTIsNC42NDg1NDMzMTIwMDMzMTIsMTEuODI3NDgxODc1MDA2OTI0LDEuMDM5ODc1MjUzNDIwMzcwNGUtMTUNCjUxMiwyNjExMjEsQiwzLDEsMC4wLDcuNjg5Nzc2NzM3OTk5NjM3LDQuNzA5NTQwMjc2OTk5MTc4LDEyLjM5OTMxNzAxNDk5ODgxNSwxLjAzOTg3NTI1MzQyMDM3MDRlLTE1DQo1MTIsMjYxMTIxLEUsMywyLDkuMDc1Njk5OTI3MDUyNDgzZS0wNSw4LjE1NDMwNzU0MTk5NzY5Niw0LjM2MzY4NDc0ODAwMTEwNiwxMi41MTgwODMwNDY5OTgwNzIsNS41OTE4ODYyMjc1NjY2OTJlLTE3DQo1MTIsMjYxMTIxLEUsNCwxLDAuMDAwMTAzNzA1OTk5MjA4NjE0MjMsOC42MzI1NzQzMDgwMDE0OTUsNS41NzIyNDQ1Mjk5OTc5ODgsMTQuMjA0OTIyNTQzOTk4NjkxLDUuNTkxODg2MjI3NTY2NjkyZS0xNw0KNTEyLDI2MTEyMSxCLDQsMiwwLjAsNy45NzU1MDIyNzgwMDA5MzEsNC43MDMxNTQ5MzYwMDM1NTcsMTIuNjc4NjU3MjE0MDA0NDg4LDEuMDM5ODc1MjUzNDIwMzcwNGUtMTUNCjUxMiwyNjExMjEsQiw1LDEsMC4wLDguMTEwMDgyNzQ5OTk3NjczLDQuNTQxMjQzOTA2OTk4NzE2LDEyLjY1MTMyNjY1Njk5NjM4OSwxLjAzOTg3NTI1MzQyMDM3MDRlLTE1DQo1MTIsMjYxMTIxLEUsNSwyLDkuNzk2NzAwMDMyODkzNTY4ZS0wNSw4LjQ4NDI1MDMwNTAwMjIwNiw0Ljg0MTk3MzEyODAwMDk3MywxMy4zMjYzMjE0MDAwMDM1MDgsNS41OTE4ODYyMjc1NjY2OTJlLTE3DQo1MTIsMjYxMTIxLEUsNiwxLDAuMDAwMTIzNDk1MDAzMjA4NTE4MDMsOS4wNjA4MjM2MzE5OTY3NTUsNC44MDIwMDM4NjcwMDAxNjcsMTMuODYyOTUwOTk0MDAwMTMsNS41OTE4ODYyMjc1NjY2OTJlLTE3DQo1MTIsMjYxMTIxLEIsNiwyLDAuMCw3LjUwODU1NzY5MDAwMTkwOCw0LjkxMjc5NTcwNTAwMDQ2MjUsMTIuNDIxMzUzMzk1MDAyMzcsMS4wMzk4NzUyNTM0MjAzNzA0ZS0xNQ0KNTEyLDI2MTEyMSxCLDcsMSwwLjAsNy4yNjAyOTI1NjAwMDAzOCw0LjA2MzI5MTk3Njk5OTYwOCwxMS4zMjM1ODQ1MzY5OTk5ODgsMS4wMzk4NzUyNTM0MjAzNzA0ZS0xNQ0KNTEyLDI2MTEyMSxFLDcsMiw5Ljg0MTgwMDA1NjE2NTA4NGUtMDUsNy42MDMxMzQ0OTY5OTYwNjMsNC4xMjc2NjE2MjAwMDQ4MDEsMTEuNzMwODk0NTM1MDAxNDI2LDUuNTkxODg2MjI3NTY2NjkyZS0xNw0K'}}
RECORDED_PROVENANCE = {'positive': {'started_utc': '2026-10-16T14:22:00.414821+00:00', 'completed_utc': '2026-10-16T14:26:25.975584+00:00', 'cpu': 'AMD EPYC 9V74 80-Core Processor', 'python': '3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]', 'numpy': '2.3.5', 'scipy': '1.17.0', 'summary_rows': 14, 'raw_timing_rows': 98, 'engine_sha256': '41ae1f86063e82fcffce7e9d183958c6a61477add5391e8d870c012f60a8157d'}, 'negative': {'started_utc': '2026-10-16T14:26:26.316098+00:00', 'completed_utc': '2026-10-16T14:30:35.103231+00:00', 'cpu': 'AMD EPYC 9V74 80-Core Processor', 'python': '3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]', 'numpy': '2.3.5', 'scipy': '1.17.0', 'summary_rows': 14, 'raw_timing_rows': 98, 'engine_sha256': '7c28c9842eab5ff120e7842c4cc8125584359fb549f6e1e18c2c3a9a8327ab3b'}, 'noncoercive': {'started_utc': '2026-10-16T14:30:35.384937+00:00', 'completed_utc': '2026-10-16T14:52:06.549524+00:00', 'cpu': 'AMD EPYC 9V74 80-Core Processor', 'python': '3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]', 'numpy': '2.3.5', 'scipy': '1.17.0', 'summary_rows': 70, 'raw_timing_rows': 490, 'engine_sha256': '951f65e282ad1ba3e0a67c14d2d5b336052ba908762937eaeae2f1064da2f381'}}


def restore_payload(payload, destination):
    destination = Path(destination)
    for relative, item in payload.items():
        relative_path = Path(relative)
        if relative_path.is_absolute() or ".." in relative_path.parts:
            raise ValueError("Unsafe embedded path: " + relative)
        content = base64.b64decode(item["base64"], validate=True)
        if hashlib.sha256(content).hexdigest() != item["sha256"]:
            raise ValueError("Embedded checksum failed: " + relative)
        path = destination / relative_path
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)

restore_payload(SOURCE_PAYLOAD, WORKDIR)
RECORDED_ROOT = WORKDIR / "recorded_run"
restore_payload(RECORDED_PAYLOAD, RECORDED_ROOT)
print(f"Restored {len(SOURCE_PAYLOAD)} source/template files and {len(RECORDED_PAYLOAD)} recorded data files.")
for case, info in RECORDED_PROVENANCE.items():
    print(case, "| completed UTC:", info["completed_utc"], "| CPU:", info["cpu"])

세 벤치마크를 순서대로 실행합니다. 서로 다른 벤치마크를 병렬로 돌리지 않으며,
각 실행에서 수치 스레드를 1개로 제한합니다. 각 선형계의 LU 분해를 매번 새로 수행합니다.
시간 측정에는 계수 준비, 행렬 및 우변 조립, LU 분해 및 풀이가 포함됩니다.
공통 격자 생성, 오차 평가, 검증, 그래프 생성은 제외합니다.

In [ ]:
CANONICAL_PROTOCOL = (
    SIZES == [8, 16, 32, 64, 128, 256, 512]
    and K_VALUES == [4, 8, 12, 16, 20]
    and REPEATS == 7 and ASSEMBLY_ORDER == 8 and ERROR_ORDER == 8
)

def run_command(arguments):
    print("Running:", " ".join(map(str, arguments)), flush=True)
    subprocess.run([str(value) for value in arguments], check=True, env=os.environ.copy())

if RUN_NEW_EXPERIMENT:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    ACTIVE_ROOT = WORKDIR / ("fresh_run_" + stamp)
    ACTIVE_ROOT.mkdir(parents=True)
else:
    ACTIVE_ROOT = RECORDED_ROOT
    # Replay always uses the complete embedded protocol.
    CANONICAL_PROTOCOL = True

shutil.copytree(WORKDIR / "code", ACTIVE_ROOT / "code", dirs_exist_ok=True)

if RUN_NEW_EXPERIMENT:
    engines = [("positive", "ou_positive_fem.py"),
               ("negative", "ou_be_fem.py"),
               ("noncoercive", "ou_noncoercive_fem.py")]
    for case, engine in engines:
        command = [sys.executable, ACTIVE_ROOT / "code" / engine,
                   "--output", ACTIVE_ROOT / "results" / case,
                   "--sizes", *map(str, SIZES), "--repeats", str(REPEATS),
                   "--assembly-order", str(ASSEMBLY_ORDER),
                   "--error-order", str(ERROR_ORDER), "--batch-size", str(BATCH_SIZE)]
        if case == "noncoercive":
            command += ["--k", *map(str, K_VALUES)]
        run_command(command)
    print("Fresh FEM computations completed:", ACTIVE_ROOT)
else:
    print("Recorded-data replay; no FEM or timing measurements repeated:", ACTIVE_ROOT)

다음 셀은 CSV와 원시 시간 기록의 일관성, 상대오차 정규화, 계수 항등식 및 조립을 검사합니다.
기본 전체 새 실행에서는 $n=512$의 민감도 검사도 수행합니다.
기록 재생 모드는 새 민감도 풀이를 하지 않으며, 원래 검증 기록을 보존합니다.

In [ ]:
validation_output = ACTIVE_ROOT / "results" / (
    "independent_validation.json" if RUN_NEW_EXPERIMENT else "replay_validation.json")
verify_command = [sys.executable, ACTIVE_ROOT / "code" / "verify_results.py",
                  "--results-root", ACTIVE_ROOT / "results",
                  "--output", validation_output]
if not CANONICAL_PROTOCOL:
    verify_command.append("--allow-subset")
if RUN_NEW_EXPERIMENT and FULL_VALIDATION and not QUICK_MODE and CANONICAL_PROTOCOL:
    verify_command += ["--n", str(max(SIZES))]
else:
    verify_command.append("--structural-only")
    print("Structural/data validation only; fine-grid sensitivity solves are skipped in this mode.")
run_command(verify_command)
print("This notebook validation report:", validation_output)

원래 그림 파일명을 유지하여 PDF와 미리보기 PNG를 생성합니다.
전체 논문 설정일 때만 새 데이터와 실행 환경을 반영한 Section LaTeX를 생성합니다.
결과 표의 모든 수치는 CSV에서 읽으며, 그림 파일이 없을 때 가짜 그림으로 대체하지 않습니다.

In [ ]:
if RUN_NEW_EXPERIMENT:
    plot_sizes = SIZES
    plot_k_values = K_VALUES
else:
    plot_sizes = [8, 16, 32, 64, 128, 256, 512]
    plot_k_values = [4, 8, 12, 16, 20]

if plot_k_values == [4, 8, 12, 16, 20]:
    run_command([sys.executable, ACTIVE_ROOT / "code" / "make_all_figures.py",
                 "--results-root", ACTIVE_ROOT / "results", "--output", ACTIVE_ROOT,
                 "--expected-sizes", *map(str, plot_sizes)])
else:
    print("Custom k subset: paper figure layout skipped; all computed CSV results remain available.")

if CANONICAL_PROTOCOL:
    run_command([sys.executable, ACTIVE_ROOT / "code" / "build_sections.py",
                 "--root", ACTIVE_ROOT])
else:
    print("Noncanonical settings: paper Section text is intentionally not generated.")

In [ ]:
def show_csv_table(path):
    with Path(path).open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    fields = [key for key in ("k", "n", "ndof", "method", "l2", "l2_relative",
                              "h1", "h1_relative", "total_median_s") if key in rows[0]]
    lines = [" | ".join(fields), " | ".join(["---"] * len(fields))]
    for row in rows:
        values = []
        for key in fields:
            value = row[key]
            if key in {"l2", "l2_relative", "h1", "h1_relative"}:
                value = f"{float(value):.6e}"
            elif key == "total_median_s":
                value = f"{float(value):.6f}"
            values.append(value)
        lines.append(" | ".join(values))
    display(Markdown("\n".join(lines)))

for case in ("positive", "negative", "noncoercive"):
    display(Markdown("### " + case))
    show_csv_table(ACTIVE_ROOT / "results" / case / "convergence.csv")

figure_names = [
    "ou_positive_errors", "ou_positive_relative_errors", "ou_positive_runtime",
    "ou_be_errors", "ou_be_relative_errors", "ou_be_runtime",
    "ou_noncoercive_l2", "ou_noncoercive_h1",
    "ou_noncoercive_l2_relative", "ou_noncoercive_h1_relative",
    "ou_noncoercive_runtime",
]
for name in figure_names:
    path = ACTIVE_ROOT / (name + ".png")
    if path.exists():
        display(Markdown("`" + name + ".pdf`"))
        display(Image(filename=str(path), width=900))

section_path = ACTIVE_ROOT / "ou_fem_sections_512.tex"
if section_path.exists():
    display(FileLink(str(section_path)))
print("Relative errors are dimensionless ratios. H1 is the full unweighted norm.")

마지막으로 결과 CSV, 원시 시간 기록, 실행 환경, 검증 JSON, 소스코드, Section LaTeX,
PDF 그림을 하나의 ZIP으로 묶습니다. 새 실행에서는 마지막 격자의 절점 해 `.npz`도 포함합니다.
기록 재생에는 큰 절점 해 배열을 내장하지 않으므로 `.npz`가 포함되지 않습니다.

LaTeX 원고에는 `ou_fem_sections_512.tex`의 내용을 필요한 위치에 넣고,
그림 PDF 11개를 주 `.tex`와 같은 폴더에 두세요. Section 파일은 본문 조각이며
원고의 앞선 이론 정리와 라벨을 참조합니다. LaTeX 자체를 컴파일할 때에는
원고의 기존 전처리부와 정리 환경을 사용하세요.

In [ ]:
archive_base = WORKDIR / (ACTIVE_ROOT.name + "_download")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=ACTIVE_ROOT))
print("Download archive:", archive_path)
display(FileLink(str(archive_path)))

# Set True and run this cell in Colab to trigger a browser download.
DOWNLOAD_ARCHIVE = False
if DOWNLOAD_ARCHIVE:
    try:
        from google.colab import files
    except ImportError:
        print("Outside Colab: use the ZIP link above or your file explorer.")
    else:
        files.download(str(archive_path))